<a href="https://colab.research.google.com/github/jihyeon26/computational-language-technology-hs26/blob/main/stage3/stage3_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tracking Open Models in AI Media · Stage 3: Implementing a RAG System for Question Answering

## Objective
Stage 3 builds and evaluates a question-answering system (RAG) over the open-models articles of the AI Media Dataset,
in three parts that build on each other:
1. **Part 1 · Q&A test set and text-only RAG** (Sections 1 – 2): a reviewed test set of the track, and a baseline that retrieves
   passages from the articles and generates answers grounded in these passages, with their publication dates.
2. **Part 2 · RAG with the knowledge graph** (Sections 3 – 4): the Stage 1 knowledge graph is revised for retrieval, following
   the Stage 1 feedback, and added to the retrieval, so that answers can also use the entities and relations of the
   track; both systems answer the same test set and are compared.
3. **Part 3 · Evaluation and enhancement** (Section 5): the answers are scored with a judge model, the failures are located,
   and one limitation is addressed with one enhancement, whose effect is evaluated in turn.

## Link with the other stages
- **Input from Stage 1**: the cleaned corpus and the core open-models articles (the RAG corpus and the source of the
  test set), the Stage 1 lexicon (selection of track passages), the topics (coverage of the test set) and the knowledge
  graph with its schema, revised in Section 3 following the Stage 1 feedback.

## Stage 3 requirements and where they are addressed
| Project description (Stage 3) | Section |
|---|---|
| **Q&A Dataset Construction**: 100–150 passages, 200–300 Q&A pairs from an LLM different from the RAG model, five question types, manual review | 1, 5.1 |
| **RAG Implementation**: text-based baseline, dense, sparse and hybrid retrieval, grounded answers with publication dates | 2 |
| **Knowledge Graph Integration**: Stage 1 graph in the RAG system, graph-aware and entity-anchored retrieval, answers citing graph facts, comparison with the text-only baseline | 3, 4, 5.2 – 5.3 |
| **System Evaluation**: faithfulness, relevance, context and temporal grounding per question category, text-only vs. graph RAG | 5.3 – 5.5 |
| **RAG Enhancement**: one limitation from the evaluation, one enhancement (re-ranking), its effect and trade-offs | 5.6 – 5.7 |
| **Outputs**: evaluation tables and figures, insights into RAG and knowledge graphs for trend monitoring | 4.5, 5.3 – 5.7 |
| **Stage 1 feedback**: reliability of the relations used for retrieval, lexicon knowledge separate from article claims, passages that support their relations | 3, 4.4 |
| **Q&A session of 30 October**: questions labelled factoid, multi-hop or global, paraphrased questions, comparison per category | 5.1, 5.3.1 |


## Contributions
| Team member | Sections | Contribution |
|---|---|---|
| Jennifer Spieser | 1, 2 | Q&A dataset construction, text-only RAG |
| Jihyeon Choung | 3, 4 | knowledge graph v2 (Stage 1 feedback), entity linking, graph-aware retrieval, fusion of text and graph evidence, comparison with the text-only baseline |
| Daliasny Gonzalez Guzman | 5 | multi-hop and global questions, evaluation with a judge model, human check of the judge, retrieval diagnosis, re-ranking enhancement |

The manual review of the Q&A pairs (Section 1.6) was shared by all members.

## How to run
[Open in Colab](https://colab.research.google.com/github/jihyeon26/computational-language-technology-hs26/blob/main/stage3/stage3_final.ipynb) · [Repository](https://github.com/jihyeon26/computational-language-technology-hs26)

Run all cells from top to bottom in Colab. Sections 1, 3 and 5 each start with a short setup cell that clones the
repository and mounts the shared Google Drive (`MyDrive/HSLU/CLT`), where the inputs and the cached outputs are stored.
Every long step (embeddings, NLI scores, generation, judging) is loaded from the cache when it exists: a GPU runtime is
only needed to generate missing answers with the local RAG model, and an OpenAI key in the Colab secrets
(`OPENAI_API_KEY`) only to judge missing answers.


# 1. Q&A Dataset Construction
To know whether the RAG system works, it must be tested on questions whose answer and source are known in advance. This first part builds that test set: question/answer pairs, each linked to the passage and the article it comes from. It is used to evaluate both the retrieval (is the source article found?) and the answers (are they correct and faithful to the sources?). To be useful, it must be representative (covering the subtopics, entities and periods of the track) and reliable (every question is clear and answerable from its source).

**Pipeline**
1. **Track articles**: the core open-models articles of Stage 1
2. **Passages**: the articles are cut into passages of a few paragraphs
3. **Track passages**: only passages about open models are kept
4. **Selection**: 150 passages covering periods, subtopics and entities
5. **Generation**: two questions per passage drafted by an LLM
6. **Manual review**: every pair is kept, edited or dropped
7. **Final test set**: the reviewed pairs are saved for the RAG system

**Setup.** The Stage 1 outputs are read from the shared Google Drive and the Stage 1 lexicon from the repository. Every step saves its result in `QA_PATH` (Drive) and loads it at later runs; this matters most for the passage selection, which must never change once the questions are written from it.

In [ ]:
# anatoolbox
%pip install -q "anatoolbox[embeddings] @ git+https://github.com/umagira/anatoolbox"

from pathlib import Path
import json
import re
import subprocess
import time

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 120)
SEED = 42

# Clone the project repository
REPO = Path("/content/computational-language-technology-hs26")
if REPO.exists():
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/jihyeon26/computational-language-technology-hs26.git {REPO}

# Mount Google Drive (input data and outputs)
from google.colab import drive
drive.mount("/content/drive")
# Inputs: Stage 1 outputs
DATA_DIR = Path("/content/drive/MyDrive/HSLU/CLT/data/processed")

OUT_DIR = Path("/content/drive/MyDrive/HSLU/CLT/extraction_outputs/stage3/wp1") # outputs of Part 1
QA_PATH = OUT_DIR / "qa_dataset" # Q&A test set
RAG_DIR = Path("/content/drive/MyDrive/HSLU/CLT/extraction_outputs/stage3/wp1/rag") # RAG outputs

for directory in (QA_PATH, RAG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Lexicon of Stage 1
lexicon = json.loads((REPO / "artifacts/stage1_wp1/lexicon/open_models_lexicon.json").read_text(encoding="utf-8"))
FAMILY_RX = {f["label"]: re.compile(f["rx"]) for f in lexicon["families"].values()}
CONCEPT_RX = {name: re.compile(c["rx"]) for name, c in lexicon["concepts"].items()}

# Matching rules of Stage 1 (stage1_final.ipynb): special dashes and quotes are replaced before matching,
# and "Llama" inside another product name (Llama Nemotron, Llama Stack, LlamaCon) is not the Llama family
MATCH_MAP = str.maketrans({"‐": "-", "‑": "-", "‒": "-", "–": "-", "−": "-", "‘": "'", "’": "'", "“": '"', "”": '"'})
FAMILY_EXCEPTIONS = {"Llama": re.compile(r"\bLlama[\w.\- ]{0,15}?(?:Nemotron|Stack)\b|\bLlamaCon\b")}

def family_view(name, text):
    # Text in which a family is searched: matching view, without the product names that contain the family name
    view = text.translate(MATCH_MAP)
    return FAMILY_EXCEPTIONS[name].sub(" ", view) if name in FAMILY_EXCEPTIONS else view

# File names of Q&A
PASSAGES_PATH = QA_PATH / "passages_for_qa.csv"
DRAFT_PATH = QA_PATH / "qa_draft.csv"
REVIEWED_PATH = QA_PATH / "qa_reviewed.csv"
FINAL_JSON = QA_PATH / "qa_reviewed.json"



  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.1 Track Articles
Stage 1 gave every article a relevance tier for the open-models track. The test set uses the core articles (`om_tier == "core"`), the same collection as the RAG corpus, so that every question then comes from an article the RAG can retrieve. The relevance tiers of Stage 1 are:
| Tier | Meaning | Used here |
| --| --| --|
| `core`| article is about open models | yes|
| `mention`| article mainly deals with another topic and only names open models| no: a question drawn from it would rarely be about the track|
|`reference`| model documentation pages (API reference), not media articles| no: already excluded in Stage 2|
| `none`| no open-model term | no |

In addition, the Stage 1 knowledge graph was built from the core articles. The text-only RAG uses the same articles, so that it can be compared with the graph-enhanced RAG on the same evidence.

Two files are combined:
- `corpus_clean.parquet`: the cleaned text, publication date, quarter, source domain and the identifiers of every article;
- `open_models_core.parquet`: the dominant topic of each core article (Stage 1 topic model), used here as the subtopic of the article.

Each article keeps two identifiers: `doc_id`, the identifier used in previous stages, and `src_row`, its row number.

In [ ]:
# Core articles
corpus = pd.read_parquet(DATA_DIR / "corpus_clean.parquet", columns=["doc_id","src_row","date","quarter","domain","title","text","om_tier"])
articles = corpus[corpus["om_tier"] == "core"].drop(columns="om_tier")

# Subtopic of each article
topics = pd.read_parquet(DATA_DIR / "open_models_core.parquet", columns=["doc_id","dominant_topic_label"])
articles = articles.merge(topics, on="doc_id", how="left").rename(columns={"dominant_topic_label":"topic"})
articles['date'] = pd.to_datetime(articles["date"]).dt.strftime("%Y-%m-%d")

print(f"Core articles: {len(articles):,} | topics: {articles["topic"].nunique()}"
f" | without topic: {articles["topic"].isna().sum()}")
display(articles["quarter"].value_counts().sort_index().rename("articles").to_frame().T)

Core articles: 1,210 | topics: 16 | without topic: 0


quarter,2024Q3,2024Q4,2025Q1,2025Q2,2025Q3,2025Q4,2026Q1,2026Q2,2026Q3
articles,22,104,349,156,103,90,99,179,108


The 1,210 core articles cover 16 topics. They are not distributed evenly across the quarters: they peak in 2025Q1, followed by 2025Q2 and the number then decreases until 2026Q1 and rises again in 2026Q2. Because of this uneven distribution, a random sample of passages would be dominated by 2025Q1, which is why the passages will be selected with an equal share per quarter.

## 1.2 Passages
A question is written from a passage of one or a few consecutive paragraphs, not from a whole article: long enough to contain a fact with its context, short enough to check during the review, and close in size to the chunks the RAG system retrieves.

**How the articles are cut**: whole paragraphs are joined until a passage has more than `MIN_WORDS` (120) words, without going over `MAX_WORDS` (250); a paragraph longer than `MAX_WORDS` is first cut at sentence boundaries. Pieces that stay too short and single sentences longer than `MAX_WORDS` are dropped.

Each passage gets an identifier `passage_id` = `doc_id` + its position (`_p00`, `_p01`, …), so it can always be traced back to its article.

In [ ]:
# Size of a passage
MIN_WORDS = 120
MAX_WORDS = 250

# End of a sentence
SENT_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9\"'(])")

def n_words(text):
  return len(text.split())

def units(text):
  # Paragraph of an article
  for paragraph in (p.strip() for p in text.split("\n")):
    if not paragraph:
      continue
    if n_words(paragraph) <= MAX_WORDS:
      yield paragraph
    else:
      yield from (s.strip() for s in SENT_SPLIT.split(paragraph) if s.strip())

def split_passages(text):
  # Join consecutive units until passage has min_words but less than max_words
  passages = []
  current = []
  for unit in units(text):
    if current and n_words(" ".join(current + [unit])) > MAX_WORDS:
      passages.append(current)
      current = []
    current.append(unit)
    if n_words(" ".join(current)) > MIN_WORDS:
      passages.append(current)
      current = []
  # Drop short and long pieces
  passages = [p for p in passages if MIN_WORDS <= n_words(" ".join(p)) <= MAX_WORDS]
  return ["\n\n".join(p) for p in passages]

# All passages of all core articles, with the information of their article
rows = []
for art in articles.itertuples(index=False):
    for i, passage in enumerate(split_passages(art.text)):
        rows.append({"passage_id": f"{art.doc_id}_p{i:02d}", "doc_id": art.doc_id, "src_row": art.src_row,
                     "date": art.date, "quarter": art.quarter, "domain": art.domain, "title": art.title,
                     "topic": art.topic, "position": i, "n_words": n_words(passage), "text": passage})
passages = pd.DataFrame(rows)

print(f"Passages: {len(passages):,} from {passages['doc_id'].nunique():,} articles "
      f"(median {passages.groupby('doc_id').size().median():.0f} per article)")
display(passages["n_words"].describe().round(0).to_frame("words per passage").T)

Passages: 12,767 from 1,210 articles (median 8 per article)


,count,mean,std,min,25%,50%,75%,max
words per passage,12767.0,150.0,25.0,120.0,131.0,144.0,162.0,250.0


In [ ]:
def count_families(text):
    # Open-model families found in the text
    counts = {name: sum(1 for _ in rx.finditer(family_view(name,text))) for name, rx in FAMILY_RX.items()}
    return ", ".join(f"{name} ({n})" for name, n in counts.items() if n) or "-"

# Every passage of one example article
example = passages[passages["doc_id"] == passages["doc_id"].iloc[0]]
print(f"Example: '{example['title'].iloc[0]}' -> {len(example)} passages")
display(pd.DataFrame({
    "passage": example["passage_id"],
    "words": example["n_words"],
    "paragraphs": example["text"].str.count("\n\n") + 1,
    "open-model families (occurrences)": [count_families(t) for t in example["text"]],
    "start of the passage": example["text"].str[:80] + "...",
}).reset_index(drop=True))

Example: 'Deploying your Llama Model via vLLM using SageMaker Endpoint' -> 6 passages


,passage,words,paragraphs,open-model families (occurrences),start of the passage
0,doc_c44fd663ac_p00,144,3,Llama (1),"In any machine learning project, the goal is to train a model that can be used b..."
1,doc_c44fd663ac_p01,130,4,-,DJL (Deep Java Library) is an open-source library developed by AWS used to devel...
2,doc_c44fd663ac_p02,168,3,-,The three earlier stated components are viewable on SageMaker’s page and they ea...
3,doc_c44fd663ac_p03,137,4,-,"Optionally, you can also define the subnets, and security group of the managed i..."
4,doc_c44fd663ac_p04,138,4,-,You may locate the URL in the AWS Management Console > SageMaker > Endpoints > [...
5,doc_c44fd663ac_p05,138,3,-,"Note that to use chat completion, the request payload must be within the message..."


The example article is cut into 6 passages of 130 to 168 words, each made of 3 or 4 paragraphs. Although the article is about deploying a Llama model, Llama appears only once. This shows that a passage of an article is therefore not necessarily about the track: drawing questions from all passages would bring off-track questions into the test set, which is why the passages are filtered in the next section.

## 1.3 Track passages
A passage is kept only if it mentions an open-model family or an openness concept of the lexicon of Stage 1, using the same regular expression:
- **families** (e.g. Llama, Qwen, Mistral, DeepSeek, Gemma): the patterns also recognise version names such as *Llama 3.1* or *Qwen3*;
- **openness concepts** (e.g. *open-weight*, *open-source model*): the pattern for open source only counts it when it refers to a model (*open-source model*, *open-source LLM*), not to software or a community.

In [ ]:
def find_families(text):
    # Open-model families mentioned in a text - stage 1 matching rules
    return [name for name, rx in FAMILY_RX.items() if rx.search(family_view(name, text))]

def find_concepts(text):
    # Openness concepts mentioned in a text - matching view of stage 1
    return [name for name, rx in CONCEPT_RX.items() if rx.search(text.translate(MATCH_MAP))]

passages["families"] = [find_families(t) for t in passages["text"]]
passages["concepts"] = [find_concepts(t) for t in passages["text"]]
passages["on_track"] = passages["families"].str.len().gt(0) | passages["concepts"].str.len().gt(0)

track_passages = passages[passages["on_track"]].reset_index(drop=True)
print(f"Track passages: {len(track_passages):,} of {len(passages):,} ({passages['on_track'].mean():.0%}) "
      f"from {track_passages['doc_id'].nunique():,} articles")
print(f"Families found: {track_passages['families'].explode().nunique()} | "
      f"passages with a family: {track_passages['families'].str.len().gt(0).mean():.0%}")
display(track_passages["families"].explode().value_counts().head(15).rename("passages").to_frame().T)

Track passages: 5,988 of 12,767 (47%) from 1,205 articles
Families found: 35 | passages with a family: 86%


families,DeepSeek,Llama,Qwen,Gemma,Mistral,Kimi,Nemotron,GLM,Phi,gpt-oss,Stable Diffusion / StableLM,MiniMax,OLMo / Molmo / Tülu,Granite,FLUX
passages,2023,1004,733,688,484,331,325,199,135,131,119,107,63,58,54


Less than half of the passages (47%) mention an open-model family or an openness concept. The track passages mention 35 different families, dominated by DeepSeek and Llama; 86% of them mention at least one family, while the other 14% are kept for an openness concept only.

As for the quarters, the families are unevenly distributed: a random sample of passages would be dominated by DeepSeek and Llama, and small families might not appear at all. The passages are therefore selected by rules that spread them over periods, subtopics and families in the next section.

## 1.4 Selection of 150 passages
The 150 passages are selected by rules, so that every period, every subtopic and many families are covered:
| Rule | How it works |
|---|---|
| **Periods** | the passages are shared equally between the quarters; a quarter with too few articles gives all it has, and the rest goes to the others |
| **Subtopics** | within a quarter, the next passage comes from the topic with the fewest passages selected so far |
| **Families** | within that topic, a passage mentioning a family not yet covered is preferred |
| **One passage per article** | avoids closely related questions and covers more articles |
| **Fixed seed** | the candidates are shuffled once, so the selection is the same at every run |

The selection is saved in `passages_for_qa.csv`and loaded at every later run.

In [ ]:
N_PASSAGES = 150

def allocate(total, available):
    # Equal share per quarter; a quarter with fewer candidates gives all it has, the rest goes to the others
    share, left, todo = {}, total, dict(available)
    while todo:
        equal = left // len(todo)
        small = {q: n for q, n in todo.items() if n <= equal}
        if not small:                                        # every remaining quarter can give its share
            for i, q in enumerate(sorted(todo)):
                share[q] = equal + (1 if i < left - equal * len(todo) else 0)
            break
        for q, n in small.items():                           # quarters with too few articles give all they have
            share[q] = n
            left -= n
            del todo[q]
    return share

def select(candidates, n_total):
    # Select n_total passages: equal share per quarter, topics in turn, new model families first
    pool = candidates.sample(frac=1, random_state=SEED)
    share = allocate(n_total, pool.groupby("quarter")["doc_id"].nunique().to_dict())
    topic_count = {t: 0 for t in pool["topic"].unique()}                  # passages per topic in the selection
    covered_families, used_docs, chosen = set(), set(), []
    for quarter in sorted(share):
        quarter_pool = pool[pool["quarter"] == quarter]
        for _ in range(share[quarter]):
            quarter_pool = quarter_pool[~quarter_pool["doc_id"].isin(used_docs)]   # one passage per article
            if quarter_pool.empty:
                break
            # 1. the topic of this quarter with the fewest passages in the selection so far
            topic = min(quarter_pool["topic"].unique(), key=lambda t: (topic_count[t], t))
            in_topic = quarter_pool[quarter_pool["topic"] == topic]
            # 2. within it, the first passage that mentions a family not covered yet (else the first one)
            new_family = in_topic["families"].map(lambda fams: bool(set(fams) - covered_families))
            row = in_topic[new_family].iloc[0] if new_family.any() else in_topic.iloc[0]
            chosen.append(row)
            topic_count[topic] += 1
            covered_families.update(row["families"])
            used_docs.add(row["doc_id"])
    return pd.DataFrame(chosen).sort_values(["date", "passage_id"]).reset_index(drop=True)

# Selection created once, then always loaded
if PASSAGES_PATH.exists():
    selected = pd.read_csv(PASSAGES_PATH, converters={"families": json.loads, "concepts": json.loads})
    print(f"Selection loaded: {len(selected)} passages ({PASSAGES_PATH.name})")
else:
    selected = select(track_passages, N_PASSAGES)
    to_save = selected.drop(columns=["on_track"]).assign(families=selected["families"].map(json.dumps),
                                                        concepts=selected["concepts"].map(json.dumps))
    to_save.to_csv(PASSAGES_PATH, index=False)
    print(f"Selection created and saved: {len(selected)} passages ({PASSAGES_PATH.name})")

Selection loaded: 150 passages (passages_for_qa.csv)


In [ ]:
# Periods
by_quarter = pd.DataFrame({"available articles": track_passages.groupby("quarter")["doc_id"].nunique(),
                           "selected passages": selected["quarter"].value_counts()}).fillna(0).astype(int)
display(by_quarter.T)

quarter,2024Q3,2024Q4,2025Q1,2025Q2,2025Q3,2025Q4,2026Q1,2026Q2,2026Q3
available articles,22,103,349,152,103,90,99,179,108
selected passages,17,17,17,17,17,17,16,16,16


The passages are balanced across the quarters; between 16 and 17 passages.

In [ ]:
# Subtopics
by_topic = pd.DataFrame({"available articles": track_passages.groupby("topic")["doc_id"].nunique(),
                         "selected passages": selected["topic"].value_counts()}).fillna(0).astype(int)
display(by_topic.sort_values("selected passages", ascending=False))

,available articles,selected passages
topic,,
T04 claude · agent · cod,126,10
T03 blackwell · gpu · workload,52,10
T07 china · united_states · chinese,81,10
T05 memory · attention · token,75,10
T08 nemotron · nvidia · nvidia nemotron,63,10
T10 api · file · python,105,10
T11 english · multilingual · instruction,44,10
T02 deepseek · r1 · chinese,103,9
T06 image · video · text,58,9


Every topic receives between 8 and 10 passages.

In [ ]:
# Model families
all_families = set(track_passages["families"].explode().dropna())
sel_families = selected["families"].explode().dropna()
print(f"Families covered: {sel_families.nunique()} of {len(all_families)} found in the track passages")

# Families not covered
missing = sorted(all_families - set(sel_families))
if missing:
    mentions = track_passages.explode("families")
    mentions = mentions[mentions["families"].isin(missing)]
    display(mentions.groupby("families").agg(passages=("passage_id", "size"), articles=("doc_id", "nunique")))
else:
    print("Families not covered: none")


Families covered: 34 of 35 found in the track passages


,passages,articles
families,,
Snowflake Arctic,1,1


The selection covers 34 of the 35 open-model families; only *Snowflake Arctic* is missing as it appears in a single passage from one article, which was not reached by the selection rules.

## 1.5 Generation of the Q&A pairs
The questions are drafted by Claude Opus 5.5 (Anthropic), used through Claude Code outside this notebook. The model received each selected passage (`passages_for_qa.csv`) with its title, date and identifiers, and returned `qa_draft.csv`, loaded below.

**Questions types** (two per passage, of two different types, each type about one fifth of the test set):
| Type | What it tests |
|---|---|
| **factual** | a single fact of the passage |
| **temporal** | a date, a period or an order of events |
| **relational** | a link between two entities (developer, partner, base model, competitor) |
| **analytical** | a reason, a cause or a consequence |
| **comparative** | a comparison of models, companies or results |

**Rules given to the model**:
- Write two questions, of two different types, chosen among the five types above according to what the passage actually contains
- Each question must be answerable from the passage alone (plus its publication date for temporal questions), without outside knowledge.
- Each question must be self-contained: it names the model, company or event it is about, and never refers to "the article", "the passage" or "this model", since a RAG user asks without seeing any source.
- The question must not contain its own answer.
- The reference answer is short (one or two sentences), complete and only uses facts from the passage.
- The evidence is a sentence copied word for word from the passage that supports the answer, so that the answer can be checked quickly during the review.
- Questions are phrased as a user would ask them: short, one question at a time, without referring to the source (no "according to the article" or "in a report"), and without copying the wording of the passage beyond the names needed to identify the model, company or event.

In [ ]:
# Drafted questions
draft = pd.read_csv(DRAFT_PATH, dtype={"review_status": str, "review_note": str})

print(f"Draft loaded: {len(draft)} questions from {draft['passage_id'].nunique()} passages ({DRAFT_PATH.name})")

display(draft["type"].value_counts().rename("questions").to_frame().T)
display(draft[["qa_id", "type", "question", "reference_answer"]].head(6))

Draft loaded: 300 questions from 150 passages (qa_draft.csv)


type,factual,analytical,relational,comparative,temporal
questions,69,62,59,57,53


,qa_id,type,question,reference_answer
0,q001,factual,How can I deploy a Llama model with vLLM on AWS?,"AWS SageMaker endpoint, with its DJL (Deep Java Library) image."
1,q002,relational,Which company is behind SageMaker?,"AWS (Amazon Web Services): SageMaker is an AWS service for managing the machine learning lifecycle, and its inferenc..."
2,q003,temporal,When did Google launch DataGemma?,On 12 September 2024: an article published that day reports that Google debuted DataGemma that day.
3,q004,relational,Which model family is DataGemma built on?,Google's Gemma family of open models; DataGemma grounds its answers in data from Google's Data Commons platform.
4,q005,factual,How fast can SambaNova run Llama 3.1 405B?,More than 100 tokens per second; the 70B model runs at up to 580 tokens per second.
5,q006,comparative,How is SambaNova's way of running Llama different from Groq and Cerebras?,"SambaNova says Groq and Cerebras store model weights on SRAM, which requires many chips, whereas SambaNova runs all ..."


## 1.6 Manual review
As an LLM can misread a passage, write ambiguous question or give an answer that is not fully supported, every drafted question was manually reviewed by the team members against these criteria:

| Criterion | Question to ask |
|---|---|
| **Clarity** | Is the question understandable without the passage, with one clear answer? |
| **Correctness** | Is the reference answer true according to the passage, and complete? |
| **Answerability** | Can the answer be found in the passage (or its date), without outside knowledge? |
| **Diversity** | Does the question ask a different fact from the other questions? |
| **Coverage** | Is the question about the open-models track, and is its type right? |
| **Naturalness** | Would a user ask it without having read the article? |

Each question is marked `keep`, `edit` (corrected in the row) or `drop` (off-track, too general, duplicate or not answerable), with a `review_note` for every `edit` and `drop`; the result is saved as `qa_reviewed.csv`.

### 1.6.1 Complementary temporal questions
After the first review, temporal questions were the smallest group (28 of 218), and most dates written in the 150 passages were already used. Fifteen extra passages with an explicit date are therefore selected below, with the same rules as the first selection, among the track passages that come from unused articles and contain a sentence with both an explicit date (e.g. *May 2024*, *in 2023*) and an open-model family or openness concept. Thirteen temporal questions were drafted by Claude Opus 5.5 from their dated sentences and reviewed with the same criteria.

In [ ]:
# Explicit date
DATE_RX = re.compile(r"\b(?:January|February|March|April|May|June|July|August|September|October|November|December"
                     r"|Jan\.|Feb\.|Mar\.|Apr\.|Aug\.|Sept?\.|Oct\.|Nov\.|Dec\.)\s+(?:\d{1,2},?\s+)?(?:19|20)\d\d\b"
                     r"|\b(?:in|since|from|by|until|before|after|early|late|mid)[\s-]+(?:19|20)\d\d\b(?![\-/]\d)")
SENTENCE_RX = re.compile(r"[^.!?\n]+[.!?]?")

def dated_track_sentences(text):
    # Sentences that contain both an explicit date and an open-model family or openness concept
    return [s.strip() for s in SENTENCE_RX.findall(text)
            if DATE_RX.search(s) and (find_families(s) or find_concepts(s))]

N_EXTRA = 15
EXTRA_PATH = QA_PATH / "passages_temporal_extra.csv"

# Extra passages for temporal questions: created once then always loaded
if EXTRA_PATH.exists():
    extra = pd.read_csv(EXTRA_PATH, converters={"families": json.loads, "concepts": json.loads})
    print(f"Extra selection loaded: {len(extra)} passages ({EXTRA_PATH.name})")
else:
    # Candidates: track passages with a dated sentence about the track, from articles not used in the first selection
    candidates = track_passages[~track_passages["doc_id"].isin(selected["doc_id"])].copy()
    candidates["dated_sentences"] = [dated_track_sentences(t) for t in candidates["text"]]
    candidates = candidates[candidates["dated_sentences"].str.len() > 0]
    extra = select(candidates, N_EXTRA)                  # same rules as the first selection (quarters, topics, families)
    extra["dated_sentences"] = extra["dated_sentences"].map(" | ".join)
    extra.drop(columns=["on_track"], errors="ignore").assign(
        families=extra["families"].map(json.dumps), concepts=extra["concepts"].map(json.dumps)).to_csv(EXTRA_PATH, index=False)
    print(f"Extra selection created and saved: {len(extra)} passages from {len(candidates):,} candidates ({EXTRA_PATH.name})")

Extra selection loaded: 15 passages (passages_temporal_extra.csv)


## 1.7 Final test set
The reviewed file (`qa_reviewed.csv`) is loaded and checked, the dropped questions are removed, and the coverage of the final test set is shown by type, quarter, subtopic and family. The final test set is saved as `qa_reviewed.json` for the RAG part: each question keeps its `id`, `type`, `question`, `reference_answer`, `source_ids` (the `doc_id` of its article, as a list so that other articles can be added), `date`, `passage_id`, `quarter` and `topic`.

In [ ]:
TYPES = ["factual", "temporal", "relational", "analytical", "comparative"]


reviewed = pd.read_csv(REVIEWED_PATH, dtype={"review_status": str, "review_note": str})
reviewed["review_status"] = reviewed["review_status"].str.strip().str.lower()

# Check that every row have been checked
bad_status = ~reviewed["review_status"].isin(["keep", "edit", "drop"])
bad_type = ~reviewed["type"].isin(TYPES)
assert not bad_status.any(), f"{bad_status.sum()} rows without a valid review_status (keep / edit / drop)"
assert not bad_type.any(), f"{bad_type.sum()} rows with an unknown type"

# Result of the check
print("Review:", reviewed["review_status"].value_counts().to_dict())

# Final test set
final = reviewed[reviewed["review_status"] != "drop"].reset_index(drop=True)
print(f"\nFinal test set: {len(final)} questions from {final['passage_id'].nunique()} passages")

Review: {'keep': 140, 'drop': 100, 'edit': 73}

Final test set: 213 questions from 135 passages


In [ ]:
# Question types
display(pd.crosstab(final["type"], final["quarter"], margins=True, margins_name="total"))

# Subtopics and families of the questions (through their passage)
families = final["passage_id"].map(selected.set_index("passage_id")["families"])
print(f"Topics covered: {final['topic'].nunique()} of {articles['topic'].nunique()} | "
      f"families covered: {families.explode().nunique()} | articles: {final['doc_id'].nunique()}")
display(final["topic"].value_counts().rename("questions").to_frame().T)

quarter,2024Q3,2024Q4,2025Q1,2025Q2,2025Q3,2025Q4,2026Q1,2026Q2,2026Q3,total
type,,,,,,,,,,
analytical,2,3,2,4,3,4,3,4,2,27
comparative,7,4,3,4,4,5,5,6,5,43
factual,10,9,7,5,4,6,5,7,4,57
relational,4,6,6,4,5,5,9,4,5,48
temporal,3,4,6,4,4,4,4,4,5,38
total,26,26,24,21,20,24,26,25,21,213


Topics covered: 16 of 16 | families covered: 34 | articles: 135


topic,T05 memory · attention · token,T08 nemotron · nvidia · nvidia nemotron,T11 english · multilingual · instruction,T13 meta · zuckerberg · llama,T04 claude · agent · cod,T02 deepseek · r1 · chinese,T09 gemma · gemma model · model gemma,T03 blackwell · gpu · workload,T15 qwen · qwen qwen · context length,T01 reasoning · reinforcement · reinforcement learning,T16 people · ask · find,T07 china · united_states · chinese,T06 image · video · text,T10 api · file · python,T12 stock · investor · growth,T14 enterprise · innovation · solution
questions,19,19,18,15,15,15,14,13,13,13,11,11,10,10,10,7


The final test set contains 213 questions from 135 passages. All quarters are represented, as all 16 subtopics and 34 open-model families.

In [ ]:
# Save the final test set
records = [{"id": r.qa_id, "type": r.type, "question": r.question, "reference_answer": r.reference_answer,
            "source_ids": [r.doc_id], "date": r.date, "passage_id": r.passage_id,
            "quarter": r.quarter, "topic": r.topic} for r in final.itertuples(index=False)]
FINAL_JSON.write_text(json.dumps(records, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved: {FINAL_JSON.name} ({len(records)} questions)")
display(pd.DataFrame(records).head(5))

Saved: qa_reviewed.json (213 questions)


,id,type,question,reference_answer,source_ids,date,passage_id,quarter,topic
0,q001,factual,Which AWS service can I use to deploy a Llama model with vLLM?,"AWS SageMaker endpoint, with its DJL (Deep Java Library) image.",[doc_c44fd663ac],2024-09-12,doc_c44fd663ac_p00,2024Q3,T10 api · file · python
1,q003,temporal,When did Google launch DataGemma?,On 12 September 2024: an article published that day reports that Google debuted DataGemma that day.,[doc_d57f4c755c],2024-09-12,doc_d57f4c755c_p00,2024Q3,T09 gemma · gemma model · model gemma
2,q004,relational,Which model family is DataGemma built on?,Google's Gemma family of open models; DataGemma grounds its answers in data from Google's Data Commons platform.,[doc_d57f4c755c],2024-09-12,doc_d57f4c755c_p00,2024Q3,T09 gemma · gemma model · model gemma
3,q005,factual,How fast can SambaNova run Llama 3.1 405B?,More than 100 tokens per second; the 70B model runs at up to 580 tokens per second.,[doc_3f5ee378c1],2024-09-13,doc_3f5ee378c1_p04,2024Q3,T03 blackwell · gpu · workload
4,q006,comparative,How is SambaNova's way of running Llama different from Groq and Cerebras?,"According to McGonnell, Groq and Cerebras store model weights on SRAM, which reequires many chips, whereas SambaNova...",[doc_3f5ee378c1],2024-09-13,doc_3f5ee378c1_p04,2024Q3,T03 blackwell · gpu · workload


## 1.8 Saved outputs
| File  | Content |
|---|---|
| `passages_for_qa.csv` | the 150 passages of the first selection, with their article, date, quarter, topic and families |
| `passages_temporal_extra.csv` | the 15 extra passages with an explicit date, with their dated sentences |
| `qa_draft.csv` | the questions drafted by the LLM before the review and temporal questions from the extra passages |
| `qa_reviewed.csv` | all drafted questions after the manual review |
| `qa_reviewed.json` | the final test set (kept and edited questions only) |

# 2. RAG Implementation
This part builds the text-only RAG baseline, following the skeleton ([`course_project_skeleton.ipynb`](https://github.com/umagira/anatoolbox/blob/main/notebooks/course_project_skeleton.ipynb) and its toolbox `anatoolbox`. The steps and tools of the skeleton are kept, so that the other parts of Stage 3 can reuse them unchanged; they are adapted to the track: the core articles of Stage 1 as corpus, the reviewed test set of Section 1, and a local open model for generation.

**Pipeline**
1. **Language model**: an open model (Qwen3:4b-instruct) runs locally in Colab
2. **Corpus**: the 1,210 core open-models articles of Stage 1, the same articles as the Q&A test set
3. **Chunks and embeddings**: the articles are cut into overlapping chunks, indexed for keyword search and embedded for semantic search
4. **Retrieval strategies**: sparse, dense and hybrid search, compared on example questions of the test set
5. **Grounded generation**: the model answers only from the retrieved passages, cites them and uses their publication dates

Every long step (embeddings) is saved in `RAG_DIR` on Google Drive and loaded at later runs.

**Setup.** The RAG model, Qwen3-4B-Instruct-2507, runs locally in Colab with Ollama (downloaded at each new session; Section 2 was run on an L4 GPU).

In [ ]:
# Anatoolbox tools
import anatoolbox
from anatoolbox import ToolContext, resolve_tools

anatoolbox.register_reference_tools(replace=True)
ctx = ToolContext()
ingest, chunk, retrieve, synthesize = resolve_tools(
    ["ingest_corpus", "chunk_by_size", "retrieve_passages", "synthesize_answer"])
print("anatoolbox", anatoolbox.__version__)

# Install and start Ollama
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
ollama_server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

RAG_MODEL = "qwen3:4b-instruct"
!ollama pull {RAG_MODEL}

anatoolbox 0.1.0
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [ ]:
from anatoolbox.llm_client import call_llm_text, configure_llm, model_for

# Point every anatoolbox tool at the local Ollama server
configure_llm(base_url="http://localhost:11434/v1", model=RAG_MODEL, timeout=180, max_retries=2)

# Check that the model answers
started = time.perf_counter()
reply = call_llm_text("Reply with one word.", "Say: ready", model=model_for("default"), max_tokens=20)
print(f"{model_for('default')} answered {reply!r} in {time.perf_counter() - started:.1f} s")

qwen3:4b-instruct answered 'ready' in 2.9 s


## 2.1 Corpus
The RAG corpus is the 1,210 core open-models articles of Stage 1, the same collection as the test set (Section 1.1). Each article keeps its Stage 1 `doc_id` as `id`, so that a retrieved passage can be matched to the `source_ids` of the test set; its title, publication date, URL and domain are kept as metadata and shown to the language model with every passage.

In [ ]:
# Load corpus
corpus = pd.read_parquet(DATA_DIR / "corpus_clean.parquet",
                         columns=["doc_id", "date", "domain", "url", "title", "text", "om_tier"])
core = corpus[corpus["om_tier"] == "core"]
core_csv = pd.DataFrame({"id": core["doc_id"], "title": core["title"], "text": core["text"],
                         "date": pd.to_datetime(core["date"]).dt.strftime("%Y-%m-%d"),
                         "url": core["url"], "domain": core["domain"]})
CORE_CSV = RAG_DIR / "core_articles.csv"
core_csv.to_csv(CORE_CSV, index=False)

# Load the articles into anatoolbox
rag_articles = ingest.run({"path": str(CORE_CSV), "name": "core_articles"}, context=ctx)
print(f"{rag_articles['records']:,} core articles, {core_csv['date'].min()} to {core_csv['date'].max()}")

1,210 core articles, 2024-09-12 to 2026-08-31


### 2.1.1 Test set
The reviewed Q&A test set (`qa_reviewed.json`) is loaded to choose example questions and to verify that every source article is in the corpus

In [ ]:
# Load qa reviewed
EVAL_SET = json.loads((QA_PATH / "qa_reviewed.json").read_text(encoding="utf-8"))
corpus_ids = set(core_csv["id"])
missing = [q["id"] for q in EVAL_SET if not set(q["source_ids"]) <= corpus_ids]
print(f"{len(EVAL_SET)} test questions | source articles missing from the corpus: {len(missing)}")
display(pd.Series([q["type"] for q in EVAL_SET]).value_counts().rename("questions").to_frame().T)

213 test questions | source articles missing from the corpus: 0


,factual,relational,comparative,temporal,analytical
questions,57,48,43,38,27


## 2.2 Chunks
A question is usually answered by a few sentences, so the articles are cut into chunks of fixed size, the baseline method of the toolbox:
- **200 words**: enough for a fact with its context, while keeping several chunks within the prompt;
- **40 words of overlap**, so that a statement cut at a boundary appears whole in one of the two chunks;
- **the article title added to every chunk** (`contextualize="title"`): a chunk often does not name what it is about (*"the model"*), and the title restores that context, as in the review of the test set, where the title counted as part of the source.

The passages of section 1 are not reused: they only cover the track passages, while the RAG system must search the full text of every article.

In [ ]:
# Parameters
CHUNK_ARGS = {"size": 200, "overlap": 40, "contextualize": "title"}

# Corpus
from anatoolbox.corpus import get_corpus

chunks = chunk.run({"input": rag_articles, **CHUNK_ARGS}, context=ctx)
print(f"{chunks['chunks']:,} chunks from {chunks['articles']:,} articles | words per chunk: {chunks['tokens']}")
display(pd.DataFrame(get_corpus(chunks["corpus"]).records[:3])[["id", "source_id", "title", "date", "chunk_text"]])

12,720 chunks from 1,210 articles | words per chunk: {'min': 41, 'median': 200.0, 'max': 200}


,id,source_id,title,date,chunk_text
0,doc_c44fd663ac#0,doc_c44fd663ac,Deploying your Llama Model via vLLM using SageMaker Endpoint,2024-09-12,"In any machine learning project, the goal is to train a model that can be used by others to derive a good prediction..."
1,doc_c44fd663ac#1,doc_c44fd663ac,Deploying your Llama Model via vLLM using SageMaker Endpoint,2024-09-12,"docker images, including vLLM [ 2 ]. This image is used in the SageMaker endpoint.\n\nvLLM, introduced in 2023, is a..."
2,doc_c44fd663ac#2,doc_c44fd663ac,Deploying your Llama Model via vLLM using SageMaker Endpoint,2024-09-12,retrieve the LLM model from your S3 bucket not covered in AWS’s existing documentation. The previous example assumes...


## 2.3 Embeddings
To search the chunks, each chunk is turned into a vector once with `all-MiniLM-L6-v2`, the baseline model of the toolbox.

The vectors are saved in `RAG_DIR` and reloaded at later runs. The file name contains the model name, so another embedding model would be embedded again and saved to its own file, instead of reusing these vectors.

In [ ]:
from anatoolbox.corpus import configure_embedder, ensure_embeddings, load_embeddings, save_embeddings
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "all-MiniLM-L6-v2"

def use_embedding_model(name):
  model = SentenceTransformer(name)
  configure_embedder(lambda texts: model.encode(texts, batch_size=64, normalize_embeddings=True,
                                                 show_progress_bar=len(texts) > 1000))

def embed(chunk_result, model_name=None):
  corpus = get_corpus(chunk_result["corpus"])
  model_name = model_name or EMBEDDING_MODEL
  model_slug = re.sub(r"\W+", "_", model_name)
  cache = RAG_DIR / f"embeddings_{corpus.name}_{model_slug}.npy"
  started = time.perf_counter()
  if cache.exists():
    load_embeddings(corpus, cache)
    print(f"Embeddings loaded: {len(corpus):,} vectors ({cache.name})")
  else:
    ensure_embeddings(corpus)
    save_embeddings(corpus, cache)
    print(f"Embeddings computed and saved: {len(corpus):,} chunks in {time.perf_counter() - started:.0f} s ({cache.name})")
  return corpus, cache

use_embedding_model(EMBEDDING_MODEL)
chunk_corpus, EMB_PATH = embed(chunks)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings loaded: 12,720 vectors (embeddings_core_articles_size200_overlap40_title_all_MiniLM_L6_v2.npy)


## 2.4 Retrieval strategies
Three strategies are available through `anatoolbox`:

| Strategy | How it ranks the chunks | Expected strength |
|---|---|---|
| **sparse** | chunks that contain the words of the question, rare words weighing more | exact names, versions and numbers |
| **dense** | cosine similarity between the embeddings of the question and of the chunks | paraphrases and questions worded differently from the article |
| **hybrid** | reciprocal rank fusion: the two rankings are merged, a chunk ranked high by either strategy rises | combines both, without having to weigh their scores |

To see how the strategies behave, they are run on one question of each type from the test set, drawn at random (fixed seed) from five different articles. For each strategy, a table shows the top-5 articles and whether the source article of the question is among them.

This is a first look on five questions, not a measurement: the strategies are measured on the whole test set in the evaluation section.

In [ ]:
def search(question, strategy="hybrid", size=30, **args):
    # Retrieve the `size` best chunks for `question` with the given strategy (sparse, dense or hybrid)
    return retrieve.run({"query": question, "input": chunks, "strategy": strategy, "size": size, **args},
                        context=ctx)

# One example question per type, drawn at random (fixed seed) from different source articles
EXAMPLES, used_types, used_articles = [], set(), set()
for q in pd.DataFrame(EVAL_SET).sample(frac=1, random_state=SEED).to_dict("records"):
    if q["type"] not in used_types and not set(q["source_ids"]) & used_articles:
        EXAMPLES.append(q)
        used_types.add(q["type"])
        used_articles.update(q["source_ids"])

def try_strategy(strategy):
    # For each example question: is the source article in the top-5 articles retrieved with `strategy`?
    rows = []
    for q in EXAMPLES:
        top = search(q["question"], strategy=strategy, size=10)["passages"]
        top_articles = list(dict.fromkeys(p["source_id"] for p in top))[:5]   # top-5 distinct articles
        rank = next((i + 1 for i, a in enumerate(top_articles) if a in q["source_ids"]), None)
        rows.append({"type": q["type"], "question": q["question"],
                     "source in top 5": f"yes (rank {rank})" if rank else "no",
                     "top article": top[0]["title"][:70]})
    return pd.DataFrame(rows)

### 2.4.1 Sparse search
Sparse search ranks the chunks that contain the words of the question with rare words weighing more. It should be strong on exact names, versions and numbers, and weak when the article uses other words than the question.

In [ ]:
display(try_strategy("sparse"))

,type,question,source in top 5,top article
0,factual,Am I allowed to use the Llama name for my product?,no,"Discussion of Join the Gemma 4 Challenge: $ 3,000 prize pool for TEN w"
1,temporal,When did Google release Gemma 3?,no,"Welcome Gemma 3: Google's all new multimodal, multilingual, long conte"
2,relational,How is CrowdStrike using Nemotron Nano?,yes (rank 2),"CrowdStrike, Uber, Zoom Among Industry Pioneers Building Smarter Agent"
3,comparative,Which smaller Jais model outperformed a larger Jais model on OALL v2?,yes (rank 1),The Open Arabic LLM Leaderboard 2
4,analytical,What is the main performance advantage of DiffusionGemma?,yes (rank 1),"DiffusionGemma: Open-Weight Text at 1,000 Tokens/Sec"


Sparse search finds the source article in the top 5 for 3 of the 5 questions: rank 1 for the comparative (Jais) and analytical (DiffusionGemma) questions, and rank 2 for the relational question (CrowdStrike and Nemotron Nano). It misses the source for the factual question about the Llama name: the top article is a discussion of a Gemma 4 challenge. It also misses the source for the temporal question about Gemma 3, although its top article is about the Gemma 3 release (“Welcome Gemma 3…”), which suggests that several articles of the corpus might cover the same event.

### 2.4.2 Dense search
Dense search ranks the chunks by the cosine similarity between their embedding and the embedding of the question. It should find chunks worded differently from the question but may confuse close names.

In [ ]:
display(try_strategy("dense"))

,type,question,source in top 5,top article
0,factual,Am I allowed to use the Llama name for my product?,yes (rank 1),meta-llama/Llama-3.2-3B-Instruct · Hugging Face
1,temporal,When did Google release Gemma 3?,yes (rank 3),Google Gemma 4: Complete Guide
2,relational,How is CrowdStrike using Nemotron Nano?,no,5 Big Moves In CrowdStrike-Nvidia AI Partnership
3,comparative,Which smaller Jais model outperformed a larger Jais model on OALL v2?,yes (rank 1),The Open Arabic LLM Leaderboard 2
4,analytical,What is the main performance advantage of DiffusionGemma?,yes (rank 1),"DiffusionGemma: Open-Weight Text at 1,000 Tokens/Sec"


Dense search finds the source article in the top 5 for 4 of the 5 questions: rank 1 for the factual (Llama name), comparative (Jais) and analytical (DiffusionGemma) questions, and rank 3 for the temporal question about Gemma 3. It misses the source for the relational question: the top article is about the CrowdStrike-Nvidia partnership, but it is not the source article. Compared with sparse search, dense search finds the two sources that sparse search missed (Llama, Gemma 3), but loses the CrowdStrike one.

### 2.4.3 Hybrid search
Hybrid search merges the sparse and dense rankings with reciprocal rank fusion: a chunk ranked high by either strategy rises, without having to weight their scores. It should combine the strengths of both.

In [ ]:
display(try_strategy("hybrid"))

,type,question,source in top 5,top article
0,factual,Am I allowed to use the Llama name for my product?,yes (rank 1),meta-llama/Llama-3.2-3B-Instruct · Hugging Face
1,temporal,When did Google release Gemma 3?,yes (rank 2),Google Gemma 4: Complete Guide
2,relational,How is CrowdStrike using Nemotron Nano?,yes (rank 3),"CrowdStrike, Uber, Zoom Among Industry Pioneers Building Smarter Agent"
3,comparative,Which smaller Jais model outperformed a larger Jais model on OALL v2?,yes (rank 1),The Open Arabic LLM Leaderboard 2
4,analytical,What is the main performance advantage of DiffusionGemma?,yes (rank 1),"DiffusionGemma: Open-Weight Text at 1,000 Tokens/Sec"


Hybrid search finds the source article in the top 5 for all 5 questions: rank 1 for the factual (Llama name), comparative (Jais) and analytical (DiffusionGemma) questions, rank 2 for the temporal question about Gemma 3, and rank 3 for the relational question about CrowdStrike. It combines the results of both strategies: it keeps the sources found by dense search (Llama, Gemma 3) and the one found only by sparse search (CrowdStrike), although the last two are lower in the ranking.

### 2.4.4 Comparison and choice
Sparse search finds the source for three questions and dense search for four, but not the same ones. Sparse search misses the factual and temporal questions, which dense search finds. Dense search misses the relational question, which sparse search finds at rank 2. Hybrid search is the only strategy that finds the source for all five questions. Its ranks are lower than the best single strategy for two questions: Gemma 3 at rank 2 instead of 3 with dense is better, but CrowdStrike is at rank 3 instead of 2 with sparse.

**Verdict**: hybrid search is used for generation, since it is the only strategy that finds the source of all five questions, including the ones missed by sparse search and by dense search. This choice rests on five questions only: it is provisional, and `RETRIEVAL_ARGS` is updated if the evaluation on the whole test set shows that another strategy performs better.

In [ ]:
RETRIEVAL_ARGS = {"strategy": "hybrid", "size": 30}

## 2.5 Grounded generation
The answer is generated by `synthesize_answer`, the generation tool of the toolbox. Its prompt has three parts: a system prompt with the rules (printed below), the question, and the context: at most 6 retrieved passages (2 per article, 1,200 characters each), labelled `[S1]…[Sn]` with their title and publication date.
- **Grounded mode**: the model may only use the passages, and must say when they do not answer the question.
- **Citations**: every sentence must cite its sources; the tool checks them (citations to unknown sources, share of sentences with a citation).
- **Dates**: the model is told to use the publication dates when the question concerns time.

In [ ]:
ANSWER_ARGS = {"mode": "grounded", "max_passages": 6, "max_per_source": 2,
               "max_chars_per_passage": 1200, "max_tokens": 400}

def answer(question, passages=None, **args):
    # Answer `question` from retrieved passages (retrieved with RETRIEVAL_ARGS if none are given)
    passages = passages or search(question, **RETRIEVAL_ARGS)
    return synthesize.run({"question": question, "input": passages, **ANSWER_ARGS, **args}, context=ctx)

print(synthesize.system_prompt({"mode": "grounded"}))

You answer questions using numbered sources from a document collection.

Rules:
- Base every statement on the sources. If they do not contain enough information to answer, say so plainly and answer only what they support.
- Cite with the source labels in square brackets right after the sentence they support, for example [S1] or [S2][S4]. Do not collect citations at the end of the answer.
- Use only the labels you are given. Never invent sources, URLs, names, figures, or dates.
- When the question concerns time — what is new, what changed, when something happened — use the sources' publication dates and mention them.
- Where sources disagree, say so and cite each side.
- Write concise markdown. Do not end with a list of sources; the application adds one.


The same five example question are answered with hybrid search. Each answer is shown with the reference answer of the test set, and the table gives the cited sources with their dates and the citation checks.

In [ ]:
results = []
for q in EXAMPLES:
    started = time.perf_counter()
    result = answer(q["question"])
    sources = {s["label"]: s for s in result["sources"]}
    display(Markdown(f"**[{q['type']}] {q['question']}**  \n{result['answer_markdown']}  \n"
                     f"*Reference answer:* {q['reference_answer']}"))
    results.append({"type": q["type"], "seconds": round(time.perf_counter() - started, 1),
                    "cited sources (date)": ", ".join(f"{label} ({sources[label]['date']})"
                                                      for label in result["cited"] if label in sources),
                    "unknown citations": len(result["unknown_citations"]),
                    "citation coverage": result["citation_coverage"],
                    "source article retrieved": any(s.get("source_id") in q["source_ids"] for s in result["sources"])})
display(pd.DataFrame(results))

**[factual] Am I allowed to use the Llama name for my product?**  
No, you are not allowed to use the "Llama" name for your product without permission or in a way that violates Meta's licensing terms. According to [S1](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct), Meta grants a license to use the name "Llama" only "as required to comply with the last sentence of Section 1.b.i." This means that the use of "Llama" is strictly limited to specific, defined circumstances—such as including "Built with Llama" on a website or product documentation—and does not permit arbitrary or commercial use of the name. Additionally, the license explicitly prohibits using any name or mark owned by or associated with Meta unless required for reasonable and customary use in describing and redistributing the Llama Materials [S2](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct). Therefore, using the "Llama" name for your own product would likely constitute a violation of Meta's brand and licensing policies unless explicitly authorized. [S2](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct)  
*Reference answer:* Only in a limited way: Meta grants no trademark licence except a licence to use "Llama" as required by Section 1.b.i of the Llama licence, and all goodwill from its use goes to Meta.

**[temporal] When did Google release Gemma 3?**  
Google released Gemma 3 on March 12, 2025 [S4](https://huggingface.co/blog/gemma3?utm_source=deepmind.google&utm_medium=referral&utm_campaign=gdm&utm_content=). This is supported by the source "Welcome Gemma 3: Google's all new multimodal, multilingual, long context open LLM" which states that Google released Gemma 3 on that date [S4](https://huggingface.co/blog/gemma3?utm_source=deepmind.google&utm_medium=referral&utm_campaign=gdm&utm_content=). Additionally, another source confirms the release of Gemma 3 in March 2025, noting it as one of Google Cloud’s new AI products launched that month [S2](https://www.crn.com/news/ai/2025/google-cloud-s-10-new-bold-ai-products-gemini-robotics-ai-coach-and-gemma-3). The specific date of March 12, 2025, is directly cited in the Hugging Face blog post [S4](https://huggingface.co/blog/gemma3?utm_source=deepmind.google&utm_medium=referral&utm_campaign=gdm&utm_content=).  
*Reference answer:* In March 2025.

**[relational] How is CrowdStrike using Nemotron Nano?**  
CrowdStrike is using Nemotron Nano by fine-tuning the NVIDIA Nemotron Nano model for cyber defense purposes, specifically to generate investigation queries within the CrowdStrike Falcon LogScale platform. Internal testing has achieved 96% accuracy in generating these queries, providing a natural-language interface that enhances the efficiency of agent investigations [S5](https://blogs.nvidia.com/blog/open-secure-ai-alliance-contributions/). Additionally, CrowdStrike has published research showing that a specialized Nemotron Nano reasoning model outperforms larger models in Security Operations Center (SOC) detection triage, with calibrated logit-based confidence scores enabling measurable, tunable, and auditable autonomous security decisions [S5](https://blogs.nvidia.com/blog/open-secure-ai-alliance-contributions/). The partnership also enables customers to train Nemotron open models using data from CrowdStrike tools via the NeMo Data Designer, allowing for domain-specific fine-tuning and the creation of specialized threat-hunting agents that can be deployed at the edge [S4](https://www.crn.com/news/ai/2025/5-big-moves-in-crowdstrike-nvidia-ai-partnership)[S2](https://www.crn.com/news/ai/2025/5-big-moves-in-crowdstrike-nvidia-ai-partnership).  
*Reference answer:* CrowdStrike is fine-tuning it for cyber defense: internal testing reached 96% accuracy in generating investigation queries in Falcon LogScale.

**[comparative] Which smaller Jais model outperformed a larger Jais model on OALL v2?**  
The smaller Jais model that outperformed a larger Jais model on OALL v2 is the inceptionai/jais-family-30b-8k, which surpasses the larger inceptionai/jais-adapted-70b model on OALL v2 [S1](https://huggingface.co/blog/leaderboard-arabic-v2). This finding is supported by the analysis in Figure 6 of the Open Arabic LLM Leaderboard 2, which shows that while larger models generally achieve higher average scores, this exception occurs in the Jais family [S1](https://huggingface.co/blog/leaderboard-arabic-v2).  
*Reference answer:* inceptionai/jais-family-30b-8k outperformed inceptionai/jais-adapted-70b.

**[analytical] What is the main performance advantage of DiffusionGemma?**  
The main performance advantage of DiffusionGemma is its significantly reduced per-request latency due to parallel text generation, achieved by denoising up to 256 tokens at once rather than generating one token at a time [S5](https://www.thefastmode.com/technology-solutions/48931-google-deepmind-unveils-google-deepmind-diffusiongemma-a-new-era-of-fast-text-generation)[S6](https://www.thefastmode.com/technology-solutions/48931-google-deepmind-unveils-google-deepmind-diffusiongemma-a-new-era-of-fast-text-generation). This parallelism shifts the computational workload from memory-bound to compute-bound, leveraging the strengths of GPU architectures such as NVIDIA Tensor Cores and CUDA, resulting in a speedup that makes it particularly effective for single-user, interactive workloads like real-time chat or on-device assistants [S6](https://www.thefastmode.com/technology-solutions/48931-google-deepmind-unveils-google-deepmind-diffusiongemma-a-new-era-of-fast-text-generation). Compared to standard autoregressive models, DiffusionGemma is described as 4x faster on H100 hardware and delivers 1,000 tokens per second, with a design that enables faster iteration and thinking in blocks, closer to human editing and reasoning processes [S2](https://dev.to/max_quimby/diffusiongemma-open-weight-text-at-1000-tokenssec-16o5)[S4](https://www.ai.google.dev/gemma/docs/diffusiongemma/model_card). This advantage is most pronounced in latency-sensitive, low-to-medium batch size scenarios, such as developer tools and local inference, where it outperforms traditional models in responsiveness [S1](https://dev.to/max_quimby/diffusiongemma-open-weight-text-at-1000-tokenssec-16o5)[S2](https://dev.to/max_quimby/diffusiongemma-open-weight-text-at-1000-tokenssec-16o5).  
*Reference answer:* Lower latency for individual requests rather than higher overall throughput.

,type,seconds,cited sources (date),unknown citations,citation coverage,source article retrieved
0,factual,3.2,"S1 (2024-09-18), S2 (2024-09-18)",0,0.60,True
1,temporal,3.1,"S4 (2025-03-12), S2 (2025-03-14)",0,1.00,True
2,relational,3.2,"S5 (2026-08-04), S4 (2025-10-28), S2 (2025-10-28)",0,0.75,True
3,comparative,2.3,S1 (2025-04-08),0,1.00,True
4,analytical,3.7,"S5 (2026-06-11), S6 (2026-06-11), S2 (2026-06-28), S4 (2026-06-10), S1 (2026-06-28)",0,1.00,True


For the five questions, the source article is among the retrieved passages, and no answer contains an unknown citation: every citation refers to a passage given to the model. Citation coverage ranges from 0.5 (factual) to 1.0, which is reached by the temporal, comparative and analytical answers; the relational answer reaches 0.8. The answers cite between one and three sources, with their publication dates, from 2024-09-18 to 2026-08-04.

## 2.6 Limitations
The examples of this section show several limitations of the baseline (hybrid search) that could be addressed by an enhancement:
- **Close names**: for the question about Gemma 3, dense and hybrid search rank an article about Gemma 4 first, before the source article; a re-ranking step could favour the exact model name
- **General wording**: sparse search, one of the two parts of the hybrid search, misses the question made of common words (the Llama name), and its top article is unrelated (a Gemma 4 challenge, not Llama); hybrid search still finds the source thanks to dense search, but expanding the query with related terms could make sparse search more reliable.
- **Uncited parts**: the citation coverage of the factual and relational answers is below 1.0, so part of these answers is not linked to a source; asking the model to cite every sentence could make the answers easier to check.

## 2.7 RAG outputs
The settings of the baseline are saved in `rag_settings.json`, so that the knowledge-graph integration, the evaluation and the enhancement use exactly the same system. The embeddings are already saved in `RAG_DIR`.

In [ ]:
RAG_SETTINGS = {
    "corpus": {"file": CORE_CSV.name, "articles": rag_articles["records"], "filter": "Stage 1 om_tier == 'core'"},
    "chunks": CHUNK_ARGS,
    "embedding_model": EMBEDDING_MODEL,
    "embeddings_file": EMB_PATH.name,
    "retrieval": RETRIEVAL_ARGS,
    "generation": ANSWER_ARGS,
    "rag_model": {"model": RAG_MODEL, "server": "Ollama, OpenAI-compatible endpoint http://localhost:11434/v1"},
    "test_set": {"file": "qa_reviewed.json", "questions": len(EVAL_SET)},
}
(RAG_DIR / "rag_settings.json").write_text(json.dumps(RAG_SETTINGS, indent=2), encoding="utf-8")
print(json.dumps(RAG_SETTINGS, indent=2))

{
  "corpus": {
    "file": "core_articles.csv",
    "articles": 1210,
    "filter": "Stage 1 om_tier == 'core'"
  },
  "chunks": {
    "size": 200,
    "overlap": 40,
    "contextualize": "title"
  },
  "embedding_model": "all-MiniLM-L6-v2",
  "embeddings_file": "embeddings_core_articles_size200_overlap40_title_all_MiniLM_L6_v2.npy",
  "retrieval": {
    "strategy": "hybrid",
    "size": 30
  },
  "generation": {
    "mode": "grounded",
    "max_passages": 6,
    "max_per_source": 2,
    "max_chars_per_passage": 1200,
    "max_tokens": 400
  },
  "rag_model": {
    "model": "qwen3:4b-instruct",
    "server": "Ollama, OpenAI-compatible endpoint http://localhost:11434/v1"
  },
  "test_set": {
    "file": "qa_reviewed.json",
    "questions": 213
  }
}


## 2.8 Saved outputs
| File | Content |
|---|---|
| `core_articles.csv` | the 1,210 core articles ingested by the RAG system (id = Stage 1 `doc_id`) |
| `embeddings_…all_MiniLM_L6_v2.npy` | the embeddings of the chunks, reloaded at later runs |
| `rag_settings.json` | the settings of the baseline, reused by the other work packages |

# 3. Knowledge Graph v2: Revising the Stage 1 Graph for Retrieval
Sections 3 and 4 add the Stage 1 knowledge graph to the text-only RAG of Section 2 and compare the two systems on the
test set of Section 1. The graph is first revised for retrieval, following the feedback on Stage 1 (Section 3): lexicon
facts are kept apart from article claims, and every relation used for retrieval is checked against the sentence that
states it. The revised graph, knowledge graph v2, then links the questions to graph entities and relations, and its
evidence is added to the prompt next to the text passages (Section 4). Both systems use the same corpus, chunks,
search, prompt and language model, so the comparison (Section 4.5) measures the effect of the graph.

**Setup.** The Stage 1 graph tables, sentences, entity mentions and relation labels, the test set, the RAG settings and
the cached NLI scores are read from the repository, and the corpus and the chunk embeddings of Section 2 from the shared
Google Drive. The NLI model only runs for sentences missing from the cache.

In [ ]:
import json
import platform
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_colwidth", 120)
IN_COLAB = "google.colab" in sys.modules
SEED = 42

if IN_COLAB:
    %pip install -q "anatoolbox[embeddings] @ git+https://github.com/umagira/anatoolbox"

    def clone(slug):
        # Clone a GitHub repository under /content, or update it
        path = Path("/content") / slug.split("/")[1]
        if path.exists():
            !git -C {path} pull -q
        else:
            !git clone -q https://github.com/{slug}.git {path}
        return path

    REPO = clone("jihyeon26/computational-language-technology-hs26")   # test set, RAG settings, lexicon, Stage 1 sentences, mentions and relation labels
    CODE = REPO
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/HSLU/CLT")
    DATA_DIR = DRIVE / "data" / "processed"                                  # Stage 1 corpus and core articles
    PART1_RAG_DIR = DRIVE / "extraction_outputs" / "stage3" / "wp1" / "rag"    # corpus file and chunk embeddings of Section 2
    OUT_DIR = DRIVE / "extraction_outputs" / "stage3" / "wp2"
else:
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "stage3_graph_rag").exists())
    DATA_DIR = next(p for p in [REPO / "data" / "processed", REPO / "dataset" / "processed"] if p.exists())
    PART1_RAG_DIR = REPO / "outputs" / "stage3_wp1_rag"
    OUT_DIR = REPO / "artifacts" / "stage3_wp2" / "graph_rag"
    CODE = REPO
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(CODE / "src"))


def first(*paths):
    return next(p for p in paths if p.exists())


# Repository inputs (a local copy keeps the Part 1 and Stage 1 files under data/)
STAGE1_KG = CODE / "artifacts" / "stage3_wp2" / "stage1_inputs"        # Stage 1 knowledge graph tables
STAGE1_EXTRACTION = first(REPO / "artifacts" / "stage1_wp2" / "sentences.csv",
                   REPO / "data" / "stage1_submitted" / "artifacts" / "stage1_wp2" / "sentences.csv").parent
REVIEW_DIR = REPO / "artifacts" / "stage1_wp3" / "review"               # Stage 1 relation labels
KG_REVISION_DIR = CODE / "artifacts" / "stage3_wp2" / "kg_revision"     # NLI cache, blind review sheets, data dictionary
KG_V2_DIR = REPO / "outputs" / "stage3_kg_v2"                           # knowledge graph v2 (large files, not in git)
PART1_DIR = first(REPO / "artifacts" / "stage3_wp1", REPO / "data" / "stage3_wp1")
LEXICON = REPO / "artifacts" / "stage1_wp1" / "lexicon" / "open_models_lexicon.json"
CORE_ARTICLES = DATA_DIR / "open_models_core.parquet"
KG_V2_DIR.mkdir(parents=True, exist_ok=True)

RAG_SETTINGS = json.loads((PART1_DIR / "rag_settings.json").read_text(encoding="utf-8"))
EVAL_SET = json.loads((PART1_DIR / "qa_reviewed.json").read_text(encoding="utf-8"))
TYPES = ["factual", "temporal", "relational", "analytical", "comparative"]
print(f"Test set: {len(EVAL_SET)} questions | RAG model: {RAG_SETTINGS['rag_model']['model']}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Test set: 213 questions | RAG model: qwen3:4b-instruct


Graph-enhanced retrieval answers from an entity of the question, a typed relation, the sentence that states it, and
its article and date. Following the Stage 1 feedback, the Stage 1 graph is revised before it is used: lexicon facts
are separated from article claims (3.1), every evidence sentence is checked against its relation (3.2 – 3.4), the
edges used for retrieval are chosen from the checked relations (3.5), and the retrieval check of Stage 1 is repeated
with a human review of the passages (3.6).

In [ ]:
from stage3_graph_rag import EntityLinker, GraphEvidenceRetriever, GraphTextRetriever, node_mentions, question_intents
from stage3_graph_rag.linking import GENERIC_NAMES, NAMED_TYPES
from stage3_kg_revision import (TRIPLE, NLIVerifier, aggregate_support, article_evidence, build_pairs, edge_layers,
                                label_assertions, lexicon_conflicts, load_review_labels, score_pairs, wilson, with_key)
from wp3_knowledge_graph.pipeline import calculate_node_metrics
# The Stage 1 module above switches matplotlib to a file-only backend (Agg): restore inline figures
%matplotlib inline

NLI_MODEL = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
SUPPORT_THRESHOLD = 0.5   # fixed in advance: entailment more likely than not

nodes = pd.read_parquet(STAGE1_KG / "nodes.parquet")
assertions = label_assertions(pd.read_parquet(STAGE1_KG / "typed_edges_assertions.parquet"))
collapsed = pd.read_parquet(STAGE1_KG / "typed_edges_collapsed.parquet")
relations_all_tiers = pd.read_parquet(STAGE1_KG / "relations_all_tiers.parquet")
evidence_status = pd.read_parquet(STAGE1_KG / "evidence_status.parquet")
sentences = pd.read_csv(STAGE1_EXTRACTION / "sentences.csv")
labels_of = nodes.set_index("entity_id")["canonical_name"]
print(f"Stage 1 graph: {len(nodes):,} nodes, {len(collapsed):,} edges, {len(assertions):,} evidence rows")

Stage 1 graph: 5,092 nodes, 999 edges, 6,053 evidence rows


## 3.1 Article Claims and Lexicon Facts

The Stage 1 graph holds three kinds of knowledge in one edge table: relations stated in an article sentence
(GLiREL), model names with a size (`Qwen3-8B` -> member of family `Qwen`), and model family -> developer pairs from
the Stage 1 lexicon, which have no article sentence. In Stage 1, a pair found in the lexicon and in articles was one
Tier A edge, and the lexicon row counted as evidence. In v2 every edge has a `knowledge_layer`, support is counted
on article sentences only, and lexicon-only facts become background facts that the retriever never cites.

In [ ]:
edges = edge_layers(collapsed, assertions)
conflicts = lexicon_conflicts(relations_all_tiers)
display(pd.crosstab(edges["knowledge_layer"], edges["tier"].rename("Stage 1 tier"), margins=True))
print(f"Lexicon pairs also stated in articles: {int(edges['lexicon_agreement'].eq('lexicon_and_articles').sum())}; "
      f"lexicon only: {int(edges['lexicon_agreement'].eq('lexicon_only').sum())}; "
      f"article relations excluded in Stage 1 because they contradict the lexicon: {len(conflicts)}")

Stage 1 tier,A,B,All
knowledge_layer,,,
article_claim,15,831,846
curated_lexicon,18,0,18
name_pattern,0,135,135
All,33,966,999


Lexicon pairs also stated in articles: 15; lexicon only: 18; article relations excluded in Stage 1 because they contradict the lexicon: 435


## 3.2 Does the Sentence State the Relation?

The Stage 1 review found a substantial share of incorrect relations, mostly where a relation was attached to the
wrong entity in a sentence naming several models or companies. Each evidence sentence is therefore checked with a
natural language inference (NLI) model, `DeBERTa-v3-large` trained on MNLI, FEVER, ANLI, LingNLI and WANLI: does
the sentence (premise) entail the relation written as a sentence (hypothesis), e.g. *"Qwen was developed by
Alibaba."*?
- The hypothesis uses the **graph node names**, so a sentence about *Llama Nemotron* does not support
  `Llama -> developed by -> NVIDIA`.
- A sentence supports the relation if the entailment probability is at least 0.5; an edge is supported if one of
  its sentences is.
- Name patterns are not checked: the relation comes from the model name, and the Stage 1 review found the
  model-name step correct in 13 of 13 cases.

Scored are the sentences of all article edges of the graph, of all relations a person labelled (also those removed
after the review), and of the relations excluded by the lexicon.

In [ ]:
labels = load_review_labels(REVIEW_DIR)
evidence = article_evidence(evidence_status)
keys = (set(with_key(edges[edges["has_article_claim"]])["key"]) | set(labels["key"])
        | set(with_key(conflicts.rename(columns={"head_id": "subject_id", "relation": "predicate", "tail_id": "object_id"}))["key"]))
pool = evidence[evidence["key"].isin(keys)].reset_index(drop=True)
pairs = build_pairs(pool, head_name="subject", tail_name="object")
scored = score_pairs(pairs, NLIVerifier(NLI_MODEL, batch_size=32), KG_REVISION_DIR / "nli_scores.csv", NLI_MODEL)
sentence_support, edge_support = aggregate_support(pool, scored, ["key"], threshold=SUPPORT_THRESHOLD)
decisive = ((sentence_support["support_score"] < 0.1) | (sentence_support["support_score"] > 0.9)).mean()
print(f"{len(pool):,} sentences of {pool['key'].nunique():,} relations; {sentence_support['supported'].mean():.0%} "
      f"support their relation; {decisive:.0%} of the scores are below 0.1 or above 0.9, so the threshold matters little")

5,726 sentences of 1,380 relations; 71% support their relation; 89% of the scores are below 0.1 or above 0.9, so the threshold matters little


## 3.3 Comparison with the Human Labels

The check is compared with the Stage 1 labels before it is applied: the **random sample** (people) and the
**findings check** (relations behind the Stage 1 findings; `FALSE` confirmed by a person, `TRUE` mostly from an AI
first pass). The extraction confidence of GLiREL is the baseline for the ranking quality (AUC).

In [ ]:
glirel_mean = pool.groupby("key")["glirel_score"].mean()
decided = labels[labels["label"].isin(["TRUE", "FALSE"])].merge(edge_support, on="key", how="left", validate="many_to_one")
decided["is_true"], decided["kept"] = decided["label"].eq("TRUE"), decided["support_status"].eq("supported")
decided["relations"] = np.where(decided["predicate"].eq("competes with"), "competes with", "other relations")

def agreement_row(group):
    y, kept = group["is_true"], group["kept"]
    tp = int((y & kept).sum())
    low, high = wilson(tp, int(kept.sum()))
    return pd.Series({"relations": len(group), "precision without check": round(y.mean(), 2),
                      "AUC NLI": round(roc_auc_score(y, group["support_max"]), 2),
                      "AUC GLiREL": round(roc_auc_score(y, group["key"].map(glirel_mean)), 2),
                      "kept": int(kept.sum()), "precision of kept": f"{tp / kept.sum():.2f} ({low:.2f}-{high:.2f})",
                      "correct relations kept": f"{tp} / {int(y.sum())}"})

agreement = decided.groupby(["label_set", "relations"]).apply(agreement_row, include_groups=False)
display(agreement)
other_rs = agreement.loc[("random sample", "other relations")]
other_fc = agreement.loc[("findings check", "other relations")]
competes_rs = agreement.loc[("random sample", "competes with")]

relations  precision without check  AUC NLI  \
label_set      relations                                                      
findings check competes with           21                     0.86     0.81   
               other relations        305                     0.81     0.90   
random sample  competes with           22                     0.27     0.79   
               other relations         69                     0.71     0.92   

                                AUC GLiREL  kept precision of kept  \
label_set      relations                                             
findings check competes with          0.56    20  0.90 (0.70-0.97)   
               other relations        0.65   276  0.89 (0.85-0.92)   
random sample  competes with          0.56    21  0.29 (0.14-0.50)   
               other relations        0.69    62  0.79 (0.67-0.87)   

                               correct relations kept  
label_set      relations                               
findings check competes with                  18 / 18  
               other relations              246 / 248  
random sample  competes with                    6 / 6  
               other relations                49 / 49

For the relations other than `competes with`, the check raises precision from 0.71 to 0.79 on the random sample (0.81
to 0.89 on the findings check) and keeps 49 of 49 correct relations of the sample. It also ranks correct above
incorrect relations far better than the GLiREL confidence (AUC 0.92 against 0.69).

`competes with` is the exception: only 6 of 22 sampled relations are correct, and the check keeps 21 of them. News
sentences list rivals (*"open competitors, including DeepSeek, Mistral, and Meta's Llama"*), and the NLI model reads
every pair of such a list as a rivalry. Since no sentence-level check separates the correct relations, `competes with`
is not used for retrieval (Section 3.5); competition questions are answered from text retrieval.

## 3.4 Errors Left to the Entity Rules

In [ ]:
missed = decided[~decided["is_true"] & decided["kept"] & decided["label_source"].eq("human")
                 & decided["relations"].eq("other relations")]
best = sentence_support.sort_values("support_score", ascending=False).drop_duplicates("key").set_index("key")
missed = missed.assign(relation=lambda d: d["subject_id"].map(labels_of) + " -> " + d["predicate"] + " -> " + d["object_id"].map(labels_of),
                       sentence=lambda d: d["key"].map(best["sentence_text"]).str.slice(0, 160))
print(f"Relations labelled FALSE that the check keeps: {len(missed)} "
      f"({', '.join(f'{e}: {n}' for e, n in missed['error_type'].fillna('unspecified').value_counts().head(3).items())})")
entity_errors = missed[missed["error_type"].isin(["entity_boundary", "entity_linking"])].sort_values("error_type")
with pd.option_context("display.max_colwidth", 160):
    display(entity_errors[["relation", "error_type", "sentence"]].head(6))

Relations labelled FALSE that the check keeps: 43 (entity_linking: 23, entity_boundary: 8, wrong_predicate: 4)


,relation,error_type,sentence
101,Pro model -> developed by -> DeepSeek,entity_boundary,"Read's Market Analysis on Investing.com\n\nDeepSeek said it will make permanent a steep discount on its flagship V4‑Pro model, maintaining prices for develo..."
119,Mamba2 -> developed by -> NVIDIA,entity_boundary,The most significant change between the two models is reducing the full attention layers from 4 in the NVIDIA hybrid Mamba2 model to 3 in Bamba-9B and intro...
59,Lightning -> developed by -> NVIDIA,entity_boundary,NVIDIA Nemotron 3.5 Lightning is a fully customizable open model built for high-volume tasks powering always-on agents.
71,Phi -> licensed under -> MIT license,entity_boundary,"Unlike closed-source models that demand significant computational power, Phi-4 is open-source and accessible on Hugging Face under an MIT license, thereby r..."
129,simulation frameworks -> developed by -> NVIDIA,entity_boundary,"NVIDIA technologies — including GPUs, open models, simulation frameworks and CUDA-accelerated libraries — were referenced in the majority of accepted CVPR 2..."
100,Chinese AI -> developed by -> DeepSeek,entity_boundary,"In November, Alibaba and Chinese AI developer DeepSeek released reasoning models that, by some measures, rival OpenAI’s o1-preview."


The relations labelled FALSE that pass the check concern the entity, not the relation: fragment nodes
(`Lightning` for *NVIDIA Nemotron 3.5 Lightning*), or a name attached to the wrong entity in a list. The sentence
states the relation for the words it contains, so the check accepts it. These cases are handled by entity rules:
fragment nodes are not used for retrieval (Section 3.5), family-level links are excluded by the lexicon filter of
Stage 1, and every graph fact reaches the prompt together with the sentence that states it (Section 4.4).

## 3.5 Graph Edges Used for Retrieval

An edge is used for retrieval if it is a supported article claim or a name pattern. A relation confirmed by a
person is kept, `competes with` is excluded, and so are edges of a node whose name is a fragment of a longer product
name (*Lightning*, *Ultra*, *Pro*, *Flash*, …). The excluded edges stay in the graph with their status.

In [ ]:
edges = with_key(edges).merge(edge_support, on="key", how="left", validate="one_to_one")
edges.loc[edges["knowledge_layer"].eq("curated_lexicon"), "support_status"] = "no_passage"
edges.loc[edges["knowledge_layer"].eq("name_pattern"), "support_status"] = "name_pattern"
human_true = edges["review_status"].isin(["reviewed_true", "checked_true_human"])
edges["relation_reliability"] = np.where(edges["predicate"].eq("competes with"), "low", "standard")
is_fragment = lambda ids: ids.map(labels_of).astype(str).str.strip().str.lower().isin(GENERIC_NAMES)
edges["name_fragment"] = is_fragment(edges["subject_id"]) | is_fragment(edges["object_id"])
edges["retrieval_eligible"] = ((edges["support_status"].isin(["supported", "name_pattern"])
                                | (edges["knowledge_layer"].eq("article_claim") & human_true))
                               & edges["relation_reliability"].eq("standard") & ~edges["name_fragment"])

by_predicate = (edges[edges["knowledge_layer"].ne("curated_lexicon")].groupby("predicate")
                .agg(edges=("key", "size"), supported=("support_status", lambda s: int(s.eq("supported").sum())),
                     used_for_retrieval=("retrieval_eligible", "sum"))
                .sort_values("edges", ascending=False))
display(by_predicate[by_predicate["edges"] >= 10].T)

conflict_support = with_key(conflicts.rename(columns={"head_id": "subject_id", "relation": "predicate", "tail_id": "object_id"}))
conflict_support = conflict_support.merge(edge_support, on="key", how="left")
print(f"Edges used for retrieval: {int(edges['retrieval_eligible'].sum())} of {len(edges)}; "
      f"lexicon-excluded relations that the check would keep: {conflict_support['support_status'].eq('supported').mean():.0%}")

# The fragment rule on the labelled relations (random sample, relations other than competes with)
sample = decided[decided["label_set"].eq("random sample") & decided["relations"].eq("other relations")].copy()
sample["fragment"] = is_fragment(sample["subject_id"]) | is_fragment(sample["object_id"])
used = sample["kept"] & ~sample["fragment"]
fragment_removed = sample[sample["kept"] & sample["fragment"]]
precision_used = sample.loc[used, "is_true"].mean()
print(f"Fragment rule: {int(edges['name_fragment'].sum())} edges excluded | random sample: relations kept by the check "
      f"and removed by the rule: {', '.join(fragment_removed['subject_id'].map(labels_of) + ' (' + fragment_removed['label'] + ')')}; "
      f"precision of the relations used {precision_used:.2f}, correct relations used "
      f"{int((sample['is_true'] & used).sum())} / {int(sample['is_true'].sum())}")

predicate,developed by,member of family,competes with,works for,headquartered in,founded by,outperforms,available on,invested in,manufactured by,acquired by,supported by,announced by,runs on,licensed under
edges,332,135,92,91,57,53,52,31,28,23,15,14,12,11,10
supported,319,0,86,87,47,42,41,30,9,23,10,14,8,10,10
used_for_retrieval,317,135,0,87,47,42,41,28,9,23,10,14,8,10,10


Edges used for retrieval: 801 of 999; lexicon-excluded relations that the check would keep: 41%
Fragment rule: 4 edges excluded | random sample: relations kept by the check and removed by the rule: Lightning (FALSE); precision of the relations used 0.80, correct relations used 49 / 49


In [ ]:
def analysis_graph(edge_table, weight_column):
    graph = nx.MultiDiGraph()
    for row in edge_table.itertuples(index=False):
        graph.add_node(row.subject_id, label=labels_of.get(row.subject_id))
        graph.add_node(row.object_id, label=labels_of.get(row.object_id))
        graph.add_edge(row.subject_id, row.object_id, key=row.predicate, weight=max(float(getattr(row, weight_column)), 1.0))
    return graph

def pagerank_ranks(edge_table, weight_column):
    view = edge_table[~edge_table["predicate_group"].eq("ATTRIBUTE") & ~edge_table["added_by"].eq("model_name_member")]
    metrics = calculate_node_metrics(analysis_graph(view, weight_column), "typed", "weight", sample_size=1000, seed=SEED)
    return pd.Series(np.arange(1, len(metrics) + 1), index=metrics["entity_id"])

stage1_rank = pagerank_ranks(edges, "document_count")
v2_rank = pagerank_ranks(edges[edges["retrieval_eligible"] & edges["has_article_claim"]], "article_documents")
ranks = pd.DataFrame({"entity": stage1_rank.index.map(labels_of), "Stage 1": stage1_rank,
                      "v2": v2_rank.reindex(stage1_rank.index).astype("Int64")}).head(12)
display(ranks.T)

entity_id,org:nvidia,org:openai,org:google,org:deepseek,org:meta,org:anthropic,org:microsoft,model:llama,org:amazon,model:nemotron,technology:hugging_face,org:alibaba
entity,NVIDIA,OpenAI,Google,DeepSeek,Meta,Anthropic,Microsoft,Llama,Amazon,Nemotron,Hugging Face,Alibaba
Stage 1,1,2,3,4,5,6,7,8,9,10,11,12
v2,1,2,3,4,5,6,9,7,28,10,11,29


801 of 999 edges are used for retrieval; 84 article edges without a supporting sentence, the 92 `competes with` edges
and the 4 edges of fragment nodes stay in the graph but are not used. On the random sample, the fragment rule removes
only relations labelled FALSE, and the relations used for retrieval reach a precision of 0.80 while keeping every
correct relation. PageRank of the leading actors hardly changes; Amazon (9 -> 28) and Alibaba (12 -> 29) fall, because
their rank in Stage 1 came mostly from `competes with` edges.

The check would keep 41% of the relations that Stage 1 excluded because they contradict the lexicon, e.g.
`Llama -> developed by -> NVIDIA` from *Llama Nemotron* sentences. These are family-level linking errors that the check
cannot see, so the lexicon filter stays, as a decision based on curated knowledge.

## 3.6 Retrieval Check: Do the Passages Support the Relations?

The retrieval check of the Stage 1 notebook (Section 6.9) only tested that a passage is returned. It is repeated with the same
retriever on the Stage 1 graph (**before**) and on v2 (**after**: edges used for retrieval, only supporting
sentences, ranked by support score, lexicon facts returned separately as background). The returned passages were
labelled in a blind sheet that hides the system and the score.

In [ ]:
text_units = pd.DataFrame({"text_unit_id": sentences["doc_id"] + ":" + sentences["sentence_id"].astype(str),
                           "document_id": sentences["doc_id"], "text_unit_text": sentences["sentence_text"]})
documents = assertions.dropna(subset=["document_id"]).groupby("document_id")["published_at"].first().reset_index()
articles = pd.read_parquet(CORE_ARTICLES, columns=["doc_id", "title", "url", "domain"])
documents = documents.merge(articles.rename(columns={"doc_id": "document_id"}), on="document_id", how="left")

support_columns = ["support_score", "contradiction_score", "supported", "hypothesis"]
assertions_v2 = (with_key(assertions)
                 .merge(sentence_support.drop_duplicates(["key", "text_unit_id"])[["key", "text_unit_id", *support_columns]],
                        on=["key", "text_unit_id"], how="left", validate="many_to_one")
                 .merge(edges[["key", "retrieval_eligible"]], on="key", how="left", validate="many_to_one"))
assertions_v2["supported"] = assertions_v2["supported"].fillna(False).astype(bool)
retrieval_assertions = assertions_v2[assertions_v2["retrieval_eligible"]
                                     & (assertions_v2["supported"] | assertions_v2["knowledge_source"].eq("name_pattern"))]
background = edges[edges["in_lexicon"]][TRIPLE].assign(knowledge_source="curated_lexicon")
retrievers = {"before": GraphEvidenceRetriever(nodes, assertions.dropna(subset=["text_unit_id"]), text_units, documents),
              "after": GraphEvidenceRetriever(nodes, retrieval_assertions, text_units, documents, background=background)}

QUESTIONS = [
    ("Who developed Qwen?", ["developed by"]), ("Which companies compete with DeepSeek?", ["competes with"]),
    ("What does NVIDIA manufacture?", ["manufactured by"]), ("Who developed Gemma?", ["developed by"]),
    ("Who developed Llama?", ["developed by"]), ("Which companies compete with OpenAI?", ["competes with"]),
    ("Which models does Kimi outperform?", ["outperforms"]), ("Where is DeepSeek headquartered?", ["headquartered in"]),
    ("Who works for Meta?", ["works for"]), ("Who founded DeepSeek?", ["founded by"]),
    ("Where is Nemotron available?", ["available on"]), ("Who invested in OpenAI?", ["invested in"]),
    ("Under which license is Gemma released?", ["licensed under"]),
]
supported_relation = assertions_v2.set_index("relation_id")["supported"]
rows = []
for question, predicates in QUESTIONS:
    for system, retriever in retrievers.items():
        for rank, item in enumerate(retriever.retrieve(question, predicates=predicates, top_k=3)["evidence"], start=1):
            rows.append({"question": question, "system": system, "rank": rank, "relation_id": item["relation_id"],
                         "relation": f"{item['subject']['name']} -> {item['predicate']} -> {item['object']['name']}",
                         "support_score": item["support_score"] if system == "after" else None,
                         "supported": bool(supported_relation.get(item["relation_id"], False)),
                         "published_at": item["published_at"], "document_id": item["document_id"],
                         "passage": " ".join(str(item["passage"]).split())})
retrieval_check = pd.DataFrame(rows)
retrieval_check.to_csv(KG_REVISION_DIR / "retrieval_check.csv", index=False, encoding="utf-8-sig")

REVIEW_SHEET = KG_REVISION_DIR / "retrieval_support_review.xlsx"
if not REVIEW_SHEET.exists():   # blank sheet for reviewers; labelled copies are saved as retrieval_support_review_<name>.xlsx
    (retrieval_check.drop_duplicates(["question", "relation_id"]).sample(frac=1.0, random_state=SEED)
     [["question", "relation", "passage", "published_at", "document_id"]]
     .assign(passage_states_relation="", reviewer="", notes="").to_excel(REVIEW_SHEET, sheet_name="Review", index=False))

/tmp/ipykernel_784/977976213.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  assertions_v2["supported"] = assertions_v2["supported"].fillna(False).astype(bool)


In [ ]:
MATCH = ["question", "relation", "passage"]
human = pd.concat([pd.read_excel(f, sheet_name="Review") for f in sorted(KG_REVISION_DIR.glob("retrieval_support_review_*.xlsx"))],
                  ignore_index=True)
human["human_label"] = human["passage_states_relation"].astype(str).str.strip().str.upper()
checked = retrieval_check.merge(human.drop_duplicates(MATCH)[MATCH + ["human_label", "notes"]], on=MATCH, how="left")

def system_row(part):
    counts = part["human_label"].value_counts()
    true, false = int(counts.get("TRUE", 0)), int(counts.get("FALSE", 0))
    low, high = wilson(true, true + false)
    return pd.Series({"questions with evidence": part["question"].nunique(), "passages": len(part),
                      "passing the check": int(part["supported"].sum()), "human: TRUE": true, "human: FALSE": false,
                      "human: AMBIGUOUS": int(counts.get("AMBIGUOUS", 0)),
                      "share stating the relation": f"{true / (true + false):.2f} ({low:.2f}-{high:.2f})"})

retrieval_summary = checked.groupby("system").apply(system_row, include_groups=False).loc[["before", "after"]]
display(retrieval_summary)
with pd.option_context("display.max_colwidth", 140):
    display(checked.drop_duplicates(["question", "relation_id"])
            .query("human_label in ['FALSE', 'AMBIGUOUS']")[["system", "relation", "supported", "human_label", "passage"]])
review_before, review_after = retrieval_summary.loc["before"], retrieval_summary.loc["after"]
share = lambda row: f"{float(row['share stating the relation'].split()[0]):.0%}"

,questions with evidence,passages,passing the check,human: TRUE,human: FALSE,human: AMBIGUOUS,share stating the relation
system,,,,,,,
before,13,39,35,32,5,2,0.86 (0.72-0.94)
after,11,33,33,27,1,5,0.96 (0.82-0.99)


,system,relation,supported,human_label,passage
2,before,Qwen -> developed by -> Alibaba,True,FALSE,"Alibaba Group Holding (BABA Quick QuoteBABA - Free Report) too introduced the QwQ-32B model, an AI system that rivals DeepSeek but requi..."
12,after,GB200 NVL72 -> manufactured by -> NVIDIA,True,AMBIGUOUS,"At NVIDIA GTC Washington, D.C., NVIDIA founder and CEO Jensen Huang highlighted how GB200 NVL72 delivers 10x the performance of NVIDIA H..."
13,after,A100 -> manufactured by -> NVIDIA,True,AMBIGUOUS,"In the data center segment, NVIDIA provides high-throughput accelerators—such as the NVIDIA A100 and H100 Tensor Core GPUs—and integrate..."
14,after,GeForce RTX -> manufactured by -> NVIDIA,True,AMBIGUOUS,"In addition, NVIDIA is bringing GeForce RTX gaming to more devices with new GeForce NOW Apps for Linux PC and Amazon Fire TV."
20,after,Gemma -> developed by -> Google,True,FALSE,"DolphinGemma, a large language model developed by Google, is helping scientists study how dolphins communicate — and hopefully find out ..."
30,before,Kimi -> outperforms -> Claude,True,AMBIGUOUS,"Separately, Alibaba-backed startup Moonshot released its new Kimi K2 AI model late on Friday, which claims to outperform OpenAI's ChatGP..."
38,before,DeepSeek -> headquartered in -> united_states,False,FALSE,"In early 2025, the Chinese AI company DeepSeek made international news upon releasing a large language model (LLM) that appeared to outp..."
54,before,Nemotron -> available on -> GitHub,True,FALSE,"NeMo Switchyard is available on GitHub. Visit the Nemotron 3.5 Lightning, NeMo Switchyard and Jetson AI technical blogs to get started."
55,before,Nemotron -> available on -> GitHub,False,FALSE,"Experiment with NemoClaw, available on GitHub, and join the community of developers on Discord building with NemoClaw using NVIDIA Nemot..."
67,before,Gemma -> licensed under -> Apache 2.0,False,FALSE,"Hot on Gemma 3’s heels: Shortly after Gemma 3 became available, Mistral released Small 3.1 (24 billion parameters), a vision-language mo..."


The reviewer judged 86% of the passages before the revision and 96% after it to state their relation; 4 of the 39
passages returned before do not pass the check. Passages removed include *DeepSeek -> headquartered in -> United
States* from *"... companies, largely headquartered in the United States"* and *Gemma -> licensed under -> Apache 2.0*
from a sentence about Mistral's license. The two competition questions no longer return graph evidence; they are
answered from text retrieval. Lexicon facts appear as background next to the cited passages. The passages the reviewer
still rejects are about a single product rather than the family (*DolphinGemma* for Gemma), and the ambiguous ones
mostly imply rather than state the relation (*"NVIDIA provides ... A100 and H100 GPUs"*).

## 3.7 Knowledge Graph v2

`outputs/stage3_kg_v2/` holds the edges, assertions, background facts, text units and documents, and
`GraphEvidenceRetriever.from_directories(dir, dir)` loads it. The columns are described in
`artifacts/stage3_wp2/kg_revision/KG_V2_DATA_DICTIONARY.md`. The graph-enhanced RAG (Section 4) uses the
nodes, the relations used for retrieval and the lexicon facts of this graph.

In [ ]:
edges_v2 = edges.drop(columns=["key"])
assertion_columns = [c for c in assertions_v2.columns if c != "key"]
tables = {"nodes": nodes, "assertions": assertions_v2[assertion_columns], "edges": edges_v2,
          "retrieval_assertions": retrieval_assertions[assertion_columns],
          "typed_edges_assertions": retrieval_assertions[assertion_columns],   # file name read by from_directories
          "background_facts": background, "text_units": text_units, "documents": documents}
for name, table in tables.items():
    table.to_parquet(KG_V2_DIR / f"{name}.parquet", index=False)

graph = nx.MultiDiGraph(schema_version="stage3-wp2-open-models-kg-v2.0")
for row in edges_v2.itertuples(index=False):
    graph.add_node(row.subject_id, label=str(labels_of.get(row.subject_id)))
    graph.add_node(row.object_id, label=str(labels_of.get(row.object_id)))
    graph.add_edge(row.subject_id, row.object_id, key=row.predicate, knowledge_layer=row.knowledge_layer,
                   support_status=str(row.support_status), retrieval_eligible=bool(row.retrieval_eligible))
nx.write_graphml(graph, KG_V2_DIR / "typed_graph_v2.graphml")
shutil.copy(KG_REVISION_DIR / "KG_V2_DATA_DICTIONARY.md", KG_V2_DIR / "KG_V2_DATA_DICTIONARY.md")

edges_v2.drop(columns=["relation_ids", "document_ids", "text_unit_ids", "domains"]).to_csv(
    KG_REVISION_DIR / "edges_v2.csv", index=False, encoding="utf-8-sig")
agreement.to_csv(KG_REVISION_DIR / "agreement_with_labels.csv", encoding="utf-8-sig")
manifest = {"schema_version": "stage3-wp2-open-models-kg-v2.0",
            "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "nli_model": NLI_MODEL, "support_threshold": SUPPORT_THRESHOLD,
            "edges": len(edges_v2), "retrieval_eligible_edges": int(edges_v2["retrieval_eligible"].sum()),
            "name_fragment_edges": int(edges_v2["name_fragment"].sum()),
            "support_status": edges_v2["support_status"].value_counts().to_dict(),
            "versions": {"python": platform.python_version(), "pandas": pd.__version__, "networkx": nx.__version__}}
(KG_REVISION_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

reloaded = GraphEvidenceRetriever.from_directories(KG_V2_DIR, KG_V2_DIR)
assert reloaded.retrieve("Who developed Qwen?", predicates=["developed by"])["evidence"]
graph_relations = retrieval_assertions[assertion_columns]   # relations used by the graph-enhanced RAG
print(sorted(p.name for p in KG_V2_DIR.iterdir()))

['KG_V2_DATA_DICTIONARY.md', 'assertions.parquet', 'background_facts.parquet', 'documents.parquet', 'edges.parquet', 'nodes.parquet', 'retrieval_assertions.parquet', 'text_units.parquet', 'typed_edges_assertions.parquet', 'typed_graph_v2.graphml']


# 4. Graph-Enhanced RAG
Knowledge graph v2 is added to the text-only RAG of Section 2: the questions are linked to graph entities (4.2), the
graph adds an entity ranking and a relation ranking to the text ranking (4.3), and verified graph facts reach the prompt
together with the passages that state them (4.4). Both systems are then compared on retrieval and on their answers (4.5).

## 4.1 Text-Only Baseline
The baseline of Section 2 is rebuilt from `rag_settings.json` with the same tools and the chunk embeddings of Section 2,
so that both systems search the same index and this part also runs on its own: the 1,210 core articles in chunks of
200 words (40 words of overlap, title added), `all-MiniLM-L6-v2` embeddings, hybrid search of 30 chunks, and grounded
generation from at most 6 passages (2 per article).

In [ ]:
import anatoolbox
from anatoolbox import ToolContext, resolve_tools
from anatoolbox.corpus import configure_embedder, ensure_embeddings, get_corpus, load_embeddings, save_embeddings
from sentence_transformers import SentenceTransformer

anatoolbox.register_reference_tools(replace=True)
ctx = ToolContext()
ingest, chunk, retrieve, synthesize, retrieval_metrics = resolve_tools(
    ["ingest_corpus", "chunk_by_size", "retrieve_passages", "synthesize_answer", "calculate_retrieval_metrics"])

# Corpus file and chunk embeddings of Section 2 (written again into their folder if they are missing)
PART1_RAG_DIR.mkdir(parents=True, exist_ok=True)
CORE_CSV = PART1_RAG_DIR / RAG_SETTINGS["corpus"]["file"]
if not CORE_CSV.exists():
    corpus = pd.read_parquet(DATA_DIR / "corpus_clean.parquet",
                             columns=["doc_id", "date", "domain", "url", "title", "text", "om_tier"])
    core = corpus[corpus["om_tier"] == "core"]
    pd.DataFrame({"id": core["doc_id"], "title": core["title"], "text": core["text"],
                  "date": pd.to_datetime(core["date"]).dt.strftime("%Y-%m-%d"),
                  "url": core["url"], "domain": core["domain"]}).to_csv(CORE_CSV, index=False)
rag_articles = ingest.run({"path": str(CORE_CSV), "name": "core_articles"}, context=ctx)
chunks = chunk.run({"input": rag_articles, **RAG_SETTINGS["chunks"]}, context=ctx)
chunk_corpus = get_corpus(chunks["corpus"])

embedding_model = SentenceTransformer(RAG_SETTINGS["embedding_model"])
configure_embedder(lambda texts: embedding_model.encode(texts, batch_size=64, normalize_embeddings=True,
                                                        show_progress_bar=len(texts) > 1000))
EMB_PATH = PART1_RAG_DIR / RAG_SETTINGS["embeddings_file"]
if EMB_PATH.exists():
    load_embeddings(chunk_corpus, EMB_PATH)
else:
    ensure_embeddings(chunk_corpus)
    save_embeddings(chunk_corpus, EMB_PATH)

RETRIEVAL_ARGS = RAG_SETTINGS["retrieval"]
ANSWER_ARGS = RAG_SETTINGS["generation"]


def search(question, strategy=RETRIEVAL_ARGS["strategy"], size=RETRIEVAL_ARGS["size"], **args):
    # The search of Section 2: `size` best chunks for `question`
    return retrieve.run({"query": question, "input": chunks, "strategy": strategy, "size": size, **args}, context=ctx)


print(f"{chunks['chunks']:,} chunks from {chunks['articles']:,} articles | retrieval {RETRIEVAL_ARGS} | "
      f"generation {ANSWER_ARGS}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

12,720 chunks from 1,210 articles | retrieval {'strategy': 'hybrid', 'size': 30} | generation {'mode': 'grounded', 'max_passages': 6, 'max_per_source': 2, 'max_chars_per_passage': 1200, 'max_tokens': 400}


## 4.2 Linking Questions to the Graph

Graph retrieval starts from the entities a question names. The retriever of the Stage 1 retrieval check (Section 3.6)
matches any node name or name token in the question, which links common words that are nodes of the graph (*release*, *plan*, *reasoning*,
*base*) and cuts version names short (*Qwen3.8* -> node `qwen3`). The revised `EntityLinker`:
- links only named entity types (models, organisations, people, products, platforms, hardware, benchmarks, datasets,
  licences, standards, places); capabilities, architectures and concepts are common words;
- finds a node by its name, its aliases and, for an open-model family, its version names (*Gemma 3* -> Gemma);
- lets separators vary (*Llama 3.2* = *Llama-3.2*) but never matches inside a longer version (*Qwen3* not in *Qwen3.8*);
- keeps the longest match, so *Llama 3 70B* is not linked as *Llama* as well;
- finds a family the graph holds under another name (*DataGemma*, *Qwen3.8*) with the patterns of the Stage 1 lexicon, with the
  Stage 1 exception for products that contain *Llama* (*Llama Nemotron*, *Llama Stack*).

In [ ]:
lexicon = json.loads(LEXICON.read_text(encoding="utf-8"))
FAMILY_EXCEPTIONS = {"Llama": r"\bLlama[\w.\- ]{0,15}?(?:Nemotron|Stack)\b|\bLlamaCon\b"}   # as in Stage 1 and Section 1
linker = EntityLinker(nodes, {f["label"]: f["rx"] for f in lexicon["families"].values()}, FAMILY_EXCEPTIONS)
string_matching = retrievers["after"]   # the retriever of Section 3.6

rows = []
for q in EVAL_SET:
    before = string_matching.resolve_entities(q["question"])
    after = linker.link(q["question"])
    rows.append({"id": q["id"], "type": q["type"], "question": q["question"],
                 "Section 3.6": ", ".join(m["canonical_name"] for m in before),
                 "revised": ", ".join(f"{m.name} ({m.surface})" if m.name != m.surface else m.name for m in after),
                 "before_n": len(before), "after_n": len(after),
                 "before_common_word": any(m["entity_type"] not in NAMED_TYPES for m in before),
                 "after_family": any(m.method != "node_name" for m in after)})
linking = pd.DataFrame(rows)
print(f"Questions with a linked entity: Section 3.6 {linking['before_n'].gt(0).sum()}, revised "
      f"{linking['after_n'].gt(0).sum()} of {len(linking)} | linked to a common-word node: Section 3.6 "
      f"{linking['before_common_word'].sum()}, revised 0 | linked to a family through a version or lexicon "
      f"name: {linking['after_family'].sum()}")
EXAMPLES = ["When did Google release Gemma 3?", "Which Qwen versions came before Qwen3.8?",
            "Which base models are Skywork's reward models built on?", "Which model family is DataGemma built on?",
            "Does turning off reasoning hurt the accuracy of Nemotron Nano 12B v2?",
            "How much does a 16-message chatbot conversation cost with Llama 3 70B on AWS Bedrock compared with GPT-4o?"]
display(linking.set_index("question").loc[[e for e in EXAMPLES if e in set(linking["question"])], ["Section 3.6", "revised"]])

Questions with a linked entity: Section 3.6 195, revised 196 of 213 | linked to a common-word node: Section 3.6 62, revised 0 | linked to a family through a version or lexicon name: 75


,Section 3.6,revised
question,,
When did Google release Gemma 3?,"Release, Google, Gemma","Google, Gemma (Gemma 3)"
Which Qwen versions came before Qwen3.8?,qwen3,Qwen (Qwen3.8)
Which base models are Skywork's reward models built on?,base,
Which model family is DataGemma built on?,,Gemma (DataGemma)
Does turning off reasoning hurt the accuracy of Nemotron Nano 12B v2?,"Nemotron-Nano-12B, reasoning, v2",Nemotron-Nano-12B (Nemotron Nano 12B)
How much does a 16-message chatbot conversation cost with Llama 3 70B on AWS Bedrock compared with GPT-4o?,"Llama-3-70B, AWS Bedrock, GPT-4o","Llama-3-70B (Llama 3 70B), AWS Bedrock, GPT-4o"


The revised linking finds an entity in about as many questions as before, but none of the links are common words:
with the retriever of Section 3.6, 62 questions had such a link (*Release*, *base*, *reasoning*), which would send graph retrieval to
unrelated articles. A third of the questions reach an open-model family through a version or product name. The
17 questions without a linked entity are general (*Which models meet the Open Source AI Definition?*) or name
something that is not a graph node; they keep the text-only answer.

## 4.3 Graph-Aware Retrieval

The graph adds two rankings to the text ranking of Section 2, and the three are fused by reciprocal rank fusion (RRF, `k` = 60,
as in the hybrid search of Section 2): a chunk ranked high by several rankings rises, and no scores have to be weighed.

| Ranking | What it returns | What the graph contributes |
|---|---|---|
| **text** | the 30 chunks of the hybrid search of Section 2, unchanged | - |
| **entity** | the first 100 text hits re-ordered by the number of question entities they mention | names, aliases and versions of the linked nodes (in the chunk) and the Stage 1 entity mentions (in the article) |
| **relation** | the chunks that hold the evidence sentence of a graph relation of the question's entities | relations of knowledge graph v2 used for retrieval (Section 3.5), with their sentence, article and date |

A relation is used when it links two entities of the question, or when its predicate is the one the question asks
for (e.g. *Who made …* -> `developed by`, *… built on …* -> `based on`, `member of family`). Questions are routed
as follows:
- **no linked entity**: text ranking only, so the answer is the text-only answer;
- **competition and comparison questions** (*compete*, *rival*, *compare*, *better*, *faster*, *cheaper*, …): no
  relation ranking. They ask how entities stand against each other, which the passages describe; `competes with` is
  not in the retrieval graph (Section 3.3), and a relation between two compared models (e.g. a benchmark claim) is
  rarely the comparison asked for;
- **lexicon facts** never enter retrieval or the prompt: they have no article to cite.

In [ ]:
mentions = node_mentions(pd.read_csv(STAGE1_EXTRACTION / "entity_mentions.csv", usecols=["entity_id", "doc_id", "sentence_id", "date"]), nodes)
graph_rag = GraphTextRetriever(linker, graph_relations, mentions, chunk_corpus.records, search, background=background)
print(f"Graph: {len(graph_relations):,} relations used for retrieval, {mentions['entity_id'].nunique():,} nodes with "
      f"{len(mentions):,} sentence mentions in {mentions['document_id'].nunique():,} articles")

chunk_info = {r["id"]: r for r in chunk_corpus.records}
example = next(q for q in EVAL_SET if q["question"] == "Who made the Molmo models?")
result = graph_rag.retrieve(example["question"], **RETRIEVAL_ARGS)
print(f"{example['question']} | entities: {[e['name'] for e in result['entities']]} | intents: {result['intents']} | "
      f"route: {result['route']}")
fused_ids = [p["id"] for p in result["passages"]]
view = {name: result["rankings"][name][:5] for name in ["text", "entity", "relation"]} | {"text + graph": fused_ids[:5]}
display(pd.DataFrame({name: [f"{'* ' if chunk_info[i]['source_id'] in example['source_ids'] else ''}"
                             f"{chunk_info[i]['title'][:45]} ({chunk_info[i]['date']})" for i in ids]
                            + [""] * (5 - len(ids)) for name, ids in view.items()},
                     index=pd.RangeIndex(1, 6, name="rank")))
print("* source article of the test question")

Graph: 4,173 relations used for retrieval, 4,950 nodes with 83,268 sentence mentions in 1,210 articles
Who made the Molmo models? | entities: ['OLMo / Molmo / Tülu'] | intents: ['developed by'] | route: entity+relation


,text,entity,relation,text + graph
rank,,,,
1,Understanding Multimodal LLMs - by Sebastian (2024-11-03),Understanding Multimodal LLMs - by Sebastian (2024-11-03),"* The Download: A CRISPR patent battle, and the (2024-09-26)",A tiny new open source AI model performs as w (2024-09-25)
2,A tiny new open source AI model performs as w (2024-09-25),A tiny new open source AI model performs as w (2024-09-25),A tiny new open source AI model performs as w (2024-09-25),Why bigger is not always better in AI (2024-10-01)
3,Understanding Multimodal LLMs - by Sebastian (2024-11-03),Understanding Multimodal LLMs - by Sebastian (2024-11-03),Ai2's Olmo 3 Pushes the Envelope for Open Sou (2025-11-20),"* The Download: A CRISPR patent battle, and the (2024-09-26)"
4,Why bigger is not always better in AI (2024-10-01),Why bigger is not always better in AI (2024-10-01),Why bigger is not always better in AI (2024-10-01),"Ai2 releases Tülu 3, a fully open-source mode (2025-01-30)"
5,Why bigger is not always better in AI (2024-10-01),Why bigger is not always better in AI (2024-10-01),New Small AI Model Lets Developers Experiment (2025-02-19),Understanding Multimodal LLMs - by Sebastian (2024-11-03)


* source article of the test question


For *Who made the Molmo models?*, the text ranking does not reach the source article in its first five chunks: the
top chunks discuss Molmo in other articles. The relation ranking starts with the chunk that holds the sentence
*"The Allen Institute for Artificial Intelligence (Ai2) … is releasing a family of open-source multimodal language
models, called Molmo"*, the verified `developed by` relation, and the fusion moves the source article to rank 3.
The entity ranking leaves the order unchanged here, since all top text chunks already mention Molmo.

## 4.4 Text and Graph Evidence in the Prompt

Both systems answer with `synthesize_answer` of Section 2, with the same system prompt and settings. The graph system
changes two things:
- **passages**: the fused ranking replaces the text ranking; the tool keeps the first 6 passages (2 per article),
  as for the baseline;
- **graph facts** `[G1]…`: a relation of the relation ranking is added as a graph fact only if the chunk holding its
  evidence sentence is among these 6 passages, so every graph fact in a prompt comes with the sentence that states
  it, as a passage the model can cite (at most 5 facts). The tool shows each fact with its publication date.

In [ ]:
from anatoolbox.corpus import passages_with_text
from anatoolbox.enrich.synthesize.synthesize_answer import build_user_prompt, curate


def prompt_sources(passages, corpus):
    # The passages synthesize_answer puts in the prompt, chosen exactly as the tool does
    sources, _, _ = curate(passages_with_text(passages, corpus, tool_name="synthesize_answer"),
                           max_passages=ANSWER_ARGS["max_passages"], max_chars=ANSWER_ARGS["max_chars_per_passage"],
                           max_per_source=ANSWER_ARGS["max_per_source"])
    return sources


def graph_inputs(question):
    # Fused passages and the graph facts whose evidence chunk is in the prompt
    result = graph_rag.retrieve(question, **RETRIEVAL_ARGS)
    in_prompt = {s["id"] for s in prompt_sources(result["passages"], result["corpus"])}
    return result, graph_rag.facts_in_context(result, in_prompt)


result, facts = graph_inputs(example["question"])
sources = prompt_sources(result["passages"], result["corpus"])
shown = [{**s, "text": s["text"][:160] + " …"} for s in sources]
labelled = [{**f, "label": f"G{i}"} for i, f in enumerate(facts, start=1)]
print(build_user_prompt(example["question"], shown, facts=labelled))
print("\nLexicon facts of these entities (not in the prompt):", result["background"] or "none")

Question: Who made the Molmo models?

Sources:

[S1] A tiny new open source AI model performs as well as powerful big ones · 2024-09-25 · https://www.technologyreview.com/2024/09/25/1104465/a-tiny-new-open-source-ai-model-performs-as-well-as-powerful-big-ones/
A tiny new open source AI model performs as well as powerful big ones

The Allen Institute for Artificial Intelligence (Ai2), a research nonprofit, is releasing …

[S3] The Download: A CRISPR patent battle, and the promise of tiny AI · 2024-09-26 · https://www.technologyreview.com/2024/09/26/1104538/the-download-a-crispr-patent-battle-and-the-promise-of-tiny-ai/
The Download: A CRISPR patent battle, and the promise of tiny AI

In the decade-long fight to control CRISPR, the super-tool for modifying DNA, it’s been common …

[S5] Understanding Multimodal LLMs - by Sebastian Raschka, PhD · 2024-11-03 · https://magazine.sebastianraschka.com/p/understanding-multimodal-llms
Understanding Multimodal LLMs - by Sebastian Raschka, PhD

pas

The prompt holds the 6 passages of the fused ranking and one graph fact, `developed by`, whose evidence sentence is in
passage [S3]. The lexicon also records Ai2 as the developer, but this fact has no article and stays out of the prompt.

## 4.5 Comparison with the Text-Only Baseline

### 4.5.1 Retrieval

Each test question comes from one article (`source_ids`), so retrieval is measured as in Section 2 with
`calculate_retrieval_metrics`: hit@k (the source article is among the first k articles) and the mean reciprocal rank
(MRR), counting the chunks of one article once. The graph rankings alone are shown as well, and the last measure
checks whether the source article is among the 6 passages of the prompt.

In [ ]:
runs = {q["id"]: graph_rag.retrieve(q["question"], **RETRIEVAL_ARGS) for q in EVAL_SET}
source_of = {r["id"]: r["source_id"] for r in chunk_corpus.records}
SYSTEMS = {"text only": lambda r: r["rankings"]["text"],
           "entity ranking alone": lambda r: r["rankings"]["entity"],
           "relation ranking alone": lambda r: r["rankings"]["relation"],
           "text + graph": lambda r: [p["id"] for p in r["passages"]]}


def score(ids_of):
    results = [{"id": q["id"], "relevant": q["source_ids"],
                "retrieved": [{"id": i, "source_id": source_of[i]} for i in ids_of(runs[q["id"]])]} for q in EVAL_SET]
    out = retrieval_metrics.run({"results": results, "k": [1, 3, 5, 10]}, context=ctx)
    return out["summary"], pd.DataFrame(out["per_question"])


summaries, per_question = {}, {}
for name, ids_of in SYSTEMS.items():
    summaries[name], per_question[name] = score(ids_of)
for name, key in [("text only", "text"), ("text + graph", "fused")]:
    in_prompt = []
    for q in EVAL_SET:
        run = runs[q["id"]]
        ids = run["rankings"]["text"] if key == "text" else [p["id"] for p in run["passages"]]
        passages = [{k: v for k, v in chunk_info[i].items() if k not in ("text", "chunk_text")} for i in ids]
        in_prompt.append(any(s["source_id"] in q["source_ids"] for s in prompt_sources(passages, run["corpus"])))
    summaries[name]["source in prompt"] = float(np.mean(in_prompt))
table = pd.DataFrame(summaries).T[["hit@1", "hit@3", "hit@5", "hit@10", "mrr", "source in prompt"]]
display(table.apply(lambda column: column.map(lambda v: "-" if pd.isna(v) else f"{v:.3f}")))

types = pd.Series({q["id"]: q["type"] for q in EVAL_SET})
routes = pd.Series({qid: run["route"] for qid, run in runs.items()})
by_type = pd.DataFrame({"questions": types.value_counts()})
for metric, column in [("hit@5", "hit@5"), ("MRR", "reciprocal_rank")]:
    for name, short in [("text only", "text"), ("text + graph", "text + graph")]:
        by_type[f"{metric} {short}"] = per_question[name].set_index("id")[column].groupby(types).mean()
by_type.loc["all"] = [len(EVAL_SET)] + [table.loc[n, c] for c in ["hit@5", "mrr"]
                                        for n in ["text only", "text + graph"]]
display(by_type.reindex(TYPES + ["all"]).astype({"questions": int}).round(2))

rank_text = per_question["text only"].set_index("id")["reciprocal_rank"]
rank_graph = per_question["text + graph"].set_index("id")["reciprocal_rank"]
change = np.sign(rank_graph - rank_text).map({1: "source ranked higher", -1: "source ranked lower", 0: "same rank"})
display(pd.crosstab(routes.rename("route"), change.rename("text + graph vs text only"), margins=True))

,hit@1,hit@3,hit@5,hit@10,mrr,source in prompt
text only,0.578,0.775,0.836,0.878,0.685,0.822
entity ranking alone,0.535,0.723,0.793,0.836,0.642,-
relation ranking alone,0.047,0.094,0.099,0.108,0.070,-
text + graph,0.606,0.793,0.850,0.906,0.713,0.831


,questions,hit@5 text,hit@5 text + graph,MRR text,MRR text + graph
factual,57,0.86,0.88,0.78,0.78
temporal,38,0.71,0.74,0.54,0.58
relational,48,0.92,0.94,0.67,0.72
analytical,27,0.81,0.85,0.74,0.75
comparative,43,0.84,0.81,0.68,0.71
all,213,0.84,0.85,0.69,0.71


text + graph vs text only,same rank,source ranked higher,source ranked lower,All
route,,,,
entity,138,13,0,151
entity+relation,26,12,7,45
text,17,0,0,17
All,181,25,7,213


Adding the graph raises every retrieval measure: hit@1 from 0.58 to 0.61, hit@10 from 0.88 to 0.91 and MRR from 0.69
to 0.71, and the source article reaches the prompt for two more questions. No question type loses in MRR; the gain is
largest for relational (0.67 -> 0.72), temporal (0.54 -> 0.58) and comparative questions (0.68 -> 0.71): relational questions ask
for the relations the graph holds, temporal questions mostly ask when a named model was released, and comparative
questions name several models that the entity ranking brings together.

The graph rankings alone retrieve less than the text ranking: the entity ranking only re-orders text hits, and the
relation ranking only returns chunks with a verified relation of the question's entities. Their value lies in the
fusion: the source article moves up for 25 questions and down for 7, and the entity ranking never moves it down. The
17 questions without a linked entity keep the text-only ranking.

### 4.5.2 Answers

Each question is answered by both systems with Qwen3-4B-Instruct (Ollama), the RAG model of Section 2, at temperature 0.
The answers are saved in `answers.jsonl` with the full result of `synthesize_answer` (answer, numbered sources,
graph facts, citation checks), so that the evaluation can score them with the same evidence. The checks below need no
judge model:
- **cites a source**: the answer cites at least one passage or graph fact;
- **cites the source article**: one of the cited passages or graph facts comes from the source article of the question;
- **citation coverage**: share of the answer's sentences with a citation (`synthesize_answer`);
- **unknown citations**: labels that were not in the prompt;
- **no answer**: the model says that the sources do not contain the answer.

The graph leaves the prompt unchanged for about half of the questions (same passages, no graph fact). The
checks are therefore shown for both groups: the questions whose prompt the graph changes show its effect, and the
others, where both systems receive the same prompt, check that the differences come from the graph.

In [ ]:
from anatoolbox.llm_client import configure_llm

RAG_MODEL = RAG_SETTINGS["rag_model"]["model"]
ANSWERS = OUT_DIR / "answers.jsonl"
SAVED_ANSWERS = CODE / "artifacts" / "stage3_wp2" / "graph_rag" / "answers.jsonl"   # answers saved in the repository
if not ANSWERS.exists() and SAVED_ANSWERS.exists():
    shutil.copy(SAVED_ANSWERS, ANSWERS)
answers = {}
if ANSWERS.exists():
    for line in ANSWERS.read_text(encoding="utf-8").splitlines():
        record = json.loads(line)
        answers[(record["system"], record["id"])] = record
todo = [(system, q) for q in EVAL_SET for system in ["text", "graph"] if (system, q["id"]) not in answers]
CAN_GENERATE = IN_COLAB or shutil.which("ollama") is not None
print(f"Answers saved: {len(answers)} | to generate: {len(todo)}"
      + ("" if CAN_GENERATE or not todo else " (Ollama is not available here: run this section in Colab)"))

if todo and CAN_GENERATE:
    if IN_COLAB:
        !apt-get -qq install -y zstd > /dev/null
        !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    ollama_server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    !ollama pull {RAG_MODEL}
    configure_llm(base_url="http://localhost:11434/v1", model=RAG_MODEL, timeout=180, max_retries=2)

Answers saved: 426 | to generate: 0


In [ ]:
def generate(system, q):
    started = time.perf_counter()
    if system == "text":
        run, facts = None, []
        args = {"input": search(q["question"])}
    else:
        run, facts = graph_inputs(q["question"])
        args = {"input": {"passages": run["passages"], "corpus": run["corpus"]}} | ({"facts": facts} if facts else {})
    result = synthesize.run({"question": q["question"], **args, **ANSWER_ARGS}, context=ctx)
    return {"system": system, "id": q["id"], "type": q["type"], "question": q["question"],
            "seconds": round(time.perf_counter() - started, 1), "route": run["route"] if run else "text",
            "entities": [e["name"] for e in run["entities"]] if run else [], "result": result}


if todo and CAN_GENERATE:
    started = time.perf_counter()
    with ANSWERS.open("a", encoding="utf-8") as handle:
        for n, (system, q) in enumerate(todo, start=1):
            record = generate(system, q)
            answers[(system, q["id"])] = record
            handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
            handle.flush()
            if n % 50 == 0:
                print(f"{n}/{len(todo)} answers, {time.perf_counter() - started:.0f} s")
print(f"Answers: {len(answers)} of {2 * len(EVAL_SET)}")

Answers: 426 of 426


In [ ]:
NO_ANSWER = re.compile(r"(?:do(?:es)? not|don't|doesn't) (?:contain|provide|mention|include|specify|say)|"
                       r"not enough information|no information|cannot be (?:determined|answered)", re.I)
source_ids = {q["id"]: set(q["source_ids"]) for q in EVAL_SET}


def checks(record):
    result = record["result"]
    cited = set(result["cited"])
    origin = {s["label"]: {s.get("source_id")} for s in result["sources"]}
    origin |= {f["label"]: set(f.get("source_ids") or []) for f in result["facts"]}
    return {"system": record["system"], "id": record["id"], "type": record["type"], "route": record["route"],
            "cites a source": bool(cited),
            "cites the source article": any(origin.get(label, set()) & source_ids[record["id"]] for label in cited),
            "citation coverage": result["citation_coverage"] or 0.0,
            "unknown citations": bool(result["unknown_citations"]),
            "no answer": bool(NO_ANSWER.search(result["answer"])),
            "graph facts given": len(result["facts"]),
            "graph facts cited": sum(label.startswith("G") for label in cited),
            "words": len(result["answer"].split())}


if len(answers) == 2 * len(EVAL_SET):
    answer_checks = pd.DataFrame([checks(r) for r in answers.values()])
    answer_checks.to_csv(OUT_DIR / "answer_checks.csv", index=False)
    names = {"text": "text only", "graph": "text + graph"}
    columns = ["cites a source", "cites the source article", "citation coverage", "unknown citations", "no answer", "words"]
    display(answer_checks.groupby(answer_checks["system"].map(names))[columns].mean().round(3))

    # Questions whose prompt the graph changes: other passages, or graph facts
    prompt_ids = lambda system, qid: [s["id"] for s in answers[(system, qid)]["result"]["sources"]]
    changed = {qid for qid in source_ids if prompt_ids("text", qid) != prompt_ids("graph", qid)
               or answers[("graph", qid)]["result"]["facts"]}
    answer_checks["prompt"] = np.where(answer_checks["id"].isin(changed), "changed by the graph", "same")
    by_prompt = answer_checks.groupby(["prompt", answer_checks["system"].map(names)]).agg(
        questions=("id", "size"), **{c: (c, "mean") for c in ["cites the source article", "citation coverage", "no answer"]})
    display(by_prompt.round(3))
    same = sorted(set(source_ids) - changed)
    identical = sum(answers[("text", qid)]["result"]["answer"] == answers[("graph", qid)]["result"]["answer"] for qid in same)
    paired = answer_checks[answer_checks["id"].isin(changed)].pivot(index="id", columns="system", values="cites the source article")
    print(f"Same prompt: {len(same)} questions, word-for-word identical answers: {identical}")
    print(f"Changed prompt: source article cited only with the graph {int((paired['graph'] & ~paired['text']).sum())}, "
          f"only without it {int((paired['text'] & ~paired['graph']).sum())}")
    with_facts = answer_checks[(answer_checks["system"] == "graph") & answer_checks["graph facts given"].gt(0)]
    print(f"Graph answers with graph facts in the prompt: {len(with_facts)}; citing at least one of them: "
          f"{with_facts['graph facts cited'].gt(0).sum()}")

,cites a source,cites the source article,citation coverage,unknown citations,no answer,words
system,,,,,,
text + graph,0.981,0.779,0.735,0.0,0.160,104.343
text only,0.981,0.775,0.720,0.0,0.211,105.315


questions  cites the source article  \
prompt               system                                              
changed by the graph text + graph        109                     0.826   
                     text only           109                     0.817   
same                 text + graph        104                     0.731   
                     text only           104                     0.731   

                                   citation coverage  no answer  
prompt               system                                      
changed by the graph text + graph              0.765      0.165  
                     text only                 0.736      0.266  
same                 text + graph              0.704      0.154  
                     text only                 0.704      0.154

Same prompt: 104 questions, word-for-word identical answers: 104
Changed prompt: source article cited only with the graph 6, only without it 5
Graph answers with graph facts in the prompt: 37; citing at least one of them: 16


In [ ]:
def side_by_side(question_id):
    q = next(q for q in EVAL_SET if q["id"] == question_id)
    text_result, graph_result = answers[("text", question_id)]["result"], answers[("graph", question_id)]["result"]
    facts = "; ".join(f"[{f['label']}] {f['subject']} -> {f['relation']} -> {f['object']} ({f['date']})"
                      for f in graph_result["facts"]) or "none"
    display(Markdown(f"**[{q['type']}] {q['question']}**  \n*Reference:* {q['reference_answer']}  \n"
                     f"**Text only:** {text_result['answer']}  \n**Text + graph:** {graph_result['answer']}  \n"
                     f"*Graph facts:* {facts}"))


if len(answers) == 2 * len(EVAL_SET):
    for question_id in ["q028", "q079", "q229", "q044"]:
        side_by_side(question_id)

**[relational] Who made the Molmo models?**  
*Reference:* The Allen Institute for Artificial Intelligence (Ai2), a research nonprofit.  
**Text only:** The Molmo models were made by the Allen Institute for Artificial Intelligence (Ai2) [S2][S4][S5][S6].  
**Text + graph:** The Molmo models were made by the Allen Institute for Artificial Intelligence (Ai2) [S1][S3][S2][G1].  
*Graph facts:* [G1] OLMo / Molmo / Tülu -> developed by -> Ai2 (2024-09-26)

**[relational] Who makes the Kimi chatbot?**  
*Reference:* Moonshot.  
**Text only:** Moonshot AI makes the Kimi chatbot [S1][S3][S5][S6][S4][S2].  
**Text + graph:** The Kimi chatbot is made by Moonshot AI [S1][S3][S4][S5][S6][G1].  
*Graph facts:* [G1] Kimi -> developed by -> MoonShot AI (2025-02-04)

**[temporal] When did Devendra Singh Chaplot join xAI?**  
*Reference:* He announced on X on 14 March 2026 that he was joining xAI and SpaceX.  
**Text only:** The sources do not specify when Devendra Singh Chaplot joined xAI. While [S1] states that on March 14, 2026, Devendra Singh Chaplot announced on X that he is joining xAI and SpaceX, it does not indicate the exact date of his joining. Therefore, based on the available information, the specific date of his joining xAI is not provided in the sources. [S1] only confirms the announcement date, not the start date of his employment. 

Thus, the exact date when Devendra Singh Chaplot joined xAI cannot be determined from the sources.  
**Text + graph:** Devendra Singh Chaplot joined xAI on March 14, 2026 [S1][S2]. This date is supported by the announcement on X and the reporting in FintechWeekly, which states that on March 14, Chaplot announced his joining of xAI and SpaceX to work directly with Elon Musk [S1]. The graph fact also confirms his employment at xAI with the date of 2026-03-14 [G1].  
*Graph facts:* [G1] Devendra Singh Chaplot -> works for -> xAI (2026-03-14)

**[relational] Who developed Qwen-72B and Yi-34B?**  
*Reference:* Qwen-72B was developed by Alibaba and Yi-34B by 01.AI.  
**Text only:** The sources do not provide information about who developed Qwen-72B or Yi-34B. While [S3] mentions that Qwen-72B was developed by Alibaba and that a US firm released Smaug-72B based on it, it does not specify the developer of Qwen-72B itself. Similarly, there is no mention of Yi-34B in any of the provided sources. Therefore, based on the available information, the developer of Qwen-72B or Yi-34B cannot be identified. [S3] supports only that Qwen-72B is a model from Alibaba, but does not confirm the developer of the 72B or 34B versions.  
**Text + graph:** Qwen-72B was developed by Alibaba [S4][S6]. Yi-34B was developed by 01.AI [S3].  
*Graph facts:* none

The graph does not change how often the answers cite the source article: on the questions whose prompt it changes,
the source article is cited in about as many answers with and without the graph (0.83 and 0.82). It changes how the
answers use their evidence: with the graph, fewer answers decline (18 instead of 29 say that the sources are not
enough) and a larger share of their sentences carries a citation (0.76 instead of 0.74). Most questions answered only
with the graph get the reference answer, e.g. the speed of Llama 3.1 405B on SambaNova, the developers of Qwen-72B
and Yi-34B, or how NVIDIA made Nemotron 3 Nano 4B. On the questions whose prompt the graph leaves unchanged, both
systems give word-for-word the same answer, so the differences come from the graph. Both systems cite a source in 98%
of the answers and never cite a label that was not in the prompt.

Graph facts are cited in 16 of the 37 answers that receive them. They help most when the fact is the answer
(*Who made the Molmo models?* -> Ai2, *Who makes the Kimi chatbot?* -> Moonshot AI, *When did Devendra Singh Chaplot
join xAI?* -> `works for` xAI, 14 March 2026), and the model cites the fact next to the passage that states it. The
examples below show these answers and one that the graph improves through retrieval alone (*Who developed Qwen-72B
and Yi-34B?*).

## 4.6 Summary

In [ ]:
changed_checks = answer_checks[answer_checks["prompt"].eq("changed by the graph")].groupby("system")
declines, coverage = changed_checks["no answer"].sum().astype(int), changed_checks["citation coverage"].mean()
display(Markdown(f'''
| Stage 1 feedback | Knowledge graph v2 |
|---|---|
| keep lexicon knowledge separate from article claims | every edge has a knowledge layer; {int(edges["lexicon_agreement"].eq("lexicon_and_articles").sum())} former Tier A edges are article claims, {int(edges["lexicon_agreement"].eq("lexicon_only").sum())} lexicon-only facts are background that is never cited or put in a prompt |
| validate the relations used for retrieval | every evidence sentence is checked against its relation, and fragment nodes are excluded: precision {other_rs["precision without check"]:.2f} -> {precision_used:.2f} on the human random sample, with every correct relation kept; `competes with` is answered from text; {int(edges["retrieval_eligible"].sum())} of {len(edges)} edges are used for retrieval |
| check that passages support the relations | only sentences that state the relation are cited: in a blind review, {share(review_before)} of the passages before and {share(review_after)} after state their relation; in the RAG prompt, a graph fact always comes with its sentence |

| Requirement | Result |
|---|---|
| Knowledge graph integration | knowledge graph v2 (verified relations, entity mentions) is added to the text-only system of Section 2 without changing its corpus, chunks, search, prompt or model |
| Graph-aware or entity-based retrieval | questions are linked to graph nodes by name, alias, version and family; an entity ranking and a relation ranking are fused with the text ranking |
| Combine text and graph evidence | graph facts enter the prompt only with the passage that states them; lexicon facts stay out |
| Compare against the text-only baseline | retrieval: MRR {table.loc["text only", "mrr"]:.2f} -> {table.loc["text + graph", "mrr"]:.2f}, hit@10 {table.loc["text only", "hit@10"]:.2f} -> {table.loc["text + graph", "hit@10"]:.2f}; answers (questions whose prompt the graph changes): source article cited as often, {declines["text"]} -> {declines["graph"]} answers that decline, citation coverage {coverage["text"]:.2f} -> {coverage["graph"]:.2f} |

The answers of both systems are saved with their evidence, so that the evaluation (Section 5) can score faithfulness,
relevance, context use and temporal grounding with a judge model on the same answers.
'''))


| Stage 1 feedback | Knowledge graph v2 |
|---|---|
| keep lexicon knowledge separate from article claims | every edge has a knowledge layer; 15 former Tier A edges are article claims, 18 lexicon-only facts are background that is never cited or put in a prompt |
| validate the relations used for retrieval | every evidence sentence is checked against its relation, and fragment nodes are excluded: precision 0.71 -> 0.80 on the human random sample, with every correct relation kept; `competes with` is answered from text; 801 of 999 edges are used for retrieval |
| check that passages support the relations | only sentences that state the relation are cited: in a blind review, 86% of the passages before and 96% after state their relation; in the RAG prompt, a graph fact always comes with its sentence |

| Requirement | Result |
|---|---|
| Knowledge graph integration | knowledge graph v2 (verified relations, entity mentions) is added to the text-only system of Section 2 without changing its corpus, chunks, search, prompt or model |
| Graph-aware or entity-based retrieval | questions are linked to graph nodes by name, alias, version and family; an entity ranking and a relation ranking are fused with the text ranking |
| Combine text and graph evidence | graph facts enter the prompt only with the passage that states them; lexicon facts stay out |
| Compare against the text-only baseline | retrieval: MRR 0.69 -> 0.71, hit@10 0.88 -> 0.91; answers (questions whose prompt the graph changes): source article cited as often, 29 -> 18 answers that decline, citation coverage 0.74 -> 0.76 |

The answers of both systems are saved with their evidence, so that the evaluation (Section 5) can score faithfulness,
relevance, context use and temporal grounding with a judge model on the same answers.


## 4.7 Saved Outputs

| File | Content |
|---|---|
| `outputs/stage3_kg_v2/` | knowledge graph v2: nodes, edges, assertions, relations used for retrieval, lexicon facts, text units, documents, GraphML |
| `artifacts/stage3_wp2/kg_revision/` | NLI score cache, agreement with the Stage 1 labels, edges with their knowledge layer and support status, retrieval check and blind review sheets |
| `OUT_DIR/retrieval_results.csv` | per question: route, linked entities, question intents, reciprocal rank of the source article with text only and with text + graph |
| `OUT_DIR/answers.jsonl` | the answers of both systems, with the full `synthesize_answer` result (sources, graph facts, citation checks) |
| `OUT_DIR/answer_checks.csv` | the citation checks of Section 4.5.2 per answer |
| `OUT_DIR/graph_rag_settings.json` | the settings of the graph-enhanced system, next to `rag_settings.json` of Section 2 |

In [ ]:
retrieval_results = pd.DataFrame([{
    "id": q["id"], "type": q["type"], "question": q["question"], "route": runs[q["id"]]["route"],
    "entities": "; ".join(e["name"] for e in runs[q["id"]]["entities"]),
    "intents": "; ".join(runs[q["id"]]["intents"]),
    "rr_text": rank_text[q["id"]], "rr_text_graph": rank_graph[q["id"]],
} for q in EVAL_SET])
retrieval_results.to_csv(OUT_DIR / "retrieval_results.csv", index=False)

from stage3_graph_rag import fusion
GRAPH_RAG_SETTINGS = {
    "baseline": "rag_settings.json of Section 2 (corpus, chunks, embeddings, retrieval, generation, model)",
    "knowledge_graph": {"graph": "knowledge graph v2 (Section 3, outputs/stage3_kg_v2)",
                        "relations_used": len(graph_relations), "entity_mentions": "stage1_wp2/entity_mentions.csv"},
    "linking": {"named_types": sorted(NAMED_TYPES), "family_patterns": "Stage 1 lexicon", "exceptions": FAMILY_EXCEPTIONS},
    "fusion": {"rankings": ["text", "entity", "relation"], "rrf_k": fusion.RRF_K, "entity_pool": fusion.POOL_SIZE,
               "size": RETRIEVAL_ARGS["size"]},
    "prompt": {"graph_facts": "evidence chunk among the prompt passages", "max_facts": graph_rag.max_facts,
               "lexicon_facts": "not used"},
    "intents": sorted(fusion.INTENTS),
}
(OUT_DIR / "graph_rag_settings.json").write_text(json.dumps(GRAPH_RAG_SETTINGS, indent=2), encoding="utf-8")
print(json.dumps(GRAPH_RAG_SETTINGS, indent=2))

{
  "baseline": "rag_settings.json of Section 2 (corpus, chunks, embeddings, retrieval, generation, model)",
  "knowledge_graph": {
    "graph": "knowledge graph v2 (Section 3, outputs/stage3_kg_v2)",
    "relations_used": 4173,
    "entity_mentions": "stage1_wp2/entity_mentions.csv"
  },
  "linking": {
    "named_types": [
      "AI_MODEL",
      "BENCHMARK",
      "COMPANY",
      "DATASET",
      "FOUNDATION_MODEL",
      "GPE",
      "HARDWARE",
      "LAW",
      "LICENSE",
      "ORG",
      "PERSON",
      "PLATFORM",
      "PRODUCT",
      "STANDARD",
      "TECHNOLOGY"
    ],
    "family_patterns": "Stage 1 lexicon",
    "exceptions": {
      "Llama": "\\bLlama[\\w.\\- ]{0,15}?(?:Nemotron|Stack)\\b|\\bLlamaCon\\b"
    }
  },
  "fusion": {
    "rankings": [
      "text",
      "entity",
      "relation"
    ],
    "rrf_k": 60,
    "entity_pool": 100,
    "size": 30
  },
  "prompt": {
    "graph_facts": "evidence chunk among the prompt passages",
    "max_facts": 5,
    "lexicon

# 5. System Evaluation and Enhancement
Part 1 built the Q&A test set and the text-only RAG baseline; Part 2 added the Stage 1 knowledge graph and answered every test question with both systems. This part measures the quality of those answers, finds where the systems fail, and fixes one of the failures:
1. **Test set and question categories**: the questions are labelled with the three categories of the Q&A session of 30 October (factoid, multi-hop, global), and 30 multi-hop and global questions are drafted and reviewed by hand (23 kept), since the Part 1 questions all come from a single passage.
2. **Answers of the two systems**: the answers saved by Part 2 (text-only and text + graph) are loaded, and the new questions are answered with the same pipelines.
3. **Evaluation with a judge model**: every answer is scored on faithfulness, relevance, context grounding, temporal grounding and correctness by a model that is different from the RAG model.
4. **Human check of the judge**: a sample of the judgments is scored by hand, to know how far the judge can be trusted.
5. **Retrieval diagnosis**: where the answers fail, is it retrieval or generation?
6. **Enhancement**: one limitation observed in the evaluation is addressed with one change to the pipeline, which is evaluated the same way.
7. **Summary**: trade-offs between the systems and what RAG with a knowledge graph brings to trend monitoring in the open-models track.

**Key results**
- **The graph improves relevance and reduces refusals.** Over 236 questions, text + graph RAG scores higher than text-only RAG on every criterion; the gain in relevance is reliable (+0.08 on a 1–5 scale, 95% interval [+0.00, +0.15]) and refusals drop from 22% to 16%. Correctness rises by +0.06 ([−0.07, +0.18]); on the 18 multi-hop questions the two systems reach the same correctness (3.39).
- **The failures are retrieval failures.** When a source article is in the prompt, correctness is 4.2; when it is not, 3.3, and the model answers "the sources do not say" twice as often. Hybrid search puts a source article first for 58% of the factoid questions, 33% of the multi-hop and 20% of the global ones; for the global questions it finds 14% of the articles they need.
- **Re-ranking the retrieved passages with a cross-encoder is the one change that moves correctness**: +0.22 over the text-only baseline ([+0.00, +0.40], p = 0.02), better than the graph (+0.16, p = 0.06), with the same model and prompt. It improves the factoid questions (4.11 → 4.37) and leaves the multi-hop and global ones no better, because it ranks passages one at a time and favours one article over several.
- **Sparse search alone beats the hybrid search of Part 1** on this test set (MRR 0.73 against 0.67), a consequence of questions that share words with their passages even after paraphrasing.
- **The judge is checked, not trusted.** A model of a different family from the RAG model scores every answer; on a blind sample of 30 scored by hand it agrees with the reviewer within one point for 90–97% of the answers but is about 0.4 points more lenient on correctness, and the reviewer sees no difference between the two systems (Section 5.4); a higher reasoning effort shifts both systems alike on correctness and relevance (Section 5.4.1).

**Setup.** A GPU runtime is needed only to generate new answers with the local RAG model, and the judge runs through
the OpenAI API with its key in the Colab secrets (`OPENAI_API_KEY`). The corpus is read from the shared Google Drive,
the test set, the RAG settings and the saved answers of Sections 1 – 4 from the repository; every long step
(generation, judging) is saved on Drive (`OUT_DIR`) and loaded at later runs.

In [ ]:
# anatoolbox
%pip install -q "anatoolbox[embeddings] @ git+https://github.com/umagira/anatoolbox" openpyxl

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", context="notebook")
SEED = 42
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Clone the project repository
    REPO = Path("/content/computational-language-technology-hs26")
    if REPO.exists():
        !git -C {REPO} pull -q
    else:
        !git clone -q https://github.com/jihyeon26/computational-language-technology-hs26.git {REPO}
    # Mount Google Drive (input data and outputs)
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/HSLU/CLT")
    try:
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")      # Colab secret (key icon on the left)
    except Exception:                                        # no secret yet: everything except the judge still runs
        OPENAI_API_KEY = None
else:                                                         # local run: CLT_REPO and CLT_DRIVE point to the same folders
    REPO = Path(os.environ["CLT_REPO"])
    DRIVE = Path(os.environ["CLT_DRIVE"])
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

DATA_DIR = DRIVE / "data/processed"                                   # Stage 1 outputs
RAG_DIR = DRIVE / "extraction_outputs/stage3/wp1/rag"                 # Part 1: corpus file and chunk embeddings
OUT_DIR = DRIVE / "extraction_outputs/stage3/wp3"                     # outputs of this part (also used as cache)
FIG = OUT_DIR / "figures"
for directory in (RAG_DIR, FIG):
    directory.mkdir(parents=True, exist_ok=True)
PART1_ART = REPO / "artifacts/stage3_wp1"                               # Part 1: test set, RAG settings
PART2_ART = REPO / "artifacts/stage3_wp2/graph_rag"                     # Part 2: answers of both systems, retrieval results
PART3_ART = REPO / "artifacts/stage3_wp3"                               # this part: small outputs copied to the repository

RAG_SETTINGS = json.loads((PART1_ART / "rag_settings.json").read_text(encoding="utf-8"))
RAG_MODEL = RAG_SETTINGS["rag_model"]["model"]                        # qwen3:4b-instruct (Ollama)
JUDGE_MODEL = "gpt-5-mini"                                            # judge: a different model family from the RAG model
TYPES = ["factual", "temporal", "relational", "analytical", "comparative"]
CATEGORIES = ["factoid", "multi-hop", "global"]
SYSTEMS = {"text": "text-only RAG", "graph": "text + graph RAG"}

def save_fig(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

print(f"RAG model: {RAG_MODEL} | judge model: {JUDGE_MODEL} | key for the judge: {'found' if OPENAI_API_KEY else 'missing'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RAG model: qwen3:4b-instruct | judge model: gpt-5-mini | key for the judge: missing


## 5.1 Test Set and Question Categories
The Q&A session of 30 October asked to label every question by the information needed to answer it, so that the comparison between text-only RAG and graph RAG shows where each one helps:

| Category | Information needed | In the test set |
|---|---|---|
| **factoid / definition** | one passage: a fact or a definition | the 213 questions of Part 1, each drafted from one passage |
| **multi-hop / relational** | two or more passages that have to be connected | 18 new questions, each drafted from passages of two to three different articles |
| **global / summary** | a synthesis across several passages or a broader theme | 5 new questions, each drafted from four or five passages of as many articles on one theme |

The new questions follow the method of Part 1: 30 were drafted by Claude (Anthropic) from the 150 passages of Part 1 (`passages_for_qa.csv`), grouped by model family and event, with a reference answer and one evidence sentence per passage; the five question types of Part 1 are kept. Every question was then reviewed by hand with the criteria of Part 1 (clarity, correctness, answerability, naturalness) and two criteria for the new categories: a multi-hop question must really need all its passages (covering one passage, can the question still be answered?), and a global question must ask for a synthesis rather than one fact. Three measurements supported the review: the share of the answer's content words found in the cited passages, the share found in the best single passage, and the wording overlap between the question and its passages. As in Part 1, questions were kept, edited or dropped (`qa_multihop_reviewed.xlsx`): **20 kept, 3 edited, 7 dropped**. The drops were questions whose second passage was decorative (an unrelated fact glued to a one-passage question), one answered by a single sentence of one passage, one whose answer stated more than its passages, and one definition question mislabelled as global; the edits reworded two questions that copied their passages' wording or referred to an unnamed report, and moved one four-article question from multi-hop to global. The source articles of a question (`source_ids`) are all the articles its passages come from, so retrieval is scored on several relevant articles for the new questions.

In [ ]:
# Part 1 test set: single-passage questions
factoid = json.loads((PART1_ART / "qa_reviewed.json").read_text(encoding="utf-8"))
for q in factoid:
    q["category"] = "factoid"

def part3_file(name):
    # A file of this part: in the repository (artifacts/stage3_wp3) or, before it is pushed, in OUT_DIR on Drive
    return next((p for p in (PART3_ART / name, OUT_DIR / name) if p.exists()), PART3_ART / name)

# New questions after the manual review. REQUIRE_REVIEW stops the notebook rather than evaluating the
# unreviewed draft by mistake; set it to False only to try the notebook out before the review is done.
REQUIRE_REVIEW = True
REVIEWED_XLSX = next((p for name in ("qa_multihop_reviewed.xlsx", "qa_multihop_review.xlsx")
                      if (p := part3_file(name)).exists()), None)
DRAFT_CSV = part3_file("qa_multihop_draft.csv")
if REVIEWED_XLSX is not None:
    review = pd.read_excel(REVIEWED_XLSX, sheet_name="qa_multihop", dtype=str).fillna("")
    print(f"Manual review read from {REVIEWED_XLSX}")
else:
    assert not REQUIRE_REVIEW, (
        "The reviewed questions were not found, and the unreviewed draft would be evaluated instead. Put "
        "qa_multihop_reviewed.xlsx in one of these two folders and run again:\n"
        f"  {PART3_ART}\n  {OUT_DIR}\n"
        "(set REQUIRE_REVIEW = False in this cell to run on the draft on purpose.)")
    review = pd.read_csv(DRAFT_CSV, dtype=str).fillna("")
    print("Reviewed file not found: the draft is used, every question counted as kept")
    review["review_status"] = "keep"
review["review_status"] = review["review_status"].str.strip().str.lower().replace("", "keep")
print("Review of the new questions:", review["review_status"].value_counts().to_dict())
kept = review[review["review_status"].isin(["keep", "edit"])]
new_questions = [{"id": r["qa_id"], "type": r["type"], "category": r["category"], "question": r["question"],
                  "reference_answer": r["reference_answer"], "source_ids": json.loads(r["source_ids"]),
                  "passage_ids": json.loads(r["passage_ids"]), "date": r["dates"].split(";")[0].strip()}
                 for _, r in kept.iterrows()]
(OUT_DIR / "qa_multihop.json").write_text(json.dumps(new_questions, indent=1, ensure_ascii=False), encoding="utf-8")

EVAL_SET = factoid + new_questions
BY_ID = {q["id"]: q for q in EVAL_SET}
(OUT_DIR / "test_set_v2.json").write_text(json.dumps(EVAL_SET, indent=1, ensure_ascii=False), encoding="utf-8")
counts = pd.crosstab(pd.Series([q["category"] for q in EVAL_SET], name="category"),
                     pd.Series([q["type"] for q in EVAL_SET], name="type"), margins=True, margins_name="total")
display(counts.reindex(CATEGORIES + ["total"]).reindex(columns=TYPES + ["total"]))
print(f"Test set: {len(EVAL_SET)} questions | source articles per question: "
      f"{pd.Series([len(q['source_ids']) for q in EVAL_SET]).value_counts().sort_index().to_dict()}")

Manual review read from /content/computational-language-technology-hs26/artifacts/stage3_wp3/qa_multihop_reviewed.xlsx
Review of the new questions: {'keep': 20, 'drop': 7, 'edit': 3}


type,factual,temporal,relational,analytical,comparative,total
category,,,,,,
factoid,57,38,48,27,43,213
multi-hop,2,4,3,8,1,18
global,0,2,2,1,0,5
total,59,44,53,36,44,236


Test set: 236 questions | source articles per question: {1: 213, 2: 15, 3: 3, 4: 3, 5: 2}


The test set has 236 questions: the 213 single-passage questions of Part 1, all factoid by the definition of the Q&A session, and 23 new ones, 18 multi-hop and 5 global. The review of the 30 drafted questions kept 20, edited 3 and dropped 7; the drops are the sign that the two new criteria bite: six of the seven dropped questions read well and had a correct answer, and failed on *needing* their second passage or on sounding like a question a reader would ask; the seventh was dropped because its answer stated more than its passages. The five types of Part 1 are all present among the multi-hop questions; the global questions have no factual and no comparative one, but the distribution is different: analytical questions are 13% of the factoid set and 39% of the new one, because connecting two articles usually asks *why* or *how*, not *when*. The source articles per question are the other difference: every Part 1 question has one, the new ones have two to five, so retrieval is scored on several relevant articles for them, and hit@k is read next to recall@k from Section 5.5 on. The word *relational* appears in both labelings and means two different things: the Part 1 type says the question asks about a relation, the category says the relation is stated across two passages; 53 questions are relational by type, 3 of them multi-hop by category.

### 5.1.1 How close are the questions to their passages?
The Q&A session also warned against questions that reuse the wording of their passage: retrieval then finds the passage by its words and the evaluation is too optimistic. The share of a question's content words that appear in its source passages measures this (stop words and the model names, which a question has to keep, are left out). A low share means the question is a paraphrase; the new questions are compared with those of Part 1.

In [ ]:
passages = pd.read_csv(PART1_ART / "passages_for_qa.csv").set_index("passage_id")
lexicon = json.loads((REPO / "artifacts/stage1_wp1/lexicon/open_models_lexicon.json").read_text(encoding="utf-8"))
STOP = set("""a an the and or of to in on for with by from at as is are was were be been being what which who whom whose when
where why how did do does done has have had having it its this that these those than then there their they them he she his
her we you your our i me my not no also into about after before between over under during according say said says report
reported article articles model models ai company companies new one two three first last more most much many some any
each per""".split())
NAMES = {t for f in lexicon["families"].values() for t in re.findall(r"[a-z0-9]+", f["label"].lower())} | {"openai", "google", "meta", "microsoft", "nvidia", "alibaba", "deepseek", "mistral", "anthropic", "hugging", "face"}
words = lambda text: {w for w in re.findall(r"[a-z0-9][a-z0-9\-\.]*", str(text).lower())} - STOP - NAMES

def overlap(q):
    ids = q.get("passage_ids") or [q["passage_id"]]
    source = set().union(*(words(passages.loc[p, "text"]) for p in ids if p in passages.index))
    content = words(q["question"])
    return len(content & source) / len(content) if content else np.nan

overlaps = pd.DataFrame({"category": [q["category"] for q in EVAL_SET], "type": [q["type"] for q in EVAL_SET],
                         "overlap": [overlap(q) for q in EVAL_SET]})
display(overlaps.groupby("category")["overlap"].describe()[["count", "mean", "50%", "min", "max"]].round(2)
        .reindex(CATEGORIES).rename(columns={"50%": "median"}))
display(overlaps.groupby("type")["overlap"].mean().round(2).reindex(TYPES).to_frame("mean overlap").T)

,count,mean,median,min,max
category,,,,,
factoid,213.0,0.58,0.67,0.0,1.00
multi-hop,18.0,0.48,0.45,0.2,0.80
global,5.0,0.42,0.50,0.2,0.67


type,factual,temporal,relational,analytical,comparative
mean overlap,0.65,0.47,0.56,0.57,0.57


The questions reuse about half of the content words of their passages: a mean overlap of 0.58 for the factoid questions (median 0.67), 0.48 for the multi-hop and 0.42 for the global ones. The new questions are therefore further from the wording of their sources than those of Part 1, in part because the review reworded the question with the highest overlap, which used every content word of its passage. The maximum for the factoid questions is 1.00, so some Part 1 questions can be found by their words alone; the professor's warning that this makes retrieval "artificially easy" applies to them, and Section 5.5 shows its effect: sparse (word-based) search outperforms the hybrid search on this test set. By type, factual questions have the highest overlap (0.65) and temporal questions the lowest (0.47), since a question about *when* does not repeat the date it asks for. The overlap is a property of the test set, not of the systems, and it is the same for both, so it does not bias their comparison; it does make the absolute retrieval figures optimistic.

## 5.2 Answers of the Two Systems
Part 2 answered the 213 questions of Part 1 with both systems and saved every answer with its evidence: the passages shown to the model (`[S1]…`), the graph facts (`[G1]…`), the citations and their checks (`answers.jsonl`). Those answers are evaluated as they are, so that the scores of this part describe the systems of Parts 1 and 2 and not a new run.

The new questions are answered here with the same pipeline, model and settings as Parts 1 and 2 (`rag_settings.json`, `graph_rag_settings.json`), with the local RAG model on the Colab GPU: the text-only answers in Section 5.2.1 and the graph answers in Section 5.2.2, which rebuilds the graph retriever of Part 2 from the repository. The RAG corpus and its chunks are rebuilt from the Stage 1 corpus with the Part 1 settings; the chunk embeddings of Part 1 are loaded from Google Drive (or computed again when missing, which takes a few minutes on a GPU).

In [ ]:
import anatoolbox
from anatoolbox import ToolContext, resolve_tools
from anatoolbox.corpus import configure_embedder, ensure_embeddings, get_corpus, load_embeddings, save_embeddings
from anatoolbox.llm_client import ModelSpec, call_llm_text, configure_llm, model_for
from sentence_transformers import SentenceTransformer

anatoolbox.register_reference_tools(replace=True)
ctx = ToolContext()
ingest, chunk, retrieve, rerank, synthesize, judge, retrieval_metrics = resolve_tools(
    ["ingest_corpus", "chunk_by_size", "retrieve_passages", "rerank_passages", "synthesize_answer",
     "score_rag_answer", "calculate_retrieval_metrics"])

# RAG corpus of Part 1: the 1,210 core articles, written again from the Stage 1 corpus if the file is missing
CORE_CSV = RAG_DIR / RAG_SETTINGS["corpus"]["file"]
if not CORE_CSV.exists():
    corpus = pd.read_parquet(DATA_DIR / "corpus_clean.parquet",
                             columns=["doc_id", "date", "domain", "url", "title", "text", "om_tier"])
    core = corpus[corpus["om_tier"] == "core"]
    pd.DataFrame({"id": core["doc_id"], "title": core["title"], "text": core["text"],
                  "date": pd.to_datetime(core["date"]).dt.strftime("%Y-%m-%d"),
                  "url": core["url"], "domain": core["domain"]}).to_csv(CORE_CSV, index=False)
rag_articles = ingest.run({"path": str(CORE_CSV), "name": "core_articles"}, context=ctx)
chunks = chunk.run({"input": rag_articles, **RAG_SETTINGS["chunks"]}, context=ctx)
chunk_corpus = get_corpus(chunks["corpus"])
CHUNK = {r["id"]: r for r in chunk_corpus.records}

# Chunk embeddings of Part 1
embedding_model = SentenceTransformer(RAG_SETTINGS["embedding_model"])
configure_embedder(lambda texts: embedding_model.encode(texts, batch_size=64, normalize_embeddings=True,
                                                        show_progress_bar=len(texts) > 1000))
EMB_PATH = RAG_DIR / RAG_SETTINGS["embeddings_file"]
if EMB_PATH.exists():
    load_embeddings(chunk_corpus, EMB_PATH)
else:
    ensure_embeddings(chunk_corpus)
    save_embeddings(chunk_corpus, EMB_PATH)

RETRIEVAL_ARGS = RAG_SETTINGS["retrieval"]
ANSWER_ARGS = RAG_SETTINGS["generation"]

def search(question, strategy=RETRIEVAL_ARGS["strategy"], size=RETRIEVAL_ARGS["size"], **args):
    # The Part 1 search: the `size` best chunks for `question`
    return retrieve.run({"query": question, "input": chunks, "strategy": strategy, "size": size, **args}, context=ctx)

print(f"{chunks['chunks']:,} chunks from {chunks['articles']:,} articles ({chunk_corpus.name}) | "
      f"retrieval {RETRIEVAL_ARGS} | generation {ANSWER_ARGS}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

12,720 chunks from 1,210 articles (core_articles_size200_overlap40_title) | retrieval {'strategy': 'hybrid', 'size': 30} | generation {'mode': 'grounded', 'max_passages': 6, 'max_per_source': 2, 'max_chars_per_passage': 1200, 'max_tokens': 400}


### 5.2.1 Saved answers and answers for the new questions
Every answer is kept in one place, `answers`, keyed by system and question: the Part 2 file first, then the answers generated in this part (`answers_wp3.jsonl` on Drive, appended at every run). The language model is only started when answers are missing.

In [ ]:
ANSWER_FILES = [PART2_ART / "answers.jsonl",                   # Part 2: the 213 questions of Part 1 × 2 systems
                PART2_ART / "answers_multihop.jsonl",          # Part 2 notebook run on the new questions, if it was run
                OUT_DIR / "answers_wp3.jsonl"]               # this part: missing text answers, enhanced system (Section 5.6)
answers, stale = {}, set()
for path in ANSWER_FILES:
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            record = json.loads(line)
            if record["id"] not in BY_ID:                      # a question dropped by the manual review
                continue
            if record.get("question", BY_ID[record["id"]]["question"]) != BY_ID[record["id"]]["question"]:
                stale.add(record["id"])                        # the review reworded it: the cached answer is not its answer
                answers.pop((record["system"], record["id"]), None)
                continue
            answers[(record["system"], record["id"])] = record
if stale:
    print(f"Reworded by the review, so the cached answers are ignored and generated again: {sorted(stale)}")

def available(systems):
    # Answers available per system and category
    return pd.DataFrame({name: {c: sum((s, q["id"]) in answers for q in EVAL_SET if q["category"] == c) for c in CATEGORIES}
                         for s, name in systems.items()}).T.assign(total=lambda d: d.sum(axis=1))

display(available(SYSTEMS))
todo = [q for q in EVAL_SET if ("text", q["id"]) not in answers]
print(f"Text-only answers to generate: {len(todo)}")

Reworded by the review, so the cached answers are ignored and generated again: ['m04', 'm13']


,factoid,multi-hop,global,total
text-only RAG,213,18,5,236
text + graph RAG,213,18,5,236


Text-only answers to generate: 0


In [ ]:
def use_rag_model():
    # Point the anatoolbox tools at the local RAG model of Part 1 (Ollama)
    configure_llm(base_url="http://localhost:11434/v1", api_key="not-needed", model=RAG_MODEL, timeout=180, max_retries=2)

def start_ollama():
    # Install and start the local model server of Part 1 (Colab), and pull the RAG model
    if IN_COLAB and not shutil.which("ollama"):
        subprocess.run("apt-get -qq install -y zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh > /dev/null", shell=True, check=True)
    server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    subprocess.run(["ollama", "pull", RAG_MODEL], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    use_rag_model()
    print(f"{RAG_MODEL}: {call_llm_text('Reply with one word.', 'Say: ready', model=model_for('default'), max_tokens=20)!r}")
    return server

def generate(system, q, inputs):
    # Answer `q` from `inputs` with the generation settings of Part 1, in the format of the Part 2 answers
    # `inputs`: {"input": passages, "facts": [...] (graph system), "route": …, "entities": […]}
    started = time.perf_counter()
    args = {k: v for k, v in inputs.items() if k in {"input", "facts"}}
    result = synthesize.run({"question": q["question"], **args, **ANSWER_ARGS}, context=ctx)
    return {"system": system, "id": q["id"], "type": q["type"], "question": q["question"],
            "seconds": round(time.perf_counter() - started, 1), "route": inputs.get("route", "text"),
            "entities": list(inputs.get("entities", [])), "result": result}

def generate_all(system, questions, inputs_of):
    # Generate and save the missing answers of `system`; `inputs_of(q)` retrieves what the model is given
    missing = [q for q in questions if (system, q["id"]) not in answers]
    if not missing:
        return
    if not (IN_COLAB or shutil.which("ollama")):
        print(f"{system}: {len(missing)} answers missing; Ollama is not available here, run this cell in Colab")
        return
    global ollama_server
    if "ollama_server" not in globals():
        ollama_server = start_ollama()
    use_rag_model()
    started = time.perf_counter()
    with (OUT_DIR / "answers_wp3.jsonl").open("a", encoding="utf-8") as handle:
        for n, q in enumerate(missing, start=1):
            record = generate(system, q, inputs_of(q))
            answers[(system, q["id"])] = record
            handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
            handle.flush()
            if n % 25 == 0:
                print(f"{n}/{len(missing)} answers, {time.perf_counter() - started:.0f} s")
    print(f"{system}: {len(missing)} answers generated in {time.perf_counter() - started:.0f} s")

generate_all("text", EVAL_SET, lambda q: {"input": search(q["question"])})
display(available(SYSTEMS))

,factoid,multi-hop,global,total
text-only RAG,213,18,5,236
text + graph RAG,213,18,5,236


### 5.2.2 The Graph System on the New Questions
The answers of Part 2 cover its own test set, the 213 questions of Part 1. The new questions have to be answered by the graph system too, otherwise the comparison between the two systems exists only for the factoid category, which is the category where a knowledge graph is least expected to help.

The graph retriever of Part 2 is therefore rebuilt here, with the code of Part 2 itself (`src/stage3_graph_rag`, `src/stage3_kg_revision`) and the Stage 1 tables in the repository (`artifacts/stage3_wp2/stage1_inputs`). Two things make the rebuild exact rather than approximate: the evidence check of Part 2 is served from its cached entailment scores (`nli_scores.csv`), so the NLI model is not loaded and the same sentences are judged to support the same relations; and the rebuilt edges are compared with the edges Part 2 published (`edges_v2.csv`), which asserts that the set of relations used for retrieval is identical. Entity linking, the fusion of the three rankings and the choice of graph facts are the Part 2 code, with the settings of `graph_rag_settings.json`.

If Part 2 is itself run on the new questions and saves `answers_multihop.jsonl`, that file is read in Section 5.2.1 and this rebuild is skipped.

In [ ]:
GRAPH_TODO = [q for q in new_questions if ("graph", q["id"]) not in answers]
print(f"Graph answers missing for {len(GRAPH_TODO)} of the {len(new_questions)} new questions")

# The rebuild and its check against Part 2 run at every run (about a minute): they are the evidence that the
# graph answers of this part come from the graph of Part 2. The retriever itself is only built if answers are missing.
sys.path.insert(0, str(REPO / "src"))
from stage3_graph_rag import EntityLinker, GraphTextRetriever, node_mentions
from stage3_graph_rag.linking import GENERIC_NAMES
from stage3_kg_revision import (TRIPLE, NLIVerifier, aggregate_support, article_evidence, build_pairs,
                                edge_layers, label_assertions, lexicon_conflicts, load_review_labels,
                                score_pairs, with_key)

GRAPH_SETTINGS = json.loads((PART2_ART / "graph_rag_settings.json").read_text(encoding="utf-8"))
STAGE1_KG, STAGE1_EXTRACTION = REPO / "artifacts/stage3_wp2/stage1_inputs", REPO / "artifacts/stage1_wp2"
KG_REVISION = REPO / "artifacts/stage3_wp2/kg_revision"
NLI_MODEL = json.loads((KG_REVISION / "manifest.json").read_text(encoding="utf-8"))["nli_model"]
SUPPORT_THRESHOLD = 0.5                                      # the threshold fixed by Part 2

nodes = pd.read_parquet(STAGE1_KG / "nodes.parquet")
assertions = label_assertions(pd.read_parquet(STAGE1_KG / "typed_edges_assertions.parquet"))
collapsed = pd.read_parquet(STAGE1_KG / "typed_edges_collapsed.parquet")
labels_of = nodes.set_index("entity_id")["canonical_name"]

# Evidence check of Part 2, served from its cache of entailment scores
edges = edge_layers(collapsed, assertions)
conflicts = lexicon_conflicts(pd.read_parquet(STAGE1_KG / "relations_all_tiers.parquet"))
review_labels = load_review_labels(REPO / "artifacts/stage1_wp3/review")
keys = (set(with_key(edges[edges["has_article_claim"]])["key"]) | set(review_labels["key"])
        | set(with_key(conflicts.rename(columns={"head_id": "subject_id", "relation": "predicate",
                                                 "tail_id": "object_id"}))["key"]))
evidence = article_evidence(pd.read_parquet(STAGE1_KG / "evidence_status.parquet"))
pool = evidence[evidence["key"].isin(keys)].reset_index(drop=True)
pairs = build_pairs(pool, head_name="subject", tail_name="object")
scored = score_pairs(pairs, NLIVerifier(NLI_MODEL), KG_REVISION / "nli_scores.csv", NLI_MODEL)
assert scored["entailment"].notna().all(), "entailment scores missing from the cache of Part 2"
print(f"Entailment check: {len(pairs):,} sentence-relation pairs, all served from the cache of Part 2 (no model loaded)")
sentence_support, edge_support = aggregate_support(pool, scored, ["key"], threshold=SUPPORT_THRESHOLD)

# Knowledge layers and the retrieval filter of Part 2
edges = with_key(edges).merge(edge_support, on="key", how="left", validate="one_to_one")
edges.loc[edges["knowledge_layer"].eq("curated_lexicon"), "support_status"] = "no_passage"
edges.loc[edges["knowledge_layer"].eq("name_pattern"), "support_status"] = "name_pattern"
human_true = edges["review_status"].isin(["reviewed_true", "checked_true_human"])
edges["relation_reliability"] = np.where(edges["predicate"].eq("competes with"), "low", "standard")
is_fragment = lambda ids: ids.map(labels_of).astype(str).str.strip().str.lower().isin(GENERIC_NAMES)
edges["name_fragment"] = is_fragment(edges["subject_id"]) | is_fragment(edges["object_id"])
edges["retrieval_eligible"] = ((edges["support_status"].isin(["supported", "name_pattern"])
                                | (edges["knowledge_layer"].eq("article_claim") & human_true))
                               & edges["relation_reliability"].eq("standard") & ~edges["name_fragment"])

# The rebuild is the graph of Part 2: compared edge by edge with the edges it published
published = pd.read_csv(KG_REVISION / "edges_v2.csv", encoding="utf-8-sig")
TRIPLE_COLS = ["subject_id", "predicate", "object_id"]
check = published[TRIPLE_COLS + ["retrieval_eligible"]].merge(
    edges[TRIPLE_COLS + ["retrieval_eligible"]], on=TRIPLE_COLS, suffixes=("_published", "_rebuilt"))
agree = (check["retrieval_eligible_published"] == check["retrieval_eligible_rebuilt"]).mean()
print(f"Rebuilt graph: {len(check)} of the {len(published)} edges of Part 2 matched, same retrieval decision "
      f"for {agree:.1%}")
assert len(check) == len(published) and agree == 1.0, "the rebuilt graph differs from the graph of Part 2"

support_columns = ["support_score", "contradiction_score", "supported", "hypothesis"]
assertions_v2 = (with_key(assertions)
                 .merge(sentence_support.drop_duplicates(["key", "text_unit_id"])[["key", "text_unit_id", *support_columns]],
                        on=["key", "text_unit_id"], how="left", validate="many_to_one")
                 .merge(edges[["key", "retrieval_eligible"]], on="key", how="left", validate="many_to_one"))
assertions_v2["supported"] = assertions_v2["supported"].fillna(False).astype(bool)
graph_relations = assertions_v2[assertions_v2["retrieval_eligible"]
                                & (assertions_v2["supported"] | assertions_v2["knowledge_source"].eq("name_pattern"))]
graph_relations = graph_relations[[c for c in assertions_v2.columns if c != "key"]]
background = edges[edges["in_lexicon"]][TRIPLE].assign(knowledge_source="curated_lexicon")
assert len(graph_relations) == GRAPH_SETTINGS["knowledge_graph"]["relations_used"], "relations used differ from Part 2"
print(f"Relations used for retrieval: {len(graph_relations):,}, the number recorded by Part 2 in graph_rag_settings.json")

if GRAPH_TODO:
    # Entity linking and the fused retrieval of Part 2
    linker = EntityLinker(nodes, {f["label"]: f["rx"] for f in lexicon["families"].values()},
                          GRAPH_SETTINGS["linking"]["exceptions"])
    mentions = node_mentions(pd.read_csv(STAGE1_EXTRACTION / "entity_mentions.csv",
                                         usecols=["entity_id", "doc_id", "sentence_id", "date"]), nodes)
    graph_rag = GraphTextRetriever(linker, graph_relations, mentions, chunk_corpus.records, search, background=background)
    print(f"Graph retriever: {len(graph_relations):,} relations, {mentions['entity_id'].nunique():,} nodes mentioned "
          f"in {mentions['document_id'].nunique():,} articles | fusion {GRAPH_SETTINGS['fusion']}")

Graph answers missing for 0 of the 23 new questions
Entailment check: 7,062 sentence-relation pairs, all served from the cache of Part 2 (no model loaded)
Rebuilt graph: 999 of the 999 edges of Part 2 matched, same retrieval decision for 100.0%
Relations used for retrieval: 4,173, the number recorded by Part 2 in graph_rag_settings.json


/tmp/ipykernel_784/1888461858.py:66: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  assertions_v2["supported"] = assertions_v2["supported"].fillna(False).astype(bool)


In [ ]:
if GRAPH_TODO:
    from anatoolbox.corpus import passages_with_text
    from anatoolbox.enrich.synthesize.synthesize_answer import curate

    def prompt_sources(passages, corpus):
        # The passages synthesize_answer puts in the prompt, chosen exactly as the tool does (as in Part 2)
        sources, _, _ = curate(passages_with_text(passages, corpus, tool_name="synthesize_answer"),
                               max_passages=ANSWER_ARGS["max_passages"], max_chars=ANSWER_ARGS["max_chars_per_passage"],
                               max_per_source=ANSWER_ARGS["max_per_source"])
        return sources

    def graph_inputs(q):
        # What the graph system gives the model: the fused passages and the facts whose evidence chunk is in the prompt
        run = graph_rag.retrieve(q["question"], **RETRIEVAL_ARGS)
        in_prompt = {s["id"] for s in prompt_sources(run["passages"], run["corpus"])}
        return {"input": {"passages": run["passages"], "corpus": run["corpus"]},
                "facts": graph_rag.facts_in_context(run, in_prompt), "route": run["route"],
                "entities": [e["name"] for e in run["entities"]]}

    graph_runs = {q["id"]: graph_inputs(q) for q in GRAPH_TODO}
    routes = pd.Series([r["route"] for r in graph_runs.values()]).value_counts()
    print(f"Retrieval routes of the new questions: {routes.to_dict()} | questions with graph facts in the prompt: "
          f"{sum(bool(r['facts']) for r in graph_runs.values())} of {len(graph_runs)}")
    generate_all("graph", GRAPH_TODO, lambda q: graph_runs[q["id"]])
display(available(SYSTEMS))

# How the graph took part, read from the saved answers of the new questions (printed at every run)
new_graph = [answers[("graph", q["id"])] for q in new_questions if ("graph", q["id"]) in answers]
if new_graph:
    routes = pd.Series([r["route"] for r in new_graph]).value_counts().to_dict()
    print(f"Graph retrieval on the {len(new_graph)} new questions: routes {routes} | "
          f"graph facts in the prompt for {sum(bool(r['result']['facts']) for r in new_graph)} answers")

,factoid,multi-hop,global,total
text-only RAG,213,18,5,236
text + graph RAG,213,18,5,236


Graph retrieval on the 23 new questions: routes {'entity+relation': 18, 'entity': 4, 'text': 1} | graph facts in the prompt for 16 answers


The rebuilt graph is the graph of Part 2: all 999 published edges are matched with the same retrieval decision, and the 4,173 relations used for retrieval are the number recorded in `graph_rag_settings.json`. This matters for the comparison, because a graph system rebuilt with a different support threshold or a different set of edges would be a third system, not the second one. The rebuild costs nothing but reading: the entailment check that decides which relations are trustworthy runs from the 7,062 cached scores of Part 2 and loads no model. The last line above shows how the graph took part in answering the new questions: the route is *entity+relation* when the linker found entities in the question and relations between them in the graph, *entity* when it found entities only, and *text* when it found neither and the hybrid search alone was used. For the 23 new questions the linker found entities in 22 and relations between them in 18; one question went through the text route only, and graph facts reached the prompt for 16 of the 23 answers. The new questions therefore exercise the graph more than the Part 1 questions did, where the graph changed the prompt for 109 of 213.

### 5.2.3 Example: one multi-hop question answered by both systems
Before the scores, one new question shows what the systems produce: the passages given to the model, the graph facts if any, and the answer with its citations, next to the reference answer.

In [ ]:
SYSTEM_NAMES = dict(SYSTEMS)        # extended in Section 5.6 with the enhanced system

def show_answer(system, qid, max_sources=3):
    record = answers.get((system, qid))
    if record is None:
        print(f"{SYSTEM_NAMES[system]}: no answer yet"); return
    result = record["result"]
    sources = "; ".join(f"[{s['label']}] {s['title'][:50]} ({s['date']})" for s in result["sources"][:max_sources])
    facts = "; ".join(f"[{f['label']}] {f['subject']} {f['relation']} {f['object']}" for f in result["facts"]) or "none"
    display(Markdown(f"**{SYSTEM_NAMES[system]}** · sources: {sources}{' …' if len(result['sources']) > max_sources else ''}"
                     f" · graph facts: {facts}\n\n> {result['answer']}"))

example = next((q for q in new_questions if ("text", q["id"]) in answers), EVAL_SET[0])
display(Markdown(f"**{example['id']} ({example['category']}, {example['type']})**: {example['question']}\n\n"
                 f"*Reference answer*: {example['reference_answer']}"))
for system in SYSTEMS:
    show_answer(system, example["id"])

**m02 (multi-hop, relational)**: Which Meta model is Nvidia's Nemotron-70B built on, and which enterprise software companies were expected to be the first users of the Llama Nemotron family announced in January 2025?

*Reference answer*: Nemotron-70B (Llama-3.1-Nemotron-70B-Instruct) is based on Meta's Llama-3.1-70B-Instruct; SAP and ServiceNow were expected to be among the first to use the Llama Nemotron models.

**text-only RAG** · sources: [S1] Nvidia debuts Llama Nemotron open reasoning models (2025-03-18); [S2] Nvidia’s Newest Foundation Model Can Actually Spel (2024-10-23); [S3] Nvidia’s Nemotron model families will advance AI a (2025-01-07) … · graph facts: none

> Nvidia's Nemotron-70B is built on Meta’s open source Llama foundation models, specifically Llama-3.1-70B-Instruct Base [S2]. 

The enterprise software companies expected to be the first users of the Llama Nemotron family, as announced in January 2025, are not explicitly named in the sources. However, the sources indicate that enterprises can use the Llama Nemotron and Cosmos Nemotron models via the Nvidia AI Enterprise software platform, with access to NeMo microservices, NeMo Retriever, and Blueprints for agentic AI [S3][S4]. While the specific enterprise companies are not identified, the platform is designed for enterprise deployment and customization [S3][S4]. Therefore, the sources do not specify which enterprise software companies were expected to be the first users.

**text + graph RAG** · sources: [S1] Nvidia’s Newest Foundation Model Can Actually Spel (2024-10-23); [S2] NVIDIA Announces Nemotron Model Families to Advanc (2025-01-07); [S3] Nvidia just dropped a new AI model that crushes Op (2024-10-17) … · graph facts: [G1] Nemotron-70B member of family Nemotron; [G2] Nemotron developed by NVIDIA

> Nvidia's Nemotron-70B is built on Meta’s open source Llama foundation models, specifically Llama-3.1-70B-Instruct Base [S1]. 

Enterprise software companies expected to be the first users of the Llama Nemotron family, announced in January 2025, include SAP and ServiceNow [S4].

The two systems agree on the first hop, Llama-3.1-70B-Instruct, with the same article cited. On the second, text-only RAG retrieves the January 2025 announcement but a chunk that does not name the first users, and answers that "the sources do not specify which enterprise software companies"; text + graph RAG retrieves a different chunk of the same event and names SAP and ServiceNow. The two graph facts shown, *Nemotron-70B member of family Nemotron* and *Nemotron developed by NVIDIA*, add nothing a reader does not know; what helped was the entity route of the retriever, which pulled the Nemotron articles into the ranking by their entity mentions rather than by the words of the question. This is the pattern of the gains in Section 5.3: the graph changes *which passages* reach the prompt more often than it supplies a fact the passages lack.

## 5.3 Evaluation with a Judge Model
An answer can cite its sources and still misstate them, so the citation checks of Part 2 are completed with a judge model, as recommended in the course skeleton (slide 6.1: evaluation of RAG systems). The judge reads the question, the passages and graph facts that the system saw, the reference answer and the answer, and gives a score from 1 to 5 with a one-sentence rationale on each criterion of the project brief:

| Criterion | Question the judge answers | Applies to |
|---|---|---|
| **faithfulness** | Is every statement supported by the evidence the system was given? | all answers |
| **relevance** | Does the answer address the question directly and completely? | all answers |
| **context grounding** | Does the answer use the relevant passages and graph facts, and cite them where it uses them? | all answers |
| **temporal grounding** | Are dates and chronology correct and stated where they matter? | questions about time (otherwise null) |
| **correctness** | Does the answer agree with the reference answer on the facts that matter? | all answers |

The judge is **GPT-5 mini** (OpenAI), a model from a third family: it is not the RAG model (Qwen3-4B), as the brief requires, and not the model that drafted the questions and reference answers (Claude), so it does not grade its own reference answers. It runs at temperature 0 with a strict JSON output and a low reasoning effort (about 0.3 cents per answer). The tool also lists the statements it found unsupported and flags judgments that contradict themselves (`review_flags`); a sample of the judgments is checked by hand in Section 5.4.

In [ ]:
JUDGE_EFFORT = "low"                                     # reasoning effort of the judge; "high" is used in 3.4.1

def use_judge(effort=JUDGE_EFFORT):
    # Point the anatoolbox tools at the judge model (OpenAI API, key from the Colab secrets)
    configure_llm(base_url="https://api.openai.com/v1", api_key=OPENAI_API_KEY, model=JUDGE_MODEL,
                  models={"evaluation": JUDGE_MODEL},
                  specs={JUDGE_MODEL: ModelSpec(temperature=False, reasoning_effort=effort)}, timeout=120, max_retries=2)

def judge_endpoint():
    # The model the API actually serves, which is not necessarily the one asked for: the server names it in its reply
    from openai import OpenAI
    reply = OpenAI(api_key=OPENAI_API_KEY).chat.completions.create(
        model=JUDGE_MODEL, messages=[{"role": "user", "content": "Reply with the single word OK."}],
        max_completion_tokens=2000)
    return reply.model

CRITERIA = ["faithfulness", "relevance", "context_grounding", "temporal_grounding", "correctness"]
JUDGMENTS = OUT_DIR / "judgments.jsonl"
judgments = {}
if JUDGMENTS.exists():
    for line in JUDGMENTS.read_text(encoding="utf-8").splitlines():
        record = json.loads(line)
        # a judgment belongs to the answer it scored: a reworded question has no answer here and keeps no judgment
        if record["judge"] == JUDGE_MODEL and (record["system"], record["id"]) in answers:
            judgments[(record["system"], record["id"])] = record

def judge_one(system, qid, effort=JUDGE_EFFORT):
    # One judgment, in the record format saved on Drive
    verdict = judge.run({"answer": answers[(system, qid)]["result"],
                         "reference_answer": BY_ID[qid]["reference_answer"]}, context=ctx)
    return {"system": system, "id": qid, "judge": JUDGE_MODEL, "effort": effort,
            **{k: verdict[k] for k in ("scores", "rationales", "unsupported_claims", "review_flags")}}

def judge_all(systems):
    # Judge every answer of `systems` that has no judgment yet; one JSON line per judgment on Drive
    missing = [key for key in answers if key[0] in systems and key not in judgments]
    if not missing:
        return
    if not OPENAI_API_KEY:
        print(f"{len(missing)} answers to judge, but no OPENAI_API_KEY: add it to the Colab secrets and run again")
        return
    use_judge()
    started, failed = time.perf_counter(), []
    with JUDGMENTS.open("a", encoding="utf-8") as handle:
        for n, (system, qid) in enumerate(missing, start=1):
            try:
                record = judge_one(system, qid)
            except Exception as error:              # a timeout or rate limit costs one answer, not the run
                failed.append(f"{system}/{qid}: {type(error).__name__}")
                continue
            judgments[(system, qid)] = record
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            handle.flush()
            if n % 50 == 0:
                print(f"{n}/{len(missing)} judged, {time.perf_counter() - started:.0f} s")
    print(f"{len(missing) - len(failed)} answers judged in {time.perf_counter() - started:.0f} s"
          + (f" | failed: {failed[:5]}" if failed else ""))

if OPENAI_API_KEY:                                        # the model the API serves, checked at every run
    use_judge()
    print(f"Judge endpoint: {JUDGE_MODEL} requested, {judge_endpoint()} served, reasoning effort {JUDGE_EFFORT}")
judge_all(SYSTEMS)
print(f"Judgments: {len(judgments)} for {len(answers)} answers")
assert judgments, ("No judgment yet, so the sections below have nothing to measure. The answers above are saved on "
                   "Drive and will not be generated again: add OPENAI_API_KEY to the Colab secrets (key icon on the "
                   "left, notebook access on) and run the notebook again.")

Judgments: 708 for 708 answers


In [ ]:
NO_ANSWER = re.compile(r"(?:do(?:es)? not|don't|doesn't) (?:contain|provide|mention|include|specify|say)|"
                       r"not enough information|no information|cannot be (?:determined|answered)", re.I)   # as in Part 2

def score_table(systems):
    # One row per judged answer: scores, flags, citation checks and whether the source article was in the prompt
    rows = []
    for (system, qid), j in judgments.items():
        if system not in systems:
            continue
        q, result = BY_ID[qid], answers[(system, qid)]["result"]
        prompt_articles = {s.get("source_id") for s in result["sources"]} | {a for f in result["facts"] for a in f.get("source_ids") or []}
        rows.append({"system": system, "id": qid, "category": q["category"], "type": q["type"], **j["scores"],
                     "unsupported_claims": len(j["unsupported_claims"]), "flagged": bool(j["review_flags"]),
                     "citation_coverage": result["citation_coverage"] or 0.0, "graph_facts": len(result["facts"]),
                     "no_answer": bool(NO_ANSWER.search(result["answer"])),
                     "source_in_prompt": bool(prompt_articles & set(q["source_ids"])),
                     "all_sources_in_prompt": set(q["source_ids"]) <= prompt_articles})
    return pd.DataFrame(rows)

scores = score_table(SYSTEMS)
scores.to_csv(OUT_DIR / "judge_scores.csv", index=False)
label = lambda df: df["system"].map(SYSTEMS)

print("Mean score (1–5) per criterion; temporal grounding only where the judge applied it")
overall = scores.groupby(label(scores)).agg(answers=("id", "size"), **{c: (c, "mean") for c in CRITERIA},
                                            unsupported_claims=("unsupported_claims", "mean"),
                                            no_answer=("no_answer", "mean"), flagged=("flagged", "mean"))
display(overall.round(2))
print(f"Temporal grounding was scored for {scores['temporal_grounding'].notna().sum()} of {len(scores)} answers")

Mean score (1–5) per criterion; temporal grounding only where the judge applied it


,answers,faithfulness,relevance,context_grounding,temporal_grounding,correctness,unsupported_claims,no_answer,flagged
system,,,,,,,,,
text + graph RAG,236,4.46,4.82,4.51,4.58,4.09,0.53,0.16,0.01
text-only RAG,236,4.39,4.75,4.47,4.50,4.04,0.58,0.22,0.00


Temporal grounding was scored for 150 of 472 answers


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [1.4, 1]})
long = scores.melt(id_vars=["system"], value_vars=CRITERIA, var_name="criterion", value_name="score").dropna()
long["system"], long["criterion"] = long["system"].map(SYSTEMS), long["criterion"].str.replace("_", "\n")
sns.barplot(data=long, x="criterion", y="score", hue="system", errorbar=("ci", 95), ax=axes[0], palette=["#4C72B0", "#DD8452"])
axes[0].set(ylim=(1, 5), xlabel="", ylabel="mean score (1–5, 95% CI)", title="Judge scores of the two systems")
axes[0].legend(title="")
low = (scores.assign(low=scores["faithfulness"].le(2)).groupby([label(scores), "category"])["low"].mean().unstack()
       .reindex(columns=CATEGORIES) * 100)
low.fillna(0).plot.bar(ax=axes[1], color=["#4C72B0", "#55A868", "#C44E52"], rot=0)
axes[1].set(ylabel="% of answers", xlabel="", title="Answers with faithfulness ≤ 2", ylim=(0, low.max().max() * 1.35))
axes[1].legend(title="category", loc="upper center", ncol=3, frameon=False)
for container in axes[1].containers:                       # a bar at 0 is a result (no such answer), so it is labelled
    axes[1].bar_label(container, fmt="%.0f%%", fontsize=9)
save_fig(fig, "judge_scores_overall")

Both systems score high on the criteria about the answer's form, relevance (4.75–4.82) and faithfulness (4.39–4.46), and lower on correctness (4.04–4.09): an answer is more often loyal to its passages than right, which is what happens when the passages are the wrong ones. Text + graph RAG is ahead on every criterion by 0.04 to 0.08, and the two systems' answers carry the same number of unsupported claims (0.53 and 0.58 per answer; 26% and 27% of answers have at least one). The clearest difference is in the refusals: text-only RAG answers that the sources do not contain the information for 22% of the questions, text + graph RAG for 16%, which is where its higher relevance comes from. Temporal grounding was scored for 150 of 472 answers, those whose question or answer involve a date, at 4.50–4.58: the publication dates in the prompt are used, but not always.

By category, the picture changes. On the factoid questions the graph is 0.05 ahead in correctness (4.16 against 4.11). On the multi-hop questions the two systems are at the same correctness, 3.39, three quarters of a point below the factoid questions, and only faithfulness separates them (4.11 against 4.00). On the 5 global questions the graph is 0.20 ahead (3.60 against 3.40) while the text-only system has the higher context grounding (4.60 against 4.40); with five questions, neither figure means much. By type, the graph helps most on temporal (+0.27) and comparative (+0.13) questions and is behind on analytical ones (4.17 against 4.44, −0.27): a question that asks for a judgement is not answered better by adding two facts to the prompt, and the extra facts may distract from the passage that holds the argument.

### 5.3.1 By question category and type
The three categories of the Q&A session are the main comparison: the graph should matter most for multi-hop questions, whose answer connects entities across articles, and global questions should be the hardest for both systems, since the prompt holds at most 6 passages.

In [ ]:
by_cat = (scores.groupby(["category", label(scores)])[["correctness", "faithfulness", "relevance", "context_grounding"]]
          .mean().reindex(CATEGORIES, level=0))
by_cat["answers"] = scores.groupby(["category", label(scores)]).size()
display(by_cat.round(2))

by_type = scores.groupby(["type", label(scores)])[["correctness", "faithfulness", "temporal_grounding"]].mean().reindex(TYPES, level=0)
display(by_type.round(2))

diff = (scores.pivot_table(index="category", columns="system", values=CRITERIA, aggfunc="mean"))
delta = pd.DataFrame({c: diff[(c, "graph")] - diff[(c, "text")] for c in CRITERIA}).reindex(CATEGORIES)
fig, ax = plt.subplots(figsize=(9, 3.2))
sns.heatmap(delta, annot=True, fmt="+.2f", center=0, cmap="RdBu", vmin=-1, vmax=1, cbar_kws={"label": "graph − text"}, ax=ax)
ax.set(title="Mean score difference, text + graph minus text-only, by category", xlabel="", ylabel="")
save_fig(fig, "judge_scores_by_category")

correctness  faithfulness  relevance  \
category  system                                                   
factoid   text + graph RAG         4.16          4.48       4.84   
          text-only RAG            4.11          4.43       4.77   
multi-hop text + graph RAG         3.39          4.11       4.67   
          text-only RAG            3.39          4.00       4.56   
global    text + graph RAG         3.60          4.60       4.60   
          text-only RAG            3.40          4.40       4.40   

                            context_grounding  answers  
category  system                                        
factoid   text + graph RAG               4.54      213  
          text-only RAG                  4.49      213  
multi-hop text + graph RAG               4.28       18  
          text-only RAG                  4.22       18  
global    text + graph RAG               4.40        5  
          text-only RAG                  4.60        5

correctness  faithfulness  temporal_grounding
type        system                                                         
factual     text + graph RAG         3.86          4.34                4.45
            text-only RAG            3.90          4.32                4.77
temporal    text + graph RAG         4.07          4.39                4.44
            text-only RAG            3.80          4.18                4.42
relational  text + graph RAG         4.17          4.55                5.00
            text-only RAG            4.04          4.40                4.50
analytical  text + graph RAG         4.17          4.36                4.71
            text-only RAG            4.44          4.61                4.33
comparative text + graph RAG         4.27          4.66                5.00
            text-only RAG            4.14          4.52                4.67

### 5.3.2 Is the difference between the systems reliable?
Both systems answered the same questions, so they are compared **question by question**: for each question, the score of the graph system minus the score of the text-only system. The table gives the mean difference with a 95% bootstrap interval (1,000 resamples of the questions, as in Stage 2), the share of questions where each system is better, and `p`, the share of resamples where the graph system is not better (one-sided). Questions whose prompt the graph did not change get the same answer from both systems (Part 2), so the comparison is also shown on the questions where the prompt changed.

In [ ]:
def paired(criterion, mask=None, n=1000):
    wide = scores.pivot(index="id", columns="system", values=criterion).dropna()
    if mask is not None:
        wide = wide[wide.index.isin(mask)]
    if wide.empty:
        return None
    d = (wide["graph"] - wide["text"]).to_numpy()
    idx = np.random.default_rng(SEED).integers(0, len(d), (n, len(d)))
    boots = d[idx].mean(axis=1)
    return {"questions": len(d), "mean difference": d.mean(), "low": np.percentile(boots, 2.5), "high": np.percentile(boots, 97.5),
            "graph better": (d > 0).mean(), "text better": (d < 0).mean(), "same": (d == 0).mean(), "p": (boots <= 0).mean()}

changed = {qid for qid in BY_ID if ("text", qid) in answers and ("graph", qid) in answers
           and ([s["id"] for s in answers[("text", qid)]["result"]["sources"]] != [s["id"] for s in answers[("graph", qid)]["result"]["sources"]]
                or answers[("graph", qid)]["result"]["facts"])}
rows = []
for criterion in ["correctness", "faithfulness", "relevance", "context_grounding"]:
    for subset, mask in [("all questions", None), ("prompt changed by the graph", changed)]:
        for category, cat_mask in [("all", None)] + [(c, {q["id"] for q in EVAL_SET if q["category"] == c}) for c in CATEGORIES]:
            ids = None if mask is None and cat_mask is None else (mask or set(BY_ID)) & (cat_mask or set(BY_ID))
            result = paired(criterion, ids)
            if result:
                rows.append({"criterion": criterion, "subset": subset, "category": category, **result})
paired_table = pd.DataFrame(rows)
paired_table.to_csv(OUT_DIR / "judge_paired.csv", index=False)
display(paired_table[paired_table["category"] == "all"].set_index(["criterion", "subset"]).round(3))
print("Correctness, by category:")
display(paired_table[(paired_table["criterion"] == "correctness") & (paired_table["category"] != "all")]
        .set_index(["subset", "category"]).drop(columns="criterion").round(3))

category  questions  \
criterion         subset                                            
correctness       all questions                    all        236   
                  prompt changed by the graph      all        128   
faithfulness      all questions                    all        236   
                  prompt changed by the graph      all        128   
relevance         all questions                    all        236   
                  prompt changed by the graph      all        128   
context_grounding all questions                    all        236   
                  prompt changed by the graph      all        128   

                                               mean difference    low   high  \
criterion         subset                                                       
correctness       all questions                          0.055 -0.072  0.178   
                  prompt changed by the graph            0.039 -0.180  0.274   
faithfulness      all questions                          0.064 -0.042  0.174   
                  prompt changed by the graph            0.070 -0.109  0.274   
relevance         all questions                          0.076  0.004  0.148   
                  prompt changed by the graph            0.102  0.000  0.227   
context_grounding all questions                          0.042 -0.059  0.153   
                  prompt changed by the graph            0.047 -0.117  0.211   

                                               graph better  text better  \
criterion         subset                                                   
correctness       all questions                       0.153        0.114   
                  prompt changed by the graph         0.180        0.156   
faithfulness      all questions                       0.127        0.114   
                  prompt changed by the graph         0.172        0.180   
relevance         all questions                       0.089        0.051   
                  prompt changed by the graph         0.109        0.062   
context_grounding all questions                       0.161        0.140   
                  prompt changed by the graph         0.203        0.188   

                                                same      p  
criterion         subset                                     
correctness       all questions                0.733  0.217  
                  prompt changed by the graph  0.664  0.383  
faithfulness      all questions                0.758  0.141  
                  prompt changed by the graph  0.648  0.249  
relevance         all questions                0.860  0.019  
                  prompt changed by the graph  0.828  0.035  
context_grounding all questions                0.699  0.217  
                  prompt changed by the graph  0.609  0.288

Correctness, by category:


questions  mean difference    low  \
subset                      category                                       
all questions               factoid          213            0.056 -0.061   
                            multi-hop         18            0.000 -0.833   
                            global             5            0.200 -0.400   
prompt changed by the graph factoid          109            0.037 -0.174   
                            multi-hop         16            0.000 -0.938   
                            global             3            0.333  0.000   

                                        high  graph better  text better  \
subset                      category                                      
all questions               factoid    0.183         0.131        0.103   
                            multi-hop  0.778         0.333        0.222   
                            global     0.800         0.400        0.200   
prompt changed by the graph factoid    0.266         0.156        0.156   
                            multi-hop  0.875         0.312        0.188   
                            global     1.000         0.333        0.000   

                                        same      p  
subset                      category                 
all questions               factoid    0.765  0.186  
                            multi-hop  0.444  0.521  
                            global     0.400  0.347  
prompt changed by the graph factoid    0.688  0.380  
                            multi-hop  0.500  0.495  
                            global     0.667  0.298

The paired comparison confirms the reading above with its uncertainty. Over all 236 questions, the graph is better on relevance by +0.08 with a 95% interval of [+0.00, +0.15] (p = 0.02): the only difference whose interval excludes zero. Correctness (+0.06, [−0.07, +0.18], p = 0.22), faithfulness (+0.06, p = 0.14) and context grounding (+0.04, p = 0.22) all lean the same way and could all be zero. For 73% of the questions the two systems have the same correctness score; the graph is better on 15% and worse on 11%. Restricting the comparison to the 128 questions whose prompt the graph changed does not sharpen it (correctness +0.04, [−0.18, +0.27]): the changed prompts help about as often as they hurt.

By category, the interval for the multi-hop questions is [−0.83, +0.78] around a mean difference of exactly 0.00, with the graph better on 6 questions and worse on 4 out of 18, and the interval for the 5 global questions is [−0.40, +0.80]. With 18 multi-hop and 5 global questions, the per-category intervals are wide: they show the direction of the differences, and the 213 factoid questions give the reliable comparison.

### 5.3.3 What the judge found unsupported
The judge lists the statements an answer makes without support in its evidence. The most frequent kinds, and the answers with the lowest faithfulness, show what the systems get wrong: a date taken from the wrong article, a number attributed to the wrong model, or knowledge added from the model's memory.

In [ ]:
def judged(system, qid):
    j, q = judgments[(system, qid)], BY_ID[qid]
    return {"system": SYSTEMS[system], "id": qid, "category": q["category"], "question": q["question"],
            **{c: j["scores"][c] for c in ("faithfulness", "correctness")},
            "unsupported claims": " | ".join(j["unsupported_claims"])[:300],
            "rationale (faithfulness)": j["rationales"]["faithfulness"]}

worst = scores.sort_values(["faithfulness", "correctness"]).head(8)
display(pd.DataFrame([judged(r.system, r.id) for r in worst.itertuples()]).set_index(["system", "id"]))
print(f"Answers with at least one unsupported claim: "
      + ", ".join(f"{SYSTEMS[s]} {v:.0%}" for s, v in scores.assign(any=scores['unsupported_claims'].gt(0)).groupby('system')['any'].mean().items()))
print(f"Judgments flagged for review: {scores['flagged'].sum()} of {len(scores)} ("
      + "; ".join(f"{f}: {n}" for f, n in pd.Series([flag.split(':')[0] for j in judgments.values() for flag in j['review_flags']]).value_counts().items()) + ")")

category  \
system        id              
text-only RAG q003  factoid   
              q096  factoid   
              q097  factoid   
              q101  factoid   
              q138  factoid   
              q146  factoid   
              q248  factoid   
              q295  factoid   

                                                                      question  \
system        id                                                                 
text-only RAG q003                           When did Google launch DataGemma?   
              q096                            Can Gemma 3 run on a single GPU?   
              q097     How much more compute does DeepSeek R1 need than Llama?   
              q101                 Can I run Llama or Mistral on Intel Gaudi2?   
              q138  How much data did Skywork keep to train Skywork-Reward-V2?   
              q146                          Does Grok 2.5 use a shared expert?   
              q248                          How many models are in AuditBench?   
              q295                         When did Meta release Muse Glimmer?   

                    faithfulness  correctness  \
system        id                                
text-only RAG q003             1            1   
              q096             1            1   
              q097             1            1   
              q101             1            1   
              q138             1            1   
              q146             1            1   
              q248             1            1   
              q295             1            1   

                                                                                                                         unsupported claims  \
system        id                                                                                                                              
text-only RAG q003  The provided sources do not specify when Google launched DataGemma. | none state the exact date of the launch. | The...   
              q096  The provided sources do not contain information about whether Gemma 3 can run on a single GPU. | While [S5] states t...   
              q097  The provided sources do not contain information about how much more compute DeepSeek R1 needs compared to Llama. | t...   
              q101  No, you cannot run Llama or Mistral on Intel Gaudi2. | the sources do not specify any direct compatibility or perfor...   
              q138  The provided sources do not specify how much data Skywork kept to train Skywork-Reward-V2. | They mention screening ...   
              q146  The provided sources do not state whether Grok 2.5 uses a shared expert. | While [S3] mentions that Grok 2.5 uses a ...   
              q248  The number of models in AuditBench is four. | The reference to 'four training configurations' and the context of eva...   
              q295  The sources do not mention the release of "Muse Glimmer." | Therefore, based on the provided sources, the release da...   

                                                                                                                   rationale (faithfulness)  
system        id                                                                                                                             
text-only RAG q003  The answer asserts that none of the sources specify the launch date and that the date cannot be determined, but sour...  
              q096  The answer contradicts and ignores a direct statement in S5 that explicitly says Gemma 3 is designed to run on a sin...  
              q097  Most statements in the answer contradict or ignore explicit information in the provided sources (notably [S4] which ...  
              q101  The answer starts by denying you can run Llama or Mistral on Gaudi2, which directly contradicts S1 that explicitly l...  
              q138  The answer claims the sources do not specify how much data was kept, but S1 and S2 explicitly

Answers with at least one unsupported claim: text + graph RAG 26%, text-only RAG 27%
Judgments flagged for review: 2 of 472 (faithfulness: 3)


The worst-judged answers are of one kind: the model contradicts a passage it was given. All eight answers with faithfulness 1 and correctness 1 are text-only answers to factoid questions, and in each the judge's rationale points to the passage in the prompt that states the fact. Six of them are refusals, the model saying the sources do not contain the information when they do (the launch date of DataGemma, whether Gemma 3 runs on one GPU, the release of Muse Glimmer); the other two assert the opposite of the source (that Llama cannot run on Intel Gaudi2, that AuditBench has four models). The judge lists the refusal or the wrong statement itself as the unsupported claim. This is the generation failure of Section 5.5, as opposed to the retrieval failure: the right chunk is there and the 4B model does not find the fact in it, usually when the fact is a number or a date in a long passage. The share of answers with at least one unsupported claim is the same for both systems (26% and 27%), so the graph facts in the prompt neither add unsupported statements nor remove them. The judge flagged 2 of 472 judgments for review, both on faithfulness, where its own rationale and score disagreed.

## 5.4 Does the Judge Agree with a Person?
A judge is a model too. Before the scores above are used to compare systems, a sample of them is checked by hand: 15 questions (9 factoid, 4 multi-hop, 2 global, drawn with a fixed seed) with the answers of both systems, 30 answers in all. The sheet shows the question, the reference answer, the evidence the system saw and the answer, **without the system's name and without the judge's scores**, and the reviewer gives the same 1–5 scores for faithfulness, relevance and correctness, following the criteria of Section 5.3. The agreement with the judge is then measured: exact agreement, agreement within one point, and the Spearman correlation per criterion.

In [ ]:
CHECK_SHEET, CHECK_KEY = OUT_DIR / "judge_check_sheet.xlsx", OUT_DIR / "judge_check_key.csv"
HUMAN_CRITERIA = ["faithfulness", "relevance", "correctness"]

def reviewed_sheet():
    # The reviewed sheet: judge_check_reviewed.xlsx, or the blind sheet itself if the scores were filled in place
    for path in (part3_file("judge_check_reviewed.xlsx"), CHECK_SHEET):
        if path.exists():
            try:
                filled = pd.read_excel(path, sheet_name="check")[HUMAN_CRITERIA].notna().all(axis=1).sum()
            except Exception:
                continue
            if filled:
                return path, int(filled)
    return None, 0

def evidence_text(result, limit=1500):
    blocks = []
    for s in result["sources"]:
        text = CHUNK[s["id"]]["text"] if s["id"] in CHUNK else ""
        blocks.append(f"[{s['label']}] {s['title']} · {s['date']}\n{text[:limit]}{' …' if len(text) > limit else ''}")
    for f in result["facts"]:
        blocks.append(f"[{f['label']}] graph fact: {f['subject']} {f['relation']} {f['object']} ({f.get('date', '')})")
    return "\n\n".join(blocks)

# The sample is drawn again if it was drawn from another test set (a question it holds is no longer in it)
if CHECK_KEY.exists() and not set(pd.read_csv(CHECK_KEY)["question_id"]) <= set(BY_ID):
    print(f"{CHECK_SHEET.name} was built from a different test set: it is built again")
    CHECK_KEY.unlink()

if not CHECK_KEY.exists():
    rng = np.random.default_rng(SEED)
    sample = []
    for category, n in [("factoid", 9), ("multi-hop", 4), ("global", 2)]:
        pool = [q for q in EVAL_SET if q["category"] == category and ("text", q["id"]) in judgments]
        sample += list(rng.choice(pool, size=min(n, len(pool)), replace=False)) if pool else []
    rows = [(q, system) for q in sample for system in SYSTEMS if (system, q["id"]) in judgments]
    rng.shuffle(rows)
    sheet = pd.DataFrame([{"row": n, "question_id": q["id"], "category": q["category"], "question": q["question"],
                           "reference_answer": q["reference_answer"],
                           "evidence": evidence_text(answers[(system, q["id"])]["result"]),
                           "answer": answers[(system, q["id"])]["result"]["answer"],
                           **{c: "" for c in HUMAN_CRITERIA}, "note": "", "reviewer": ""}
                          for n, (q, system) in enumerate(rows, start=1)])
    pd.DataFrame({"row": sheet["row"], "question_id": sheet["question_id"], "system": [s for _, s in rows]}).to_csv(CHECK_KEY, index=False)
    with pd.ExcelWriter(CHECK_SHEET) as writer:
        pd.DataFrame({"HOW TO SCORE": [
            "For every row: read the question, the reference answer, the evidence (the passages and graph facts the system saw) and the answer.",
            "Give an integer score from 1 to 5 for each criterion, judging only by the criterion:",
            "faithfulness: every statement in the answer is supported by the evidence (5: all supported; 3: some unsupported; 1: most unsupported or contradicted).",
            "relevance: the answer addresses the question directly and completely (5: fully; 3: partly or drifts; 1: does not answer).",
            "correctness: the answer agrees with the reference answer on the facts that matter (5: all key facts; 3: misses or blurs some; 1: contradicts or misses).",
            "An answer that says the sources do not contain the information is faithful (5) but not correct if the reference answer exists in the corpus.",
            "Do not look at the notebook's judge scores before scoring; the system that wrote each answer is hidden on purpose."]}).to_excel(writer, sheet_name="INSTRUCTIONS", index=False)
        sheet.to_excel(writer, sheet_name="check", index=False)
    print(f"Check sheet written: {CHECK_SHEET.name} ({len(sheet)} answers)")

key = pd.read_csv(CHECK_KEY)
CHECK_REVIEWED, n_filled = reviewed_sheet()
if CHECK_REVIEWED is not None:
    print(f"Reviewed sheet: {CHECK_REVIEWED} ({n_filled} answers scored)")
    human = pd.read_excel(CHECK_REVIEWED, sheet_name="check").merge(key, on=["row", "question_id"])
    human = human[human[HUMAN_CRITERIA].notna().all(axis=1)]
    human[HUMAN_CRITERIA] = human[HUMAN_CRITERIA].astype(int)
    rows = []
    for c in HUMAN_CRITERIA:
        pairs = human.apply(lambda r: (r[c], judgments[(r["system"], r["question_id"])]["scores"][c]), axis=1)
        h, m = np.array([p[0] for p in pairs]), np.array([p[1] for p in pairs], dtype=float)
        ok = ~np.isnan(m)
        rows.append({"criterion": c, "answers": int(ok.sum()), "human mean": h[ok].mean(), "judge mean": m[ok].mean(),
                     "exact agreement": (h[ok] == m[ok]).mean(), "within 1 point": (np.abs(h[ok] - m[ok]) <= 1).mean(),
                     "spearman": pd.Series(h[ok]).corr(pd.Series(m[ok]), method="spearman"),
                     "judge higher": (m[ok] > h[ok]).mean(), "human higher": (h[ok] > m[ok]).mean()})
    agreement = pd.DataFrame(rows).set_index("criterion")
    agreement.to_csv(OUT_DIR / "judge_agreement.csv")
    display(agreement.round(2))
    human["system"] = human["system"].map(SYSTEMS)
    display(human.groupby("system")[HUMAN_CRITERIA].mean().round(2).rename(columns=lambda c: f"human {c}"))
else:
    print(f"Reviewed sheet not found: fill in {CHECK_SHEET.name} (in {OUT_DIR}) and save it there, as judge_check_reviewed.xlsx or in place")

Reviewed sheet: /content/drive/MyDrive/HSLU/CLT/extraction_outputs/stage3/wp3/judge_check_sheet.xlsx (30 answers scored)


,answers,human mean,judge mean,exact agreement,within 1 point,spearman,judge higher,human higher
criterion,,,,,,,,
faithfulness,30,4.5,4.63,0.83,0.97,0.82,0.13,0.03
relevance,30,4.4,4.80,0.50,0.97,0.48,0.43,0.07
correctness,30,4.0,4.43,0.60,0.90,0.79,0.37,0.03


,human faithfulness,human relevance,human correctness
system,,,
text + graph RAG,4.47,4.4,4.0
text-only RAG,4.53,4.4,4.0


### 5.4.1 Does the Judge Agree with a More Careful Version of Itself?
The judge of Section 5.3 runs with its reasoning effort set to **low**, which is what makes 472 judgments affordable in time and in cost. Whether that setting is enough is a question about the measurement, not about the systems, so it is tested: the same answers as the sheet above are judged a second time with the effort set to **high**, and the two settings are compared as the reviewer and the judge are compared.

Two outcomes are informative. If the scores move little, the low setting is reading the answers rather than guessing, and the cheaper setting is justified. If they move but move *in the same direction for both systems*, the comparison of Section 5.3.2 is still sound, because it is paired: a bias shared by both systems cancels in the difference. Only a shift that favours one system would change the conclusions, and that is what the last line below measures.

In [ ]:
JUDGMENTS_HIGH = OUT_DIR / "judgments_high.jsonl"
high = {}
if JUDGMENTS_HIGH.exists():
    for line in JUDGMENTS_HIGH.read_text(encoding="utf-8").splitlines():
        record = json.loads(line)
        if (record["system"], record["id"]) in judgments:
            high[(record["system"], record["id"])] = record

sample_keys = [(r["system"], r["question_id"]) for _, r in key.iterrows() if (r["system"], r["question_id"]) in judgments]
todo = [k for k in sample_keys if k not in high]
if todo and OPENAI_API_KEY:
    use_judge(effort="high")
    started = time.perf_counter()
    with JUDGMENTS_HIGH.open("a", encoding="utf-8") as handle:
        for system, qid in todo:
            record = judge_one(system, qid, effort="high")
            high[(system, qid)] = record
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            handle.flush()
    use_judge()                                           # back to the setting of Section 5.3
    print(f"{len(todo)} answers judged again at high effort in {time.perf_counter() - started:.0f} s")

if high:
    pairs = pd.DataFrame([{"system": s, "id": q, "criterion": c,
                           "low": judgments[(s, q)]["scores"][c], "high": high[(s, q)]["scores"][c]}
                          for (s, q) in high for c in CRITERIA
                          if judgments[(s, q)]["scores"].get(c) is not None and high[(s, q)]["scores"].get(c) is not None])
    effort = pairs.groupby("criterion").apply(lambda d: pd.Series({
        "answers": len(d), "low effort": d["low"].mean(), "high effort": d["high"].mean(),
        "mean difference": (d["high"] - d["low"]).mean(), "exact agreement": (d["high"] == d["low"]).mean(),
        "within 1 point": ((d["high"] - d["low"]).abs() <= 1).mean(),
        "spearman": d["low"].corr(d["high"], method="spearman")}), include_groups=False).reindex(CRITERIA).dropna(how="all")
    effort.to_csv(OUT_DIR / "judge_effort.csv")
    display(effort.round(2))
    # Does the higher effort move the two systems differently? That, and only that, would change Section 5.3.2
    shift = pairs.assign(d=pairs["high"] - pairs["low"]).groupby([pairs["system"].map(SYSTEMS), "criterion"])["d"].mean().unstack()
    display(shift.reindex(columns=CRITERIA).dropna(axis=1, how="all").round(2))
    per_system = pairs.groupby(["criterion", "system"]).size().unstack()
    enough = per_system.index[(per_system >= 10).all(axis=1)]                # a shift on fewer answers is noise
    gaps = shift.diff().iloc[-1].abs().reindex(enough).dropna()
    print("Difference between the two systems' shifts, per criterion (10 or more answers per system):",
          ", ".join(f"{c} {g:.2f}" for c, g in gaps.items()),
          "| compare with the system differences of Section 5.3.2, 0.04 to 0.08")

,answers,low effort,high effort,mean difference,exact agreement,within 1 point,spearman
criterion,,,,,,,
faithfulness,30.0,4.63,4.80,0.17,0.70,0.93,0.35
relevance,30.0,4.80,4.73,-0.07,0.93,1.00,1.00
context_grounding,30.0,4.77,4.87,0.10,0.77,0.93,0.15
temporal_grounding,10.0,4.80,4.50,-0.30,0.80,0.90,0.57
correctness,30.0,4.43,4.23,-0.20,0.70,0.83,0.67


criterion,faithfulness,relevance,context_grounding,temporal_grounding,correctness
system,,,,,
text + graph RAG,-0.07,-0.07,-0.13,-0.6,-0.2
text-only RAG,0.40,-0.07,0.33,0.0,-0.2


Difference between the two systems' shifts, per criterion (10 or more answers per system): context_grounding 0.47, correctness 0.00, faithfulness 0.47, relevance 0.00 | compare with the system differences of Section 5.3.2, 0.04 to 0.08


The high-effort judge agrees with the low-effort one within one point on 83–100% of the answers and exactly on 70–93%, and moves the means by 0.07 to 0.20: it is slightly stricter on correctness (4.43 → 4.23) and slightly more lenient on faithfulness (4.63 → 4.80). The Spearman correlations are high where the scores vary (relevance 1.00, correctness 0.67) and low where they hardly do (context grounding 0.15: nearly every answer scores 5 under both settings, so the few that differ decide the coefficient). Five of the 30 correctness scores moved by two points or more, which is the size of disagreement the human check of Section 5.4 measures on the same answers.

The test that matters for the conclusions is the last line: on **correctness** and **relevance**, the two criteria the comparison of Section 5.3.2 rests on, the higher effort shifts both systems by the same amount (−0.20 and −0.20, −0.07 and −0.07), so the differences between them would be unchanged by the more careful judge. On faithfulness and context grounding the shifts differ: the high-effort judge raises the text-only answers by 0.33–0.40 and leaves the graph answers where they were, on 15 answers per system. That is enough to reverse the sign of the two smallest system differences of Section 5.3.2 (+0.06 and +0.04, neither significant), which should therefore be read as *no difference*, not as a small advantage of the graph. The temporal shift (−0.6 for the graph system) rests on five answers and is not interpreted.

The cost settles the choice of setting: the high-effort judgments took 50 s per answer against 7 s at low effort, so judging the 708 answers of this part would take about ten hours instead of one and a half. The low setting is adequate for a paired comparison, and the sample shows where it is not: for the absolute level of faithfulness, which the judge reads differently when it reads longer.

The reviewer and the judge agree within one point on 90–97% of the 30 answers and exactly on 83% for faithfulness, 60% for correctness and 50% for relevance. The rank correlations are 0.82 for faithfulness and 0.79 for correctness, 0.48 for relevance; the judge gives relevance 5 to almost every answer that addresses the question, the reviewer distinguishes a complete answer from a partial one. The disagreements have one direction: the judge is more lenient. It scores higher than the reviewer on 13% of the answers for faithfulness, 37% for correctness and 43% for relevance, and lower on 3–7%, so the means of Section 5.3 are about 0.1 (faithfulness) to 0.4 points (relevance, correctness) above what a person gives the same answers. The absolute levels in this part are therefore optimistic by that much; the ranking of the answers, which the paired comparisons use, is the same for the two criteria that matter most.

The reviewer, who did not know which system had written each answer, gave the two systems the same scores: correctness 4.0 and 4.0, relevance 4.4 and 4.4, faithfulness 4.47 and 4.53. On this sample a person sees no difference between text-only and text + graph RAG, which is consistent with the paired result of Section 5.3.2 on all 236 questions, where the differences are within 0.08 and only one of them is distinguishable from zero. Taken with Section 5.4.1, the picture of the judge is that of a lenient but consistent reader: lenient against a person, stable against a more careful version of itself on the criteria that decide the comparison.

## 5.5 Retrieval Diagnosis: Where Do the Answers Fail?
An answer can only be right if the right article reaches the prompt. Part 2 measured retrieval on the Part 1 questions; here the measure is repeated on the full test set with the text-only search of Part 1, per category: hit@k (a source article is among the first k articles) and, since the new questions have several source articles, recall@10 (the share of the source articles among the first 10). The judge's correctness score is then split by whether the source article was in the prompt, which separates retrieval failures from generation failures.

In [ ]:
RETRIEVAL_CACHE = OUT_DIR / "retrieval"
RETRIEVAL_CACHE.mkdir(exist_ok=True)

def evaluate_retrieval(name, retrieve_fn, questions=EVAL_SET):
    # Retrieval metrics of `retrieve_fn(q)` over `questions`, per question and in summary.
    # Saved on Drive and read back when it holds exactly these questions, so re-runs do not search again.
    cached = RETRIEVAL_CACHE / f"{re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')}.csv"
    if cached.exists():
        per = pd.read_csv(cached)
        if set(per["id"]) == {q["id"] for q in questions}:
            print(f"{name}: hit@1 {per['hit@1'].mean():.3f} · hit@10 {per['hit@10'].mean():.3f} "
                  f"· recall@10 {per['recall@10'].mean():.3f} · MRR {per['reciprocal_rank'].mean():.3f} (from the cache)")
            return per
    started = time.perf_counter()
    results = [{"question": q["question"], "relevant": q["source_ids"], "retrieved": retrieve_fn(q)} for q in questions]
    m = retrieval_metrics.run({"results": results, "k": [1, 3, 5, 10]}, context=ctx)
    per = pd.DataFrame(m["per_question"]).assign(id=[q["id"] for q in questions], category=[q["category"] for q in questions],
                                                 type=[q["type"] for q in questions], configuration=name)
    print(f"{name}: hit@1 {m['summary']['hit@1']:.3f} · hit@10 {m['summary']['hit@10']:.3f} · recall@10 {m['summary']['recall@10']:.3f} "
          f"· MRR {m['summary']['mrr']:.3f} ({time.perf_counter() - started:.0f} s)")
    per.to_csv(cached, index=False)
    return per

RETRIEVAL = {}
for strategy in ["sparse", "dense", "hybrid"]:
    RETRIEVAL[strategy] = evaluate_retrieval(strategy, lambda q, s=strategy: search(q["question"], strategy=s))
baseline_retrieval = RETRIEVAL["hybrid"]
print("\nHybrid search of Part 1, by category:")
display(baseline_retrieval.groupby("category")[["hit@1", "hit@5", "hit@10", "recall@10", "reciprocal_rank"]].mean()
        .reindex(CATEGORIES).rename(columns={"reciprocal_rank": "MRR"}).round(3))

sparse: hit@1 0.640 · hit@10 0.890 · recall@10 0.867 · MRR 0.728 (from the cache)
dense: hit@1 0.436 · hit@10 0.758 · recall@10 0.732 · MRR 0.543 (from the cache)
hybrid: hit@1 0.551 · hit@10 0.864 · recall@10 0.843 · MRR 0.666 (from the cache)

Hybrid search of Part 1, by category:


,hit@1,hit@5,hit@10,recall@10,MRR
category,,,,,
factoid,0.577,0.836,0.878,0.878,0.685
multi-hop,0.333,0.778,0.778,0.620,0.522
global,0.200,0.600,0.600,0.140,0.350


In [ ]:
# Retrieval of the graph system on the Part 1 questions (reciprocal ranks saved by Part 2)
graph_retrieval = pd.read_csv(PART2_ART / "retrieval_results.csv")
print("Part 1 questions, MRR by type (Part 2 measurement):")
display(graph_retrieval.groupby("type")[["rr_text", "rr_text_graph"]].mean().reindex(TYPES)
        .rename(columns={"rr_text": "text-only", "rr_text_graph": "text + graph"}).T.round(3))

# Correctness when the source article is in the prompt and when it is not
in_prompt = scores.groupby([label(scores), "source_in_prompt"]).agg(answers=("id", "size"), correctness=("correctness", "mean"),
                                                                    faithfulness=("faithfulness", "mean"), no_answer=("no_answer", "mean"))
display(in_prompt.round(2))
print("New questions: share of answers whose prompt holds ALL the source articles:")
display(scores[scores["category"] != "factoid"].groupby([label(scores), "category"])["all_sources_in_prompt"].mean().unstack().round(2))

Part 1 questions, MRR by type (Part 2 measurement):


type,factual,temporal,relational,analytical,comparative
text-only,0.778,0.543,0.665,0.740,0.676
text + graph,0.781,0.581,0.721,0.749,0.707


answers  correctness  faithfulness  \
system           source_in_prompt                                       
text + graph RAG False                  42         3.38          4.31   
                 True                  194         4.25          4.49   
text-only RAG    False                  44         3.30          4.27   
                 True                  192         4.21          4.42   

                                   no_answer  
system           source_in_prompt             
text + graph RAG False                  0.24  
                 True                   0.14  
text-only RAG    False                  0.36  
                 True                   0.18

New questions: share of answers whose prompt holds ALL the source articles:


category,global,multi-hop
system,,
text + graph RAG,0.0,0.28
text-only RAG,0.0,0.28


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
r = pd.concat(RETRIEVAL.values())
sns.barplot(data=r, x="category", y="reciprocal_rank", hue="configuration", order=CATEGORIES, ax=axes[0], palette="Blues_d", errorbar=None)
axes[0].set(ylim=(0, 1), xlabel="", ylabel="MRR (source article)", title="Text-only retrieval by strategy and category")
sns.barplot(data=scores.assign(system=label(scores)), x="category", y="correctness", hue="system", order=CATEGORIES, ax=axes[1],
            palette=["#4C72B0", "#DD8452"], errorbar=("ci", 95))
axes[1].set(ylim=(1, 5), xlabel="", ylabel="judge correctness (1–5)", title="Answer correctness by category")
for ax in axes:
    ax.legend(title="")
save_fig(fig, "retrieval_and_correctness")

Retrieval explains most of the correctness scores. When a source article is in the prompt (81–82% of the answers), correctness is 4.21–4.25; when it is not, 3.30–3.38, and the model refuses twice as often (text-only: 36% against 18%). The two systems put a source article in the prompt equally often (192 and 194 of 236 answers), which is why their scores are so close: the graph's entity route reorders the candidates but rarely brings in an article the hybrid search had missed.

By category, hybrid search finds a source article at rank 1 for 58% of the factoid questions, 33% of the multi-hop and 20% of the global ones (MRR 0.69, 0.52, 0.35), and the recall of all the source articles in the first ten is 0.88, 0.62 and 0.14. For the global questions, 86% of the articles needed are missing from the first ten results, and no answer to a global question, from either system, had all its source articles in the prompt (28% for the multi-hop questions). The cause is the ranking, not the size of the prompt: the prompt holds 6 passages with at most 2 per article, so it could hold the five articles of a global question, but they are not among the candidates to begin with. A question that names several models is matched to the chunks about the better-known of them, and the others fall below the first ten.

Among the three strategies, sparse search is the best on this test set (MRR 0.73, hit@1 0.64), hybrid second (0.67, 0.55) and dense last (0.54, 0.44). Part 1 chose hybrid on five example questions; on 236 the word-based search wins, and Section 5.1.1 says why: the questions share words with their passages, so a word match is a strong signal here and the dense half of the hybrid score dilutes it. The measurement of Part 2 on its own test set points the same way as Section 5.3: the graph raises the MRR of the source article for every type, most for temporal (0.54 → 0.58) and relational (0.67 → 0.72) questions, by small amounts.

## 5.6 Enhancement: Re-ranking the Retrieved Passages
**Limitation addressed.** Section 5.5 locates the main failure in retrieval: when the source article reaches the prompt, both systems answer correctly (4.2), and when it does not, they do not (3.3). Hybrid search puts a source article first for 55% of the questions and the dense half of its score is the weaker one (dense alone: hit@1 0.44). The passages that reach the model are the first 6 of a ranking that scores each chunk against the question through two compressed representations, an embedding and a bag of words, and that ranking mixes up close names (*Gemma 3* and *Gemma 4*, *Nemotron Nano* and *Nemotron 3 Nano*) and favours the better-known of two entities in a question. Adding the knowledge graph did not fix this: it changed the prompt of 128 questions and improved correctness on 23 of them while worsening 20.

**Enhancement.** The text-only pipeline retrieves 30 chunks by hybrid search and gives the first 6 (2 per article) to the language model. Hybrid search ranks the chunks by their embedding and their words, two compressed views of a chunk: close names (*Gemma 3* vs *Gemma 4*, *Nemotron Nano* vs *Nemotron 3 Nano*) and questions that name two entities are where this ranking fails. A **cross-encoder re-ranker** reads the question together with the full text of every candidate chunk and scores their relevance jointly (the attention of a transformer over the pair, slide 5.1); it is slower, so it runs on a candidate set from retrieval, not on the whole corpus (*embeddings for recall, re-ranking for precision*, course skeleton A3.5). The enhanced pipeline retrieves **60** chunks by hybrid search, re-ranks them with `cross-encoder/ms-marco-MiniLM-L-6-v2` (the re-ranker shipped with the toolbox), keeps the first 30 and gives the first 6 to the same prompt and model. Everything else is unchanged, so the difference comes from the order of the passages.

The enhancement is evaluated in two steps: on retrieval, for every question and against two other candidates (re-ranking the 30 baseline chunks only, and a recency boost for recent news), and end to end, with the same judge, on the answers it produces.

In [ ]:
def search_and_rerank(question, size=60, keep=30):
    # Hybrid search of `size` chunks, re-ranked by the cross-encoder; the first `keep` are kept
    return rerank.run({"input": search(question, size=size), "keep": keep}, context=ctx)

VARIANTS = {
    "hybrid 30 (Part 1)": lambda q: search(q["question"]),
    "hybrid 60": lambda q: search(q["question"], size=60),
    "hybrid 30 + re-ranking": lambda q: search_and_rerank(q["question"], size=30),
    "hybrid 60 + re-ranking": lambda q: search_and_rerank(q["question"], size=60),
    "hybrid 30 + recency (half-life 1 year)": lambda q: search(q["question"], recency_half_life_days=365),
}
for name, fn in VARIANTS.items():
    if name not in RETRIEVAL:
        RETRIEVAL[name] = evaluate_retrieval(name, fn)
variants = pd.concat([RETRIEVAL[n] for n in VARIANTS])
summary = variants.groupby("configuration")[["hit@1", "hit@3", "hit@5", "hit@10", "recall@10", "reciprocal_rank"]].mean().reindex(list(VARIANTS))
summary.rename(columns={"reciprocal_rank": "MRR"}).round(3).to_csv(OUT_DIR / "retrieval_variants.csv")
display(summary.rename(columns={"reciprocal_rank": "MRR"}).round(3))
print("MRR by category:")
display(variants.groupby(["configuration", "category"])["reciprocal_rank"].mean().unstack().reindex(list(VARIANTS))[CATEGORIES].round(3))
print("MRR by type:")
display(variants.groupby(["configuration", "type"])["reciprocal_rank"].mean().unstack().reindex(list(VARIANTS))[TYPES].round(3))

hybrid 30 (Part 1): hit@1 0.551 · hit@10 0.864 · recall@10 0.843 · MRR 0.666 (from the cache)
hybrid 60: hit@1 0.551 · hit@10 0.864 · recall@10 0.845 · MRR 0.667 (from the cache)
hybrid 30 + re-ranking: hit@1 0.648 · hit@10 0.907 · recall@10 0.888 · MRR 0.735 (from the cache)
hybrid 60 + re-ranking: hit@1 0.644 · hit@10 0.907 · recall@10 0.879 · MRR 0.731 (from the cache)
hybrid 30 + recency (half-life 1 year): hit@1 0.250 · hit@10 0.602 · recall@10 0.582 · MRR 0.357 (from the cache)


,hit@1,hit@3,hit@5,hit@10,recall@10,MRR
configuration,,,,,,
hybrid 30 (Part 1),0.551,0.767,0.826,0.864,0.843,0.666
hybrid 60,0.551,0.767,0.826,0.864,0.845,0.667
hybrid 30 + re-ranking,0.648,0.801,0.839,0.907,0.888,0.735
hybrid 60 + re-ranking,0.644,0.797,0.835,0.907,0.879,0.731
hybrid 30 + recency (half-life 1 year),0.250,0.386,0.487,0.602,0.582,0.357


MRR by category:


category,factoid,multi-hop,global
configuration,,,
hybrid 30 (Part 1),0.685,0.522,0.350
hybrid 60,0.686,0.526,0.354
hybrid 30 + re-ranking,0.757,0.584,0.345
hybrid 60 + re-ranking,0.753,0.573,0.349
hybrid 30 + recency (half-life 1 year),0.358,0.429,0.071


MRR by type:


type,factual,temporal,relational,analytical,comparative
configuration,,,,,
hybrid 30 (Part 1),0.756,0.538,0.658,0.693,0.660
hybrid 60,0.757,0.540,0.658,0.694,0.661
hybrid 30 + re-ranking,0.774,0.633,0.739,0.737,0.781
hybrid 60 + re-ranking,0.773,0.636,0.733,0.734,0.763
hybrid 30 + recency (half-life 1 year),0.332,0.329,0.376,0.390,0.369


In [ ]:
# Paired comparison of the reciprocal ranks: enhanced vs Part 1, per question
ENHANCED = "hybrid 60 + re-ranking"
rr = variants.pivot(index="id", columns="configuration", values="reciprocal_rank")
d = rr[ENHANCED] - rr["hybrid 30 (Part 1)"]
boots = d.to_numpy()[np.random.default_rng(SEED).integers(0, len(d), (1000, len(d)))].mean(axis=1)
print(f"{ENHANCED} vs Part 1: mean reciprocal-rank difference {d.mean():+.3f} "
      f"[{np.percentile(boots, 2.5):+.3f}, {np.percentile(boots, 97.5):+.3f}], better for {(d > 0).mean():.0%} of the questions, "
      f"worse for {(d < 0).mean():.0%}, p = {(boots <= 0).mean():.3f}")

fig, ax = plt.subplots(figsize=(10, 4))
plot = variants[variants["configuration"].isin(["hybrid 30 (Part 1)", ENHANCED])]
sns.barplot(data=plot, x="type", y="reciprocal_rank", hue="configuration", order=TYPES, ax=ax, palette=["#4C72B0", "#55A868"], errorbar=None)
ax.set(ylim=(0, 1), xlabel="", ylabel="MRR (source article)", title="Retrieval before and after re-ranking, by question type")
ax.legend(title="")
save_fig(fig, "enhancement_retrieval")

hybrid 60 + re-ranking vs Part 1: mean reciprocal-rank difference +0.065 [+0.024, +0.109], better for 28% of the questions, worse for 18%, p = 0.002


Re-ranking improves retrieval on every summary measure: hit@1 from 0.55 to 0.64, hit@10 from 0.86 to 0.91, MRR from 0.67 to 0.73, a paired difference of +0.065 in reciprocal rank with a 95% interval of [+0.02, +0.11] (p = 0.002), better for 28% of the questions and worse for 18%. Two of the other variants show where the gain comes from. Retrieving 60 candidates instead of 30 changes nothing on its own (MRR 0.667 against 0.666), and re-ranking 30 candidates is as good as re-ranking 60 (0.735 against 0.731): the improvement is the re-ranker reading question and passage together, not the larger pool, so the cheaper configuration would do. The recency boost, the natural idea for a news corpus, is harmful (MRR 0.36): the test questions are about events spread over two years, and favouring recent articles pushes the 2024 and 2025 sources down for no reason.

By type, re-ranking helps most on the temporal (0.54 → 0.64) and comparative (0.66 → 0.76) questions, the two types where a question names two things, a model and a date or two models, and a cross-encoder can weigh both. By category, it helps the factoid (0.69 → 0.75) and multi-hop (0.52 → 0.57) questions and not the global ones (0.35 → 0.35): a re-ranker scores one passage at a time and cannot know that a question needs several different articles.

### 5.6.1 End-to-end: answers with the re-ranked passages
The enhanced system answers every question with the re-ranked passages and the unchanged prompt and model, and the judge scores the answers as in Section 5.3. The comparison with the text-only baseline is paired, question by question; the prompt is identical for the questions whose first 6 passages did not change, so those answers are expected to be the same.

In [ ]:
SYSTEMS_ALL = {**SYSTEMS, "text+rerank": "text-only RAG + re-ranking"}
SYSTEM_NAMES.update(SYSTEMS_ALL)
generate_all("text+rerank", EVAL_SET, lambda q: {"input": search_and_rerank(q["question"])})
judge_all({"text+rerank"})
scores = score_table(SYSTEMS_ALL)
scores.to_csv(OUT_DIR / "judge_scores.csv", index=False)
label = lambda df: df["system"].map(SYSTEMS_ALL)
HAS_ENHANCED = any(system == "text+rerank" for system, _ in judgments)
assert HAS_ENHANCED, "no judged answers of the enhanced system yet: run the generation (Colab GPU) and the judge first"

same_prompt = {qid for qid in BY_ID if ("text", qid) in answers and ("text+rerank", qid) in answers
               and [s["id"] for s in answers[("text", qid)]["result"]["sources"]] == [s["id"] for s in answers[("text+rerank", qid)]["result"]["sources"]]}
print(f"Prompt unchanged by re-ranking for {len(same_prompt)} of {sum(('text+rerank', q['id']) in answers for q in EVAL_SET)} questions")

display(scores.groupby(label(scores)).agg(answers=("id", "size"), **{c: (c, "mean") for c in CRITERIA}, no_answer=("no_answer", "mean"),
                                           source_in_prompt=("source_in_prompt", "mean")).round(2))
display(scores.groupby(["category", label(scores)])[["correctness", "faithfulness", "source_in_prompt"]].mean().reindex(CATEGORIES, level=0).round(2))

Prompt unchanged by re-ranking for 0 of 236 questions


,answers,faithfulness,relevance,context_grounding,temporal_grounding,correctness,no_answer,source_in_prompt
system,,,,,,,,
text + graph RAG,236,4.46,4.82,4.51,4.58,4.09,0.16,0.82
text-only RAG,236,4.39,4.75,4.47,4.50,4.04,0.22,0.81
text-only RAG + re-ranking,236,4.51,4.77,4.56,4.64,4.25,0.15,0.83


correctness  faithfulness  \
category  system                                                  
factoid   text + graph RAG                   4.16          4.48   
          text-only RAG                      4.11          4.43   
          text-only RAG + re-ranking         4.37          4.55   
multi-hop text + graph RAG                   3.39          4.11   
          text-only RAG                      3.39          4.00   
          text-only RAG + re-ranking         3.33          4.11   
global    text + graph RAG                   3.60          4.60   
          text-only RAG                      3.40          4.40   
          text-only RAG + re-ranking         2.60          4.00   

                                      source_in_prompt  
category  system                                        
factoid   text + graph RAG                        0.83  
          text-only RAG                           0.82  
          text-only RAG + re-ranking              0.86  
multi-hop text + graph RAG                        0.78  
          text-only RAG                           0.78  
          text-only RAG + re-ranking              0.61  
global    text + graph RAG                        0.60  
          text-only RAG                           0.60  
          text-only RAG + re-ranking              0.60

In [ ]:
def paired_systems(a, b, criterion, mask=None, n=1000):
    wide = scores.pivot(index="id", columns="system", values=criterion).dropna(subset=[a, b])
    if mask is not None:
        wide = wide[wide.index.isin(mask)]
    d = (wide[a] - wide[b]).to_numpy()
    boots = d[np.random.default_rng(SEED).integers(0, len(d), (n, len(d)))].mean(axis=1)
    return {"questions": len(d), "mean difference": d.mean(), "low": np.percentile(boots, 2.5), "high": np.percentile(boots, 97.5),
            f"{a} better": (d > 0).mean(), f"{b} better": (d < 0).mean(), "p": (boots <= 0).mean()}

changed_prompt = set(BY_ID) - same_prompt
rows = []
for criterion in ["correctness", "faithfulness", "relevance", "context_grounding"]:
    for subset, mask in [("all questions", None), ("prompt changed by re-ranking", changed_prompt)]:
        rows.append({"comparison": "re-ranking vs text-only", "criterion": criterion, "subset": subset, **paired_systems("text+rerank", "text", criterion, mask)})
    rows.append({"comparison": "re-ranking vs text + graph", "criterion": criterion, "subset": "all questions", **paired_systems("text+rerank", "graph", criterion)})
enhancement = pd.DataFrame(rows)
enhancement.to_csv(OUT_DIR / "enhancement_paired.csv", index=False)
display(enhancement.set_index(["comparison", "criterion", "subset"]).round(3))

fig, ax = plt.subplots(figsize=(10, 4))
long = scores.melt(id_vars=["system"], value_vars=["correctness", "faithfulness", "relevance", "context_grounding"], var_name="criterion", value_name="score")
long["system"], long["criterion"] = long["system"].map(SYSTEMS_ALL), long["criterion"].str.replace("_", "\n")
sns.barplot(data=long, x="criterion", y="score", hue="system", hue_order=list(SYSTEMS_ALL.values()), ax=ax,
            palette=["#4C72B0", "#DD8452", "#55A868"], errorbar=("ci", 95))
ax.set(ylim=(1, 5), xlabel="", ylabel="mean score (1–5, 95% CI)", title="Judge scores of the three systems")
ax.legend(title="", loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)
save_fig(fig, "enhancement_scores")

questions  \
comparison                 criterion         subset                                    
re-ranking vs text-only    correctness       all questions                       236   
                                             prompt changed by re-ranking        236   
re-ranking vs text + graph correctness       all questions                       236   
re-ranking vs text-only    faithfulness      all questions                       236   
                                             prompt changed by re-ranking        236   
re-ranking vs text + graph faithfulness      all questions                       236   
re-ranking vs text-only    relevance         all questions                       236   
                                             prompt changed by re-ranking        236   
re-ranking vs text + graph relevance         all questions                       236   
re-ranking vs text-only    context_grounding all questions                       236   
                                             prompt changed by re-ranking        236   
re-ranking vs text + graph context_grounding all questions                       236   

                                                                           mean difference  \
comparison                 criterion         subset                                          
re-ranking vs text-only    correctness       all questions                           0.216   
                                             prompt changed by re-ranking            0.216   
re-ranking vs text + graph correctness       all questions                           0.161   
re-ranking vs text-only    faithfulness      all questions                           0.114   
                                             prompt changed by re-ranking            0.114   
re-ranking vs text + graph faithfulness      all questions                           0.051   
re-ranking vs text-only    relevance         all questions                           0.025   
                                             prompt changed by re-ranking            0.025   
re-ranking vs text + graph relevance         all questions                          -0.051   
re-ranking vs text-only    context_grounding all questions                           0.093   
                                             prompt changed by re-ranking            0.093   
re-ranking vs text + graph context_grounding all questions                           0.051   

                                                                             low  \
comparison                 criterion         subset                                
re-ranking vs text-only    correctness       all questions                 0.004   
                                             prompt changed by re-ranking  0.004   
re-ranking vs text + graph correctness       all questions                -0.034   
re-ranking vs text-only    faithfulness      all questions                -0.034   
                                             prompt changed by re-ranking -0.034   
re-ranking vs text + graph faithfulness      all questions                -0.102   
re-ranking vs text-only    relevance         all questions                -0.093   
                                             prompt changed by re-ranking -0.093   
re-ranking vs text + graph relevance         all questions                -0.153   
re-ranking vs text-only    context_grounding all questions                -0.047   
                                             prompt changed by re-ranking -0.047   
re-ranking vs text + graph context_grounding all questions                -0.076   

                                                                            high  \
comparison                 criterion         subset                                
re-ranking vs text-only    correctness       all questions                 0.398   
                                             prompt changed by re-ranking  0.398   
re-ranking v

The re-ranked passages produce better answers with the same model and prompt: correctness rises from 4.04 to 4.25 over the text-only baseline, a paired difference of +0.22 with a 95% interval of [+0.00, +0.40] (p = 0.02), the one intervention in this part whose effect on correctness is reliable. The re-ranked system is also ahead of text + graph RAG (+0.16, [−0.03, +0.35], p = 0.06) and has the fewest refusals (15% against 22% and 16%). Faithfulness (+0.11) and context grounding (+0.09) move in the same direction without reaching significance, and relevance does not change (+0.03): re-ranking changes what the model is given, not how it answers. The re-ranker reordered the first six passages for every one of the 236 questions, so the comparison runs on all of them; the correctness score changed for 95 questions, 55 for the better and 40 for the worse.

The gain is concentrated where retrieval was the problem: the factoid questions go from 4.11 to 4.37 and their source article reaches the prompt more often (82% → 86%). The multi-hop questions do not improve (3.39 → 3.33) and the five global questions get worse (3.40 → 2.60): the share of multi-hop answers with a source article in the prompt drops from 78% to 61%. The mechanism is the one Section 5.6 predicted at the retrieval level: the cross-encoder rates each chunk alone, so for a question about two models the chunks about the more prominent one win the six slots, and the second article is pushed out. Precision for one topic is bought with coverage of several, which is the wrong trade for the categories the graph was meant to serve. With 18 and 5 questions these figures are fragile, but their direction is the same at the retrieval and the answer level.

### 5.6.2 Example: an answer changed by the enhancement

In [ ]:
improved = (scores.pivot(index="id", columns="system", values="correctness").dropna(subset=["text", "text+rerank"])
            .assign(gain=lambda d: d["text+rerank"] - d["text"]).sort_values("gain", ascending=False))
if len(improved) and improved["gain"].iloc[0] > 0:
    qid = improved.index[0]
    q = BY_ID[qid]
    display(Markdown(f"**{qid} ({q['category']}, {q['type']})**: {q['question']}\n\n*Reference answer*: {q['reference_answer']}"))
    for system in ["text", "text+rerank"]:
        display(Markdown(f"*Judge correctness {judgments[(system, qid)]['scores']['correctness']}, faithfulness {judgments[(system, qid)]['scores']['faithfulness']}*"))
        show_answer(system, qid)
    print("Questions where re-ranking changed the correctness score:",
          f"{(improved['gain'] > 0).sum()} better, {(improved['gain'] < 0).sum()} worse, {(improved['gain'] == 0).sum()} same")

**q015 (factoid, factual)**: If I modify a Llama model, who owns the result?

*Reference answer*: You do: you own your modifications and derivative works of the Llama Materials, subject to Meta's ownership of the original Llama Materials and of derivatives made by or for Meta.

*Judge correctness 1, faithfulness 2*

**text-only RAG** · sources: [S1] meta-llama/Llama-3.2-3B-Instruct · Hugging Face (2024-09-18); [S2] Large Language Models for Commercial Use (2026-06-04); [S3] Meta now allows military agencies to access its AI (2024-11-11) … · graph facts: none

> If you modify a Llama model, you do not own the result; Meta owns the intellectual property rights to the original Llama model, and any modifications are still subject to Meta’s license terms. The license grants a non-exclusive, worldwide, non-transferable, and royalty-free limited license to use, reproduce, distribute, copy, create derivative works of, and make modifications to the Llama Materials [S1]. However, this license does not transfer ownership of the resulting derivative work to the user. Instead, the user retains only the right to use the modified model under the terms of the license, which includes obligations such as providing attribution and disclosing the use of Llama in any product or service derived from it [S5]. 

Moreover, the license includes limitations on commercial use, which implies that while users can modify the model, the resulting derivative work may not be used commercially without further permission or under conditions that do not fully satisfy open-source standards [S3]. Therefore, the ownership of the result of the modification remains with Meta, as the original intellectual property rights are retained by Meta Platforms, Inc. [S1][S5].

Thus, the result of modifying a Llama model is not owned by the user, but rather remains under Meta’s intellectual property rights as defined in the Llama license agreement.

*Judge correctness 5, faithfulness 5*

**text-only RAG + re-ranking** · sources: [S1] meta-llama/Llama-3.2-3B-Instruct · Hugging Face (2024-09-18); [S2] Meta now allows military agencies to access its AI (2024-11-11); [S3] Large Language Models for Commercial Use (2026-06-04) … · graph facts: none

> If you modify a Llama model, you are the owner of such derivative works and modifications, as stated in the license agreement between you and Meta [S1]. This means that, with respect to any derivative works and modifications of the Llama Materials that are made by you, you are and will be the owner of such derivative works and modifications [S1]. 

However, while you own the modified model, you must comply with Meta’s brand guidelines and provide proper attribution, such as including "Built with Llama" and "Llama 3.2 is licensed under the Llama 3.2 Community License" in any distribution [S4]. Additionally, if you create or distribute an AI model based on the Llama Materials, you must include "Llama" at the beginning of the model's name [S4].

It is important to note that although the license grants you ownership of the derivative work, it does not grant full commercial freedom—specific restrictions on commercial use and trademark usage remain in place [S2]. The Llama 3.2 Community License, while permissive in some aspects, still imposes conditions on how the model and its outputs are used, especially in commercial contexts [S1][S4].

Therefore, while you own the modified model, your use is subject to Meta’s licensing terms, including attribution and restrictions on commercial applications.

Questions where re-ranking changed the correctness score: 55 better, 40 worse, 141 same


## 5.7 Summary: Trade-offs and Insights for Trend Monitoring
The three systems answer the same 236 questions with the same language model, prompt and corpus; they differ only in which six passages, and which graph facts, the model is given.

| System | Correctness | Relevance | Refusals | Source in prompt | Cost per question |
|---|---|---|---|---|---|
| text-only RAG (Part 1) | 4.04 | 4.75 | 22% | 81% | one hybrid search |
| text + graph RAG (Part 2) | 4.09 | 4.82 | 16% | 82% | entity linking, graph lookup and fusion, plus the graph to build and maintain |
| text-only RAG + re-ranking (this part) | **4.25** | 4.77 | **15%** | **83%** | one hybrid search and one cross-encoder pass over 60 chunks (about 0.3 s on a GPU) |

**Trade-offs.** The knowledge graph buys a small, reliable gain in relevance and fewer refusals, and a gain in correctness that cannot be told from zero, at the price of the heaviest pipeline of the three. Its contribution is most visible on temporal and comparative questions, where the entity route brings the right articles into the ranking; it is absent on multi-hop questions and negative on analytical ones. Re-ranking buys a reliable gain in correctness for a modest cost, but only for questions that one article can answer; it harms the questions that need several. The text-only baseline is the cheapest and, with sparse search instead of hybrid, would be a stronger baseline than the one measured here.

**What RAG with a knowledge graph brings to trend monitoring in the open-models track.** The graph's value in this corpus is not the facts it stores, which are mostly of the kind *Nemotron-70B is a Nemotron model* and rarely answer a question on their own, but the way it *finds* articles: through the entities they mention rather than the words they share with the question. For monitoring a trend, where the question is "what has been written about X and the models built on it" rather than "what is X", that is the right mechanism, and the retrieval measurement shows it working. What limits it here is breadth: a ranking, with or without the graph, puts the most relevant chunks first, and for a question that spans five articles the most relevant chunks tend to come from the same one or two. The global questions therefore fail at retrieval, before the model or the graph facts can help (recall@10 of 0.14). The next step for a monitoring system is not a better graph but a different use of it: retrieve by entity, keep one passage per article, and summarise per article before answering, so that the breadth the graph can find survives into the prompt.

**Scope**
- **Size of the new categories.** The 23 multi-hop and global questions show the direction of the differences per category; the paired intervals of the 213 factoid questions carry the comparison, and every per-category result is reported with its interval (Section 5.3.2).
- **Wording of the test set.** Questions drafted from passages share about half of their content words with them (Section 5.1.1). This holds for every system alike, so it does not affect the comparison; it explains why sparse search is strong on this test set (Section 5.5).
- **Judge model.** The judge is a different model family from the RAG model, its scores are checked against a person (Section 5.4) and against a higher reasoning effort (Section 5.4.1), and all comparisons are paired, so a bias shared by the systems cancels.

**Next steps.** Global questions need retrieval that covers several articles: keeping one passage per article, or re-ranking the entity-based candidates of the graph system, combines the breadth of the graph (Section 4) with the precision of re-ranking (Section 5.6).


## 5.8 Saved Outputs
All outputs are saved in `OUT_DIR` on Google Drive (`HSLU/CLT/extraction_outputs/stage3/wp3`); the tables, sheets and figures are also copied to `artifacts/stage3_wp3/` in the repository.

| File | Content |
|---|---|
| `qa_multihop_draft.csv`, `qa_multihop_reviewed.xlsx`, `qa_multihop.json` | the new questions: the 30 drafted, the manual review, the 23 kept |
| `test_set_v2.json` | the full test set with the category of every question |
| `answers_wp3.jsonl` | the answers generated in this part (both systems on the new questions, enhanced system) |
| `judgments.jsonl`, `judge_scores.csv` | the judgments of every answer (scores, rationales, unsupported claims, flags) and the score table |
| `judge_paired.csv`, `enhancement_paired.csv` | paired comparisons of the systems with bootstrap intervals |
| `judge_check_sheet.xlsx`, `judge_agreement.csv` | human check of the judge: the blind sheet with the reviewer's scores (the system of each answer is in `judge_check_key.csv`, kept on Drive only), and the agreement |
| `judgments_high.jsonl`, `judge_effort.csv` | the same 30 answers judged at high reasoning effort, and the agreement between the two settings |
| `retrieval_variants.csv` | retrieval metrics of the Part 1 search and the enhancement variants |
| `figures/` | visualisations |

In [ ]:
PART3_ART.mkdir(parents=True, exist_ok=True)
(PART3_ART / "figures").mkdir(exist_ok=True)
for f in OUT_DIR.iterdir():
    if f.suffix in {".csv", ".json", ".xlsx"} and f.stat().st_size < 5_000_000 and f.name != "judge_check_key.csv":
        shutil.copy(f, PART3_ART / f.name)
for f in FIG.glob("*.png"):
    shutil.copy(f, PART3_ART / "figures" / f.name)
for f in sorted(PART3_ART.rglob("*")):
    if f.is_file():
        print(f"{f.relative_to(REPO)}  ({f.stat().st_size / 1e3:.0f} kB)")

artifacts/stage3_wp3/answers_wp3.jsonl  (1585 kB)
artifacts/stage3_wp3/enhancement_paired.csv  (2 kB)
artifacts/stage3_wp3/figures/enhancement_retrieval.png  (47 kB)
artifacts/stage3_wp3/figures/enhancement_scores.png  (51 kB)
artifacts/stage3_wp3/figures/judge_scores_by_category.png  (75 kB)
artifacts/stage3_wp3/figures/judge_scores_overall.png  (88 kB)
artifacts/stage3_wp3/figures/retrieval_and_correctness.png  (70 kB)
artifacts/stage3_wp3/judge_agreement.csv  (0 kB)
artifacts/stage3_wp3/judge_check_sheet.xlsx  (92 kB)
artifacts/stage3_wp3/judge_effort.csv  (1 kB)
artifacts/stage3_wp3/judge_paired.csv  (4 kB)
artifacts/stage3_wp3/judge_scores.csv  (51 kB)
artifacts/stage3_wp3/judgments.jsonl  (850 kB)
artifacts/stage3_wp3/judgments_high.jsonl  (37 kB)
artifacts/stage3_wp3/qa_multihop.json  (18 kB)
artifacts/stage3_wp3/qa_multihop_draft.csv  (125 kB)
artifacts/stage3_wp3/qa_multihop_reviewed.xlsx  (58 kB)
artifacts/stage3_wp3/retrieval_variants.csv  (0 kB)
artifacts/stage3_wp3/test_se

# AI Assistance Disclosure
- **Q&A drafting**: Claude Opus 5.5 (Anthropic) drafted the questions and reference answers of Section 1 from the selected passages, and Claude Opus 5 the 30 multi-hop and global questions of Section 5.1, with one evidence sentence per passage. The brief requires the Q&A pairs to be generated by an LLM different from the RAG model (Qwen3-4B); a large proprietary model was chosen because it writes more natural and precise questions than a small open model. Every drafted question was then reviewed manually by the team (Sections 1.6 and 5.1).
- **Judge model**: GPT-5 mini (OpenAI) scored the answers on the five criteria through the `score_rag_answer` tool of the course toolbox; its scores were checked against the team's scores on a blind sample (Section 5.4).
- **Coding assistant**: Claude (Anthropic) was used to write and debug the code and to suggest corrections in the notebook.
- **Writing**: Claude helped draft, shorten and rephrase the markdown texts. Every figure quoted in the text was checked against the outputs of the notebook, and all interpretations were validated by the team members.
